# Racialised discourse in German swimming pool reviews — full analysis

Single merged flow, in three parts, each continuing directly from the one before -- shared objects (`corpus`, `const_agg`, `racialised_df`, `analysis_df`, `PREDICTOR_COLS`) are built once and reused in-memory by later parts, not reloaded or re-derived:

1. **Part 1 — Descriptive (WHEN / WHERE / WHO).** Gmaps review data alone: temporal trend, geographic distribution, reviewer ethnicity/language and content. Originally `descriptive_analysis.ipynb`.
2. **Part 2 — Socioeconomic & political relations.** Brings in constituency-level socioeconomic/political data to explain Part 1's WHERE pattern (data-quality checks first, then correlations, confound checks, and the `Arab`-share follow-up). Originally `02_socioeconomic_political_relations.ipynb`.
3. **Part 3 — Predictive modelling.** A single joint negative-binomial model testing whether Part 2's surviving predictors explain Part 1's WHERE pattern jointly, not just one-at-a-time.

See `results_narrative_for_supervisor.md` for the plain-language story and `analysis_qa_summary.md` for the full question-by-question reference.

# Racialised discourse in German swimming pool reviews: prevalence across WHEN, WHERE, and WHO

## Notebook Summary


**Data provenance** (see `gmaps_data/README.md` and `socio-economic_data/README_data_processing.md` for full detail — not reprocessed here):
- `master_corpus_final.parquet`: 1,265,806 Google Maps reviews of 5,529 verified swimming-pool POIs in Germany. Every row has non-null `org_review`/`trans_review` — **this corpus contains only reviews that have text**, not every review Google recorded for these places (many Google reviews are a star rating with no text, and are simply absent from this corpus). We call this count `total_text_reviews` throughout to keep that scope explicit. `social` (yes/no) and `racialised` (yes/no/NaN) are review-level labels; `racialised` is non-null for every `social == 'yes'` row (504,072 rows), with **7,768 `yes`**. `approach`/`theme`/`connotation`/`context` are multi-label lists populated only for those 7,768 rows ("dimension scope").
- `federal_baseline_eng_selected_political.csv`: 299 constituency-level rows with socioeconomic indicators (population, age structure, income, employment, SGB II) and Bundestag 2025 **second-vote shares** (not raw counts, `_pct`  ).

## Research questions

- **RQ1.** How prevalent is racialised discourse?
--- **When**   What is the time series trend for  racialised rate across seasons?
               What is the time series trend  for the  dimensions of racial reviews?
--- **Where**  In which constituencies is racialised rate  prevalent
--- **Who**    Who  is racialising?


This notebook (Part 1) works toward RQ1 using the gmaps review data alone, organized into
WHEN (temporal trend), WHERE (geographic distribution), and WHO (reviewer ethnicity/
language) sections.

## Setup

Load the two canonical inputs and do a first-touch shape/label sanity check against the counts documented in the corpus README (7,768 racialised, 504,072 social=yes, 1,265,806 total) — if these don't match, something upstream has drifted and everything after this point is suspect.


In [ ]:
#packages
import warnings
from pathlib import Path
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import statsmodels.formula.api as smf
from statsmodels.stats.proportion import proportions_ztest


warnings.filterwarnings("ignore", category=FutureWarning)
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 60)

from statsmodels.stats.proportion import proportion_confint

def slugify(title):
    """Turn a plot title into a filesystem-safe filename stem (used when saving figures)."""
    return re.sub(r"[^a-z0-9]+", "_", title.lower()).strip("_")[:80]

In [ ]:
ANALYSIS_DIR = Path("/home/ethel/Desktop/swimming_pool_project/spp_15052026/analysis_10072026")
GMAPS_DIR = ANALYSIS_DIR / "gmaps_data"
SOCIO_DIR = ANALYSIS_DIR / "socio-economic_data"
PROJECT_ROOT = Path("/home/ethel/Desktop/swimming_pool_project")
SHAPEFILE_PATH = PROJECT_ROOT / "btw25_geometrie_wahlkreise_shp" / "btw25_geometrie_wahlkreise_shp.shp"
FIGURES_DIR = ANALYSIS_DIR / "descriptive_analysis_figures"
FIGURES_DIR.mkdir(exist_ok=True)

corpus = pd.read_parquet(GMAPS_DIR / "master_corpus_final.parquet")
socio_political = pd.read_csv(SOCIO_DIR / "socio_political_data.csv")

print(f"corpus: {corpus.shape[0]:,} rows x {corpus.shape[1]} cols")
print(f"political/socioeconomic: {socio_political.shape[0]} constituencies x {socio_political.shape[1]} cols")

assert corpus.shape[0] == 1_265_806, "row count drifted from documented corpus size"
assert (corpus["racialised"] == "yes").sum() == 7_768, "racialised=yes count drifted"
assert (corpus["social"] == "yes").sum() == 504_072, "social=yes count drifted"
assert socio_political.shape[0] == 299, "expected exactly 299 Bundestag constituencies"
print("Sanity checks passed: corpus and political data match documented provenance.")


In [ ]:
corpus.info()
#.info()
#corpus[corpus['social']== 'yes'].groupby('racialised').size().reset_index(name='count') 

## Rebuilding the place → constituency spatial join


 *method* — dissolve the 299 constituency polygons into one Germany polygon, keep only points strictly inside it, then `sjoin_nearest` the kept points against the individual polygons so every retained point is guaranteed exactly one constituency, even ones that sit awkwardly on a shared border. 
We use that method  once at the **place level** (5,529 unique places, keyed by `unique_identifier`) and then broadcast the result back onto all 1,265,806 review rows — same logic, ~230x fewer spatial operations, and verified end-to-end against the current canonical corpus.

**CRS choice:** the shapefile's native CRS is `EPSG:25832` (ETRS89 / UTM zone 32N), a metric projection appropriate for Germany. We reproject points into that CRS rather than reprojecting to Web Mercator (`EPSG:3857`, used in the openday script) — reprojecting to the shapefile's own native CRS avoids an extra, unnecessary reprojection step and the distortion Web Mercator introduces at this latitude.

In [ ]:
import geopandas as gpd

# --- place-level table (one row per unique_identifier) ---
places = (
    corpus[["unique_identifier", "place_name", "latitude", "longitude"]]
    .drop_duplicates(subset="unique_identifier")
    .reset_index(drop=True)
)
n_missing_coords = places[["latitude", "longitude"]].isna().any(axis=1).sum()
print(f"places (unique unique_identifier): {len(places):,}")
print(f"places missing lat/lon: {n_missing_coords}")

places_geo = gpd.GeoDataFrame(
    places,
    geometry=gpd.points_from_xy(places["longitude"], places["latitude"]),
    crs="EPSG:4326",
).to_crs("EPSG:25832")

# --- constituency shapefile ---
constituencies = gpd.read_file(SHAPEFILE_PATH)
print(f"constituencies: {len(constituencies)} polygons, CRS = {constituencies.crs}")

n_invalid = (~constituencies.geometry.is_valid).sum()
print(f"invalid polygon geometries before repair: {n_invalid}")
if n_invalid:
    constituencies["geometry"] = constituencies.geometry.buffer(0)

assert constituencies.crs.to_string() == "EPSG:25832", "shapefile CRS is not what we expect"
assert constituencies["WKR_NR"].nunique() == 299, "expected 299 unique constituency numbers"


In [ ]:
# --- step 1: keep only places strictly inside Germany (dissolved boundary) ---
germany_boundary = constituencies.dissolve().geometry.iloc[0]
germany_gdf = gpd.GeoDataFrame(geometry=[germany_boundary], crs=constituencies.crs)

inside_mask = gpd.sjoin(places_geo, germany_gdf, predicate="within", how="inner").index
places_inside = places_geo.loc[places_geo.index.isin(inside_mask)].copy()
n_dropped = len(places_geo) - len(places_inside)
print(f"places strictly inside Germany polygon: {len(places_inside):,} (dropped as outside: {n_dropped})")

# --- step 2: assign each inside place to its nearest constituency polygon ---
# (guarantees a match even for a point sitting exactly on / just outside a shared border)
joined = gpd.sjoin_nearest(
    places_inside,
    constituencies[["WKR_NR", "WKR_NAME", "LAND_NR", "LAND_NAME", "geometry"]],
    how="left",
)

n_dupe_matches = joined.duplicated(subset="unique_identifier").sum()
print(f"places matched more than once (nearest-neighbour ties): {n_dupe_matches}")
joined = joined.drop_duplicates(subset="unique_identifier", keep="first")

place_constituency = joined.rename(
    columns={"WKR_NR": "constituency_code", "WKR_NAME": "constituency", "LAND_NAME": "state"}
)[["unique_identifier", "constituency_code", "constituency", "state"]].reset_index(drop=True)

print(f"final place -> constituency mapping: {len(place_constituency):,} rows")
assert place_constituency["unique_identifier"].is_unique, "join must be 1:1 on unique_identifier"
place_constituency.head()


In [ ]:
# --- broadcast the place-level mapping back onto every review row ---
corpus = corpus.merge(place_constituency, on="unique_identifier", how="left")

n_reviews_unmatched = corpus["constituency_code"].isna().sum()
n_places_unmatched = places["unique_identifier"].nunique() - place_constituency["unique_identifier"].nunique()
print(f"review rows without a constituency match: {n_reviews_unmatched:,} ({n_reviews_unmatched / len(corpus):.4%})")
print(f"places without a constituency match: {n_places_unmatched:,}")

if n_reviews_unmatched:
    print("\nplace_name of unmatched places:")
    print(
        corpus.loc[corpus["constituency_code"].isna(), "place_name"].drop_duplicates().head(10).to_list()
    )

east_states = {
    "Brandenburg", "Mecklenburg-Vorpommern", "Sachsen", "Sachsen-Anhalt", "Thüringen",
}
corpus["region"] = np.where(corpus["state"].isin(east_states), "East", "West")
corpus.loc[corpus["state"] == "Berlin", "region"] = "Berlin"

print("\nreviews by region (East / West / Berlin split out):")
print(corpus["region"].value_counts(dropna=False))


**Takeaway.**  The fresh join is a clean, complete match: all 5,528 places with reviews in `master_corpus_final` (5,529 verified POIs minus the one documented in the `gmaps_dat/README` as having no reviews) fall inside Germany and get exactly one constituency, with zero drops and zero nearest-neighbour ties — so **100% of the 1,265,806 review rows now carry a `constituency_code`/`constituency`/`state`**. T We also derive a simple `region` (East/West/Berlin) split for later comparisons, reusing the standard German reunification state grouping.

In [ ]:
# racialised=yes subset, reused throughout Part 1 (WHEN's dimension-over-time facet,
# WHERE's pool-concentration checks, WHO's ethnicity/language facet, and the
# dimension-content-overview section) -- built once here rather than re-derived per section.
racialised_df = corpus.loc[corpus["racialised"] == "yes"].copy()
print(f"racialised=yes reviews: {len(racialised_df):,}")

# Wilson score interval helper, used throughout Part 1 (WHEN's yearly/monthly rates, WHERE's
# constituency/state prevalence) -- defined once here, before either section needs it.
from statsmodels.stats.proportion import proportion_confint

def wilson(successes, n):
    if n == 0:
        return np.nan, np.nan, np.nan
    rate = successes / n
    lo, hi = proportion_confint(successes, n, method="wilson")
    return rate, lo, hi


# Part 1 — Prevalence of racialised discourse: WHEN, WHERE, and WHO

This section establishes the scale of racialised discourse in the gmaps corpus and how it distributes across constituencies. Social reviews make up 504,072 / 1,265,806 (~40%) of the corpus; racialised reviews are rarer still (7,768 / 1,265,806 ≈ 0.6% of all reviews, ~1.5% of social reviews). Spread over 299 constituencies, many will have few racialised reviews — not because there's no racialised discourse there, but because pool coverage itself is thin in the first place. We establish this coverage picture before computing rates and drawing maps.

Part 1 investigates the prevalence of racialised reviews across **WHEN** (temporal trend), **WHERE**
(geographic distribution), and **WHO** (reviewer ethnicity/language, including the content -- theme,
approach, connotation, context -- of what's said).

## Dimension classification coverage

Before the WHEN/WHERE/WHO breakdown, a quick coverage check: `approach`, `theme`, `connotation`, and `context` are populated for every one of the 7,768 racialised=yes reviews (multi-label -- a review can carry more than one value per dimension). The table below confirms 100% coverage per dimension and shows how the labels break down as a share of all racialised reviews.


In [ ]:
DIMENSIONS = ["approach", "theme", "connotation", "context"]

def is_classified(cell):
    if isinstance(cell, (list, tuple, set, np.ndarray)):
        return len(cell) > 0
    if isinstance(cell, str):
        return len(cell) > 0
    return False  # NaN or unexpected type

coverage_rows = []
label_rows = []
for dim in DIMENSIONS:
    n_classified = racialised_df[dim].apply(is_classified).sum()
    coverage_rows.append({
        "dimension": dim,
        "n_classified": n_classified,
        "pct_classified": n_classified / len(racialised_df),
    })
    counts = racialised_df[dim].explode().value_counts()
    for label, n in counts.items():
        label_rows.append({
            "dimension": dim,
            "label": label,
            "n_reviews": n,
            "pct_of_racialised_reviews": n / len(racialised_df),
        })

coverage_table = pd.DataFrame(coverage_rows).set_index("dimension")
print(f"Classification coverage (of {len(racialised_df):,} racialised reviews):")
print(coverage_table.assign(pct_classified=lambda d: (d["pct_classified"] * 100).round(1)))
assert (coverage_table["pct_classified"] == 1.0).all(), "not every racialised review is classified on every dimension"

label_breakdown = pd.DataFrame(label_rows).set_index(["dimension", "label"])
label_breakdown["pct_of_racialised_reviews"] = (label_breakdown["pct_of_racialised_reviews"] * 100).round(1)
display(label_breakdown)


# WHEN


How the racialised-review rate moves over time: the headline year-over-year trend, the
mechanics of recovering timestamps for the small residual share of reviews that lacked one, how far back the
window can honestly be extended, month/season granularity, and how the four content
dimensions (`theme`/`context`/`connotation`/`approach`) move over the same years -- closing
with formal significance tests for the trend and season effects (both overall and per
dimensional label) and a check on whether the post-2021 rise is a nationwide pattern or
concentrated in a handful of constituencies.


### Headline trend

Timestamps were only merged into the corpus recently — the new `review_timestamp` column
enables this trend for the first time — and, as of the most recent `gmaps_poi` DB refresh, **1,260,044 / 1,265,806 (99.5%)** reviews now have a parsed absolute timestamp (up from 69.5% before the refresh; the remaining 0.5% have unparseable relative time text at source). Before trusting any time trend, we first check whether that residual missingness is roughly random or structured — e.g. concentrated in specific states or disproportionately hitting racialised reviews, which would bias a trend line built only from the reviews that do have a timestamp.

In [ ]:
has_ts = corpus["review_timestamp"].notna()
print(f"reviews with a parsed timestamp: {has_ts.sum():,} / {len(corpus):,} ({has_ts.mean():.1%})")

# missingness by racialised status (restricted to social=yes, where racialised is defined)
social_mask = corpus["social"] == "yes"
miss_by_racialised = (
    corpus.loc[social_mask]
    .assign(has_ts=has_ts[social_mask])
    .groupby("racialised")["has_ts"]
    .mean()
)
print("\ntimestamp coverage by racialised status (within social=yes):")
print(miss_by_racialised)

# missingness by state
miss_by_state = corpus.assign(has_ts=has_ts).groupby("state")["has_ts"].mean().sort_values()
print("\ntimestamp coverage by state (min/max):")
print(miss_by_state.head(3))
print("...")
print(miss_by_state.tail(3))


Missingness is not badly structured: 99.44% vs. 98.22% timestamp coverage for racialised=no vs. yes (a small gap, not the kind that would badly bias a trend), and state-level coverage is now tight (99.2-99.95%) with no extreme outlier. We proceed with the trend using only rows with a parsed timestamp, treating this as a **very close approximation of the true time trend** given the residual 0.5% gap, and note per-state coverage as a caveat rather than a blocker.

## Recovering missing timestamps from the relative time text

Rather than stopping at "99.5% coverage," we can recover almost all of the rest. `review_time` is the original Google Maps **relative** time text (e.g. `"2 years ago"`, `"Edited 3 months ago"`) that the `gmaps_poi` DB parsed into the absolute `review_timestamp` we've been using — and the review-level `retrieved_time` (the scrape timestamp needed as the reference point for "N years ago") is null for **5,722 of the 5,762 rows missing `review_timestamp` (99.3%)**, which is almost certainly *why* the DB couldn't parse them, not a separate problem.

The place-level `place_retrieved_time` column, however, is **never null** and only takes **6 distinct values** across the whole corpus (a handful of scrape batches) — the review-level `retrieved_time` is presumably a more precise per-review scrape stamp where available, but the place-level batch date is a reasonable stand-in reference point when it isn't: reviews for the same place were almost certainly scraped in the same session. Using `place_retrieved_time` as the reference date, a regex against `review_time` (`(?:Edited )?<a|N> <unit(s)> ago`, treating "Edited N ago" the same as the rest of the corpus's already-parsed timestamps — it's still the most recent activity signal available for that review, consistent with how the DB's own parsing already treats it) matches **100% of the non-null `review_time` text** among the missing rows. The estimate is coarse (year/month/week granularity, not exact day), which is fine for a yearly trend but would not be fine for day-level analysis.

In [ ]:
REL_TIME_PATTERN = re.compile(
    r"^(?:Edited\s+)?(a|an|\d+)\s+(year|month|week|day|hour|minute)s?\s+ago$", re.IGNORECASE
)
UNIT_TO_DAYS = {"year": 365.25, "month": 30.44, "week": 7, "day": 1, "hour": 1 / 24, "minute": 1 / (24 * 60)}

def parse_relative_time(text):
    m = REL_TIME_PATTERN.match(text) if isinstance(text, str) else None
    if not m:
        return None
    qty_raw, unit = m.groups()
    qty = 1 if qty_raw.lower() in ("a", "an") else int(qty_raw)
    return pd.Timedelta(days=qty * UNIT_TO_DAYS[unit.lower()])

missing_ts_mask = corpus["review_timestamp"].isna()
deltas = corpus.loc[missing_ts_mask, "review_time"].apply(parse_relative_time)
n_parsed = deltas.notna().sum()
print(f"missing timestamps: {missing_ts_mask.sum():,}; recoverable via relative-time text: {n_parsed:,} "
      f"({n_parsed / missing_ts_mask.sum():.1%})")

# review_timestamp is tz-aware (parsed from the gmaps_poi DB); place_retrieved_time is tz-naive and the
# estimate ends up nanosecond-precision. We only need year/month granularity for the trend, so build
# both sides as plain tz-naive 'us'-precision series and combine, rather than assigning in place.
base = corpus["review_timestamp"].dt.tz_localize(None).astype("datetime64[us]")
estimated = pd.Series(pd.NaT, index=corpus.index, dtype="datetime64[us]")
raw_est = (corpus.loc[missing_ts_mask, "retrieved_time"] - deltas).astype("datetime64[us]")
#if there are missing use the place_retrieved_time - parsed delta to estimate the review_timestamp
raw_est = (corpus.loc[missing_ts_mask, "place_retrieved_time"] - deltas).astype("datetime64[us]")
estimated.loc[missing_ts_mask] = raw_est

corpus["review_timestamp_est"] = base.fillna(estimated)
corpus["timestamp_is_estimated"] = missing_ts_mask & corpus["review_timestamp_est"].notna()

new_coverage = corpus["review_timestamp_est"].notna().mean()
print(f"timestamp coverage after recovery: {new_coverage:.2%} (was {has_ts.mean():.1%})")
print(f"still unrecoverable (no review_time text at all): {corpus['review_timestamp_est'].isna().sum():,}")

### Trend over time for all reviews

social=='yes' total: 504,072
minus 22 with unrecoverable timestamp → 504,050
minus 14,367 in 2026 (dropped by d[d["review_year"] < 2026] in bucketed_rate_table, since 2026 is a partial year) → 489,683


In [ ]:
# --- pool pre-2016 years into a single bucket (too sparse to read year-by-year), then plot
# racialised rate (line, top) against review volume -- all reviews vs social=yes reviews (bars, bottom) ---

CUTOFF_YEAR = 2016

def make_bucket(year):
    return f"pre-{CUTOFF_YEAR}" if year < CUTOFF_YEAR else str(int(year))

def bucketed_rate_table(df, count_col, success_fn):
    d = df[df["review_year"] < 2026].copy()  # drop partial current year
    d["bucket"] = d["review_year"].apply(make_bucket)
    table = (
        d.groupby("bucket")
        .agg(n_reviews=(count_col, "size"), n_success=(count_col, success_fn))
    )
    order = [f"pre-{CUTOFF_YEAR}"] + [str(y) for y in range(CUTOFF_YEAR, 2026)]
    table = table.reindex(order).dropna()
    rates = table.apply(
        lambda r: pd.Series(wilson(r["n_success"], r["n_reviews"]), index=["rate", "ci_lo", "ci_hi"]),
        axis=1,
    )
    return table.join(rates)


# n_reviews here = all reviews per bucket, rate = social rate (not plotted, kept for volume bars)
corpus["review_year"] = corpus["review_timestamp_est"].dt.year
yearly_totals = bucketed_rate_table(corpus, "social", lambda s: (s == "yes").sum())

# n_reviews here = social reviews per bucket, rate = racialised rate
ts_social = corpus.loc[social_mask & corpus["review_timestamp_est"].notna()].copy()
ts_social["review_year"] = ts_social["review_timestamp_est"].dt.year
yearly_social = bucketed_rate_table(ts_social, "racialised", lambda s: (s == "yes").sum())

buckets = yearly_totals.index
x = list(range(len(buckets)))
width = 0.4
social_aligned = yearly_social.reindex(buckets)


fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 8), sharex=True, gridspec_kw={"height_ratios": [1, 1.2]})

# top: racialised rate, line + Wilson CI band
pct = social_aligned["rate"] * 100
ax1.plot(x, pct, marker="o", color="#C44E52", linewidth=2)
ax1.fill_between(x, social_aligned["ci_lo"] * 100, social_aligned["ci_hi"] * 100, color="#C44E52", alpha=0.15)
ax1.set_ylabel("racialised reviews,\nas % of social reviews")
print(f"N={int(yearly_social['n_reviews'].sum()):}")
# ax1.set_title(
#     f"Racialised rate and review volume by year (N={int(yearly_social['n_reviews'].sum()):,} social reviews",
    #f"pre-{CUTOFF_YEAR} pooled and 2026 partial year dropped)",
#     fontsize=16 , fontweight="bold"
# )


for spine in ["top", "right"]:
    ax1.spines[spine].set_visible(False)

# bottom: review volume, all reviews vs social = yes reviews
ax2.bar([i + width / 2 for i in x], social_aligned["n_reviews"], width=width, color="#C44E52", label="social = yes")
ax2.bar([i - width / 2 for i in x], yearly_totals["n_reviews"], width=width, color="#4C72B0", label="all reviews")
ax2.set_ylabel("reviews")
ax2.set_xlabel("review year")
ax2.set_xticks(x)
ax2.set_xticklabels(buckets, rotation=45, ha="right")
ax2.legend(fontsize=9)
for spine in ["top", "right"]:
    ax2.spines[spine].set_visible(False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "racialised_rate_pre2016_bucketed.png", dpi=150, bbox_inches="tight")
plt.show()



**Takeaway.**

- Recovering timestamps from the relative-time text lifts coverage from 99.5% to **100.00%** (only 40 reviews stay unrecoverable) -- a genuinely complete temporal picture.
- The trend holds on real (not estimated) data for nearly the whole corpus: racialised rate falls to **0.77% in 2021**, then climbs to **2.30% in 2025** -- roughly a 3x rise in four years.
- 2012-2015 stay too thin to read as a baseline (109-1,254 reviews/year).
- **2026 is still a partial year** (14,367 reviews vs. 66-80k in 2023-2025) -- its lower rate (1.46%) is a truncation artifact, not a reversal.
- The recovery step itself is now a last-mile correction, not load-bearing: estimated share per year is tiny (0.1-2.9%), down from when raw coverage was only 69.5%.

## Seasonal granularity

Beyond the yearly trend, do racialised reviews cluster in particular months or seasons? Two caveats before diving in:

1. **Month-level analysis below uses only reviews with a genuinely parsed `review_timestamp`, not the recovered-estimate values built above.** The recovery step subtracts a relative-time delta from `place_retrieved_time`, which takes only 6 distinct values and is overwhelmingly clustered in late April 2026 (>99% of scrapes). Checking the estimated-share by month confirms why this matters: month 4 (April) is 6.4% estimated vs. 0.0-0.3% for every other month — an artifact of the reference date, not a real April spike in review activity. Year-level analysis (the yearly-trend subsections above) isn't affected by this because it pools 12 months together, but month-level granularity is, so we deliberately drop back to the pre-recovery, actually-parsed-only subset (99.4% of social reviews, n=501,162) for this section.
2. `pool_season` (outdoor = Apr-Sep, indoor = Oct-Mar) is a **simplifying assumption**: we only know when a review was *posted*, not when the visit happened, and there's no indoor/outdoor venue flag in the data — a review posted in October could describe a summer visit, or an indoor pool visited in October. Treat this as a coarse proxy, not a validated visit-season label.


In [ ]:
# select only the actually-parsed (non-estimated) timestamps for the month/season analysis, since the
# estimated timestamps are only accurate to the day and we want to avoid any bias from the
ts_social_actual = corpus.loc[
    social_mask & corpus["review_timestamp_est"].notna() & ~corpus["timestamp_is_estimated"]
].copy()

ts_social_actual["review_month"] = ts_social_actual["review_timestamp_est"].dt.month
ts_social_actual["review_year"] = ts_social_actual["review_timestamp_est"].dt.year

SEASON_MAP = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"}
ts_social_actual["meteorological_season"] = ts_social_actual["review_month"].map(SEASON_MAP)

# --- outdoor vs indoor pool season (May-September vs October-April) ---
POOL_SEASON_MAP = {m: ("outdoor season" if 5 <= m <= 9 else "indoor season") for m in range(1, 13)}
ts_social_actual["pool_season"] = ts_social_actual["review_month"].map(POOL_SEASON_MAP)

print(f"actually-parsed (non-estimated) social reviews with a timestamp: {len(ts_social_actual):,} "
      f"({len(ts_social_actual) / social_mask.sum():.1%} of all social reviews)")
print(f"actually-parsed (non-estimated) racialised reviews with a timestamp, racialised=yes: "
      f"{ts_social_actual['racialised'].eq('yes').sum()}"
      f"({ts_social_actual['racialised'].eq('yes').sum() / corpus['racialised'].eq('yes').sum():.1%} of all social reviews)")


In [ ]:
# ============================================================
# SECTION 0 — SETUP: parse timestamps, define season mappings
# ============================================================
ts_social_actual["review_month"] = ts_social_actual["review_timestamp_est"].dt.month
ts_social_actual["review_year"] = ts_social_actual["review_timestamp_est"].dt.year

SEASON_MAP = {12: "Winter", 1: "Winter", 2: "Winter", 3: "Spring", 4: "Spring", 5: "Spring",
              6: "Summer", 7: "Summer", 8: "Summer", 9: "Autumn", 10: "Autumn", 11: "Autumn"}
ts_social_actual["meteorological_season"] = ts_social_actual["review_month"].map(SEASON_MAP)

def get_pool_season_year(row):
    """Indoor season spans Oct-Apr across a year boundary, so Jan-Apr belongs to
    the indoor block that started the PREVIOUS October, not a fresh one."""
    month, year = row["review_month"], row["review_year"]
    if 5 <= month <= 9:
        return year, "outdoor"
    elif month in (10, 11, 12):
        return year, "indoor"
    else:  # Jan-Apr
        return year - 1, "indoor"

ts_social_actual[["pool_year", "pool_season"]] = ts_social_actual.apply(
    lambda r: pd.Series(get_pool_season_year(r)), axis=1
)

CUTOFF_YEAR = 2016
SEASON_ORDER = ["Winter", "Spring", "Summer", "Autumn"]
THIN_THRESHOLD = 30

# ============================================================
# SECTION 0b — REUSABLE HELPERS
# ============================================================
def add_wilson_ci(df, k_col, n_col):
    rates = df.apply(
        lambda r: pd.Series(wilson(r[k_col], r[n_col]), index=["rate", "ci_lo", "ci_hi"]), axis=1
    )
    return df.join(rates)

def draw_ticks_and_gridlines(ax_top, ax_bottom, tick_positions, tick_labels):
    """Forces tick visibility + manual tick marks + vertical guide lines through both panels."""
    ax_bottom.set_xticks(tick_positions)
    ax_bottom.set_xticklabels(tick_labels, rotation=45, ha="right")
    ax_bottom.tick_params(axis="x", which="major", bottom=True, top=False,
                           direction="out", length=8, width=1.2, color="black")
    ymin, ymax = ax_bottom.get_ylim()
    tick_len = (ymax - ymin) * 0.04
    for pos in tick_positions:
        ax_bottom.plot([pos, pos], [ymin - tick_len, ymin], color="black", linewidth=1.2,
                        clip_on=False, zorder=10)
        ax_top.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)
        ax_bottom.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)

def trim_trailing_thin(df, n_col, threshold=THIN_THRESHOLD):
    """Drops trailing bucket(s) below threshold (usually an incomplete in-progress period)."""
    n_trim = 0
    while df[n_col].iloc[-(n_trim + 1)] < threshold:
        n_trim += 1
    if n_trim > 0:
        print(f"dropping {n_trim} trailing thin bucket(s):")
        print(df.iloc[-n_trim:])
        df = df.iloc[:-n_trim].copy()
    return df

def two_panel_trend_plot(df, rate_col, ci_lo_col, ci_hi_col, n_col, x_col, label_col,
                          title, ylabel_top="racialised rate", ylabel_bottom="n reviews",
                          xlabel_bottom="", color="#C44E52", n_color="#4C72B0",
                          ylim_top=None, figsize=(15, 7.5), max_ticks=20):
    fig, (ax_rate, ax_n) = plt.subplots(2, 1, figsize=figsize, gridspec_kw={"height_ratios": [2.5, 1]})

    yerr_lo = (df[rate_col] - df[ci_lo_col]).clip(lower=0)
    yerr_hi = (df[ci_hi_col] - df[rate_col]).clip(lower=0)
    ax_rate.errorbar(df[x_col], df[rate_col], yerr=[yerr_lo, yerr_hi],
                      fmt="o-", color=color, ecolor=color, capsize=3, markersize=4, linewidth=1.2)
    ax_rate.set_ylabel(ylabel_top)
    ax_rate.set_title(title)
    if ylim_top:
        ax_rate.set_ylim(*ylim_top)
    ax_rate.grid(True, alpha=0.3)

    ax_n.bar(df[x_col], df[n_col], color=n_color, width=0.8)
    ax_n.set_ylabel(ylabel_bottom)
    ax_n.set_xlabel(xlabel_bottom)

    step = max(1, len(df) // max_ticks)
    tick_pos = df[x_col].iloc[::step].tolist()
    tick_labels = df[label_col].iloc[::step].tolist()
    draw_ticks_and_gridlines(ax_rate, ax_n, tick_pos, tick_labels)

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.2, hspace=0.15)
    plt.savefig(FIGURES_DIR / f"trend_{slugify(title)}.png", dpi=150, bbox_inches="tight")
    plt.show()

In [ ]:
# ============================================================
# SECTION 1 — ALL-YEARS POOLED: by calendar month & by season
# ============================================================
ts_social_actual=ts_social_actual[ts_social_actual["review_year"] < 2026].copy()  # drop partial current year
by_month_pooled = ts_social_actual.groupby("review_month").agg(
    social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
)
by_month_pooled = add_wilson_ci(by_month_pooled, "racialised_reviews", "social_reviews")
by_season_pooled = ts_social_actual.groupby("meteorological_season").agg(
    social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
).reindex(SEASON_ORDER)
by_season_pooled = add_wilson_ci(by_season_pooled, "racialised_reviews", "social_reviews")

print("By calendar month (all years pooled):")
print(by_month_pooled)
print("\nBy meteorological season (all years pooled):")
print(by_season_pooled)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].errorbar(
    by_month_pooled.index, by_month_pooled["rate"],
    yerr=[by_month_pooled["rate"] - by_month_pooled["ci_lo"], by_month_pooled["ci_hi"] - by_month_pooled["rate"]],
    fmt="o-", color="#C44E52", ecolor="#C44E52", capsize=3,
)
axes[0].set_xlabel("calendar month")
axes[0].set_ylabel("racialised rate")
axes[0].set_title(f"By month, all years pooled (n={len(ts_social_actual):,})")
axes[0].set_xticks(range(1, 13))

axes[1].bar(
    SEASON_ORDER, by_season_pooled.loc[SEASON_ORDER, "rate"], color="#4C72B0",
    yerr=[by_season_pooled.loc[SEASON_ORDER, "rate"] - by_season_pooled.loc[SEASON_ORDER, "ci_lo"],
          by_season_pooled.loc[SEASON_ORDER, "ci_hi"] - by_season_pooled.loc[SEASON_ORDER, "rate"]],
    capsize=4,
)
axes[1].set_ylabel("racialised rate")
axes[1].set_title("By meteorological season, all years pooled")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "racialised_rate_by_month_and_season_pooled.png", dpi=150, bbox_inches="tight")
plt.show()

### what is the trend  over seasons  ( summer vs other seasons) and the indor and outdoor seasons?

In [ ]:
# ============================================================
# SECTION 3 — SEASON TREND, CHRONOLOGICAL (post-2016, one season per point)
# ============================================================
ts_post2016 = ts_social_actual[ts_social_actual["review_year"] >= CUTOFF_YEAR].copy()

SEASON_MONTH_ORDER = {"Winter": 0, "Spring": 1, "Summer": 2, "Autumn": 3}

# ============================================================
# SECTION 3b — SUMMER vs OTHER MONTHS, CHRONOLOGICAL (post-2016)
# ============================================================
ts_post2016["summer_vs_other"] = np.where(
    ts_post2016["meteorological_season"] == "Summer", "Summer", "Other months"
)

summer_chrono = ts_post2016.groupby(["review_year", "summer_vs_other"]).agg(
    social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
)
summer_chrono = add_wilson_ci(summer_chrono, "racialised_reviews", "social_reviews").reset_index()

# order: Summer first, then Other months, within each year — pick whichever reads better
GROUP_ORDER = {"Summer": 0, "Other months": 1}
summer_chrono["group_order"] = summer_chrono["summer_vs_other"].map(GROUP_ORDER)
summer_chrono = summer_chrono.sort_values(["review_year", "group_order"]).reset_index(drop=True)
summer_chrono["label"] = summer_chrono["review_year"].astype(str) + "-" + summer_chrono["summer_vs_other"]
summer_chrono["x_pos"] = range(len(summer_chrono))

print(summer_chrono[["label", "social_reviews", "rate", "ci_lo", "ci_hi"]].to_string())

two_panel_trend_plot(
    summer_chrono, rate_col="rate", ci_lo_col="ci_lo", ci_hi_col="ci_hi",
    n_col="social_reviews", x_col="x_pos", label_col="label",
    title=f"Racialised rate: Summer vs Other months, chronological (post-{CUTOFF_YEAR})",
    xlabel_bottom="year-group", color="#C44E52", figsize=(16, 6.5),
)

In [ ]:
# ============================================================
# SECTION 4 — INDOOR/OUTDOOR TREND, CHRONOLOGICAL (post-2016, corrected boundary)
# ============================================================
def make_pool_bucket(row):
    if row["pool_year"] < CUTOFF_YEAR:
        return ("pre-2016", "pool")
    return (str(row["pool_year"]), row["pool_season"])

ts_social_actual["pool_bucket_key"] = ts_social_actual.apply(make_pool_bucket, axis=1)

pool_chrono = ts_social_actual.groupby("pool_bucket_key").agg(
    social_reviews=("racialised", "size"), racialised_reviews=("racialised", lambda s: (s == "yes").sum())
).reset_index()
pool_chrono = add_wilson_ci(pool_chrono, "racialised_reviews", "social_reviews")

pool_chrono["year_part"] = pool_chrono["pool_bucket_key"].apply(lambda k: k[0])
pool_chrono["season_part"] = pool_chrono["pool_bucket_key"].apply(lambda k: k[1])
pool_chrono["sort_year"] = pool_chrono["year_part"].apply(lambda y: -1 if y == "pre-2016" else int(y))
pool_chrono["sort_season"] = pool_chrono["season_part"].apply(lambda s: 0 if s in ("outdoor", "pool") else 1)
pool_chrono = pool_chrono.sort_values(["sort_year", "sort_season"]).reset_index(drop=True)

pool_chrono["label"] = pool_chrono.apply(
    lambda r: "pre-2016" if r["year_part"] == "pre-2016"
    else (f"{r['year_part']}-outdoor" if r["season_part"] == "outdoor"
          else f"{r['year_part']}/{int(r['year_part'])+1}-indoor"),
    axis=1
)
pool_chrono = trim_trailing_thin(pool_chrono, "social_reviews")
pool_chrono["x_pos"] = range(len(pool_chrono))

two_panel_trend_plot(
    pool_chrono, rate_col="rate", ci_lo_col="ci_lo", ci_hi_col="ci_hi",
    n_col="social_reviews", x_col="x_pos", label_col="label",
    title=f"Racialised rate by indoor/outdoor season, chronological (post-{CUTOFF_YEAR}, corrected)",
    xlabel_bottom="season block (chronological)", color="#4C72B0", n_color="#55A868",
    figsize=(16, 6.5),
)

In [ ]:
corpus["review_year_est"] = corpus["review_timestamp_est"].dt.year
ts_all = corpus.loc[social_mask & corpus["review_timestamp_est"].notna()].copy()
ts_all["review_year_est"] = ts_all["review_timestamp_est"].dt.year
ts_all["is_racialised"] = (ts_all["racialised"] == "yes").astype(int)

# --- trend test 1: full 2016-2025 window ---
trend_df = ts_all[ts_all["review_year_est"].between(2016, 2025)].copy()
trend_df["year_c"] = trend_df["review_year_est"] - trend_df["review_year_est"].mean()
model_full = smf.logit("is_racialised ~ year_c", data=trend_df).fit(disp=0)
coef_full, p_full = model_full.params["year_c"], model_full.pvalues["year_c"]
print(f"2016-2025 trend: log-odds/year = {coef_full:.4f}, p = {p_full:.2e}")

# --- trend test 2: post-2021 window specifically ---
post_df = ts_all[ts_all["review_year_est"].between(2021, 2025)].copy()
post_df["year_c"] = post_df["review_year_est"] - post_df["review_year_est"].mean()
model_post = smf.logit("is_racialised ~ year_c", data=post_df).fit(disp=0)
coef_post, p_post = model_post.params["year_c"], model_post.pvalues["year_c"]
print(f"2021-2025 trend: log-odds/year = {coef_post:.4f}, p = {p_post:.2e}  "
      f"({coef_post / coef_full:.1f}x steeper than the full-period slope)")

# --- simple, easily-quotable endpoint test: 2021 vs 2025 ---
n2021, k2021 = (ts_all["review_year_est"] == 2021).sum(), ((ts_all["review_year_est"] == 2021) & (ts_all["is_racialised"] == 1)).sum()
n2025, k2025 = (ts_all["review_year_est"] == 2025).sum(), ((ts_all["review_year_est"] == 2025) & (ts_all["is_racialised"] == 1)).sum()
z, p_z = proportions_ztest([k2021, k2025], [n2021, n2025])
print(f"\n2021 rate: {k2021:,}/{n2021:,} = {k2021/n2021:.3%}   2025 rate: {k2025:,}/{n2025:,} = {k2025/n2025:.3%}")
print(f"two-proportion z-test, 2021 vs. 2025: z={z:.2f}, p={p_z:.2e}")

# --- indoor/outdoor and summer/other, pooled across all years (chi-square) ---
ts_social_actual["is_racialised"] = (ts_social_actual["racialised"] == "yes").astype(int)
ts_social_actual["summer_vs_other"] = np.where(ts_social_actual["meteorological_season"] == "Summer", "Summer", "Other")

for label, col in [("indoor/outdoor", "pool_season"), ("summer/other", "summer_vs_other")]:
    ct = pd.crosstab(ts_social_actual[col], ts_social_actual["is_racialised"])
    chi2, p, dof, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    rates = (ct[1] / ct.sum(axis=1) * 100).round(2)
    print(f"\n{label}: chi2={chi2:.1f}, dof={dof}, p={p:.2e}, Cramer's V={v:.4f}")
    print(rates)

In [ ]:
ts_social_actual["review_year_est"] = ts_social_actual["review_timestamp_est"].dt.year
trend_window = ts_social_actual[ts_social_actual["review_year_est"].between(2016, 2025)].copy()
trend_window["year_c"] = trend_window["review_year_est"] - trend_window["review_year_est"].mean()

print("--- indoor/outdoor: trend within each season, separately ---")
for season in ["indoor", "outdoor"]:
    sub = trend_window[trend_window["pool_season"] == season]
    m = smf.logit("is_racialised ~ year_c", data=sub).fit(disp=0)
    print(f"{season:8s}: n={len(sub):,}, year coef={m.params['year_c']:.4f}, p={m.pvalues['year_c']:.2e}")

m_int = smf.logit("is_racialised ~ year_c * C(pool_season)", data=trend_window).fit(disp=0)
inter_coef, inter_p = m_int.params["year_c:C(pool_season)[T.outdoor]"], m_int.pvalues["year_c:C(pool_season)[T.outdoor]"]
print(f"interaction (does outdoor's trend differ from indoor's?): coef={inter_coef:.4f}, p={inter_p:.2e}")

print("\n--- summer/other: trend within each group, separately ---")
for grp in ["Summer", "Other"]:
    sub = trend_window[trend_window["summer_vs_other"] == grp]
    m = smf.logit("is_racialised ~ year_c", data=sub).fit(disp=0)
    print(f"{grp:8s}: n={len(sub):,}, year coef={m.params['year_c']:.4f}, p={m.pvalues['year_c']:.2e}")

m_int2 = smf.logit("is_racialised ~ year_c * C(summer_vs_other)", data=trend_window).fit(disp=0)
inter_coef2, inter_p2 = m_int2.params["year_c:C(summer_vs_other)[T.Summer]"], m_int2.pvalues["year_c:C(summer_vs_other)[T.Summer]"]
print(f"interaction (does Summer's trend differ from Other's?): coef={inter_coef2:.4f}, p={inter_p2:.2e}")


**Takeaway -- yes to both: each season is independently trending upward, and the trends are significantly different from each other, not just two positive slopes that happen to differ.**

- **Indoor/outdoor:** both trend up significantly (indoor coef=0.0368/year, p=4.2e-06; outdoor coef=0.1049/year, p=1.3e-64, **~2.9x steeper**). The `year x season` interaction itself is significant (coef=0.0681, p=1.5e-11) -- outdoor's faster rise is a real, distinct effect, not two coincidentally-different slopes.
- **Summer/other:** same shape -- Summer coef=0.1134/year (p=4.8e-59) is **~2.4x steeper** than Other coef=0.0476/year (p=3.1e-12); interaction confirms it's real (coef=0.0658, p=1.7e-11).
- **Reading this alongside the pooled test above:** that earlier chi-square (outdoor 1.80% vs. indoor 1.19%, averaged over all years) shows seasons differ *on average*; this shows *why* -- both ride the same upward trend, but outdoor/summer is accelerating faster, which itself drives the widening gap.

### A third angle: is the indoor/outdoor (and summer/other) gap significant *within each individual year*, and is that itself consistent over time?

The interaction test above is a single number summarizing "does the gap grow over the whole 2016-2025 window" -- it can be significant even if the gap actually appeared abruptly at some point rather than growing smoothly throughout. The more granular version: test indoor vs. outdoor (and Summer vs. Other) **separately within each year**, so we can see not just *whether* there's an overall interaction but *when* the within-year gap actually became real.

In [ ]:
def per_year_season_test(df, group_col, groups, year_col="review_year_est"):
    rows = []
    for yr, g in df.groupby(year_col):
        ct = pd.crosstab(g[group_col], g["is_racialised"])
        if ct.shape != (2, 2):
            continue
        chi2, p, dof, _ = stats.chi2_contingency(ct)
        n = ct.values.sum()
        v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
        rate_a = ct.loc[groups[0], 1] / ct.loc[groups[0]].sum() * 100
        rate_b = ct.loc[groups[1], 1] / ct.loc[groups[1]].sum() * 100
        rows.append({
            "year": yr, f"{groups[0]}_rate": rate_a, f"{groups[1]}_rate": rate_b,
            f"n_{groups[0]}": ct.loc[groups[0]].sum(), f"n_{groups[1]}": ct.loc[groups[1]].sum(),
            "chi2": chi2, "p": p, "V": v,
        })
    return pd.DataFrame(rows)

io_yearly = per_year_season_test(trend_window, "pool_season", ["indoor", "outdoor"])
print("Indoor vs. outdoor, tested separately within each year:")
print(io_yearly.round(4).to_string(index=False))
print(f"\nyears with a significant (p<0.05) within-year indoor/outdoor gap: {io_yearly.loc[io_yearly['p']<0.05,'year'].tolist()}")

so_yearly = per_year_season_test(trend_window, "summer_vs_other", ["Summer", "Other"])
print("\n\nSummer vs. Other, tested separately within each year:")
print(so_yearly.round(4).to_string(index=False))
print(f"\nyears with a significant (p<0.05) within-year Summer/Other gap: {so_yearly.loc[so_yearly['p']<0.05,'year'].tolist()}")

fig1, ax1 = plt.subplots(figsize=(12, 12))
ax1.plot(io_yearly["year"], io_yearly["outdoor_rate"], marker="o", color="#C44E52", label="outdoor", linewidth=2)
ax1.plot(io_yearly["year"], io_yearly["indoor_rate"], marker="o", color="#4C72B0", label="indoor", linewidth=2)
sig_years_io = io_yearly.loc[io_yearly["p"] < 0.05, "year"]
for yr in sig_years_io:
    ax1.axvspan(yr - 0.4, yr + 0.4, color="gray", alpha=0.1)
ax1.set_xlabel("year", fontsize=30, fontweight="bold")
ax1.set_ylabel("racialised rate (%)", fontsize=30, fontweight="bold")
ax1.tick_params(axis="both", labelsize=30, width=2, length=8, direction="out", top=False, right=False)
leg1 = ax1.legend(fontsize=30, edgecolor="black", facecolor="white")
leg1.get_frame().set_linewidth(2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "indoor_outdoor_rate_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

fig2, ax2 = plt.subplots(figsize=(12, 12))
ax2.plot(so_yearly["year"], so_yearly["Summer_rate"], marker="o", color="#C44E52", label="Summer", linewidth=2)
ax2.plot(so_yearly["year"], so_yearly["Other_rate"], marker="o", color="#4C72B0", label="Other months", linewidth=2)
sig_years_so = so_yearly.loc[so_yearly["p"] < 0.05, "year"]
for yr in sig_years_so:
    ax2.axvspan(yr - 0.4, yr + 0.4, color="gray", alpha=0.1)
ax2.set_xlabel("year", fontsize=30, fontweight="bold")
ax2.set_ylabel("racialised rate (%)", fontsize=30, fontweight="bold")
ax2.tick_params(axis="both", labelsize=30, width=2, length=8, direction="out", top=False, right=False)
leg2 = ax2.legend(fontsize=30, edgecolor="black", facecolor="white")
leg2.get_frame().set_linewidth(2)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "summer_other_rate_by_year.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway -- the gap is not a stable, longstanding feature. It is recent, and it has been growing every year since 2022.**

Indoor/outdoor and summer/other show *almost* the same pattern of which years are significant, with one small exception worth naming honestly rather than glossing over. Indoor/outdoor: `[2018, 2019, 2022, 2023, 2024, 2025]` significant, with **2016, 2017, 2020, 2021 all non-significant** (p=0.21-0.94) -- a clean three-phase shape. Summer/other: the same six years significant, **plus 2020** (p=0.040, gap=0.26pp) -- but 2020 is the weakest result in the entire table by both measures (Cramér's V=0.0113, the second-smallest of any year, and a gap noticeably smaller than either the adjoining 2018-2019 opening (0.53-0.59pp) or the 2022+ reopening (0.54pp+)). This reads as a borderline case that happens to clear p<0.05 with a very large sample, not a real break in the three-phase story:

1. **2016-2017**: no detectable seasonal gap at all (indoor/outdoor rates within 0.02-0.4 percentage points of each other, p>0.24).
2. **2018-2019**: a real, significant gap opens up (~0.4-0.6pp, p<0.001) -- then **substantially closes** in 2020-2021: indoor/outdoor is fully non-significant both years (p=0.21, 0.37); summer/other is non-significant in 2021 (p=0.36) and only marginally, weakly significant in 2020 (p=0.040, smallest effect size of any significant year in the table). 2020-2021 is also when overall review volume patterns shifted (visible in the `n_indoor`/`n_outdoor` and `n_Summer`/`n_Other` columns) -- consistent with, though not proof of, a COVID-era disruption to normal pool-going and reviewing patterns temporarily weakening the seasonal signal.
3. **2022 onward**: the gap re-emerges and **widens every single year** -- indoor/outdoor difference grows from 0.61pp (2022) to 1.14pp (2025); Cramér's V climbs steadily (0.0262 -> 0.0405 -> 0.0380 -> 0.0384 for indoor/outdoor; 0.0235 -> 0.0375 -> 0.0400 -> 0.0467 for Summer/other). By 2025, `Summer` sits at 3.05% vs. `Other months`' 1.64% -- essentially double, and both the largest gap and the largest individual rates seen anywhere in this yearly breakdown.

**Put together with the interaction test above**: the significant `year x season` interaction isn't describing a gap that's been slowly, steadily widening since 2016 -- it's really describing this specific post-2021 acceleration. The seasonal effect is a *recent and intensifying* phenomenon, not a stable structural feature of the data, which is an important nuance for how "outdoor/summer pools attract more racialised discourse" should be framed: true now, and increasingly so, but not true throughout the whole observation window. The 2020 summer/other borderline case doesn't change this reading -- it's the weakest result in the table, not a second "gap present" phase.

## Dimension prevalence over time (racialised subset only, N=7,768)

The same year-level lens, but now applied to `theme`, `context`, `connotation`, and `approach` — computed only on the racialised=yes subset, so these are **within-racialised-reviews prevalence shares**, not the overall racialised rate. Per-year N is small before 2018 (2-245 racialised reviews/year, vs. 404-1,566/year from 2018 on) — too thin to trust individual pre-2018 years at this finer grain (splitting an already-small subset by year *and* category multiplies the sparsity problem), even though the extended-window subsection above showed individual years work fine back to 2012 for the *overall rate* (which uses the much larger `social_reviews` denominator). So here we bucket everything before 2018 into a single `pre-2018` row and show 2018-2026 individually, keeping 2026 marked as partial as usual. Month/season breakdowns are skipped for these dimensions: splitting 7,768 already-multi-label reviews by 12 months would leave many month x category cells with single-digit counts, too sparse to plot meaningfully.

In [ ]:
# ============================================================================
# SECTION 0 — SETUP: load racialised subset, multi-label matching, shared helpers
# ============================================================================
import numpy as np
ts_social = ts_social[ts_social["review_year"] < 2026].copy()  # drop partial current year
racialised_ts = ts_social[(ts_social["racialised"] == "yes") & ~ts_social["review_timestamp"].isnull()].copy()
racialised_ts["review_year"] = racialised_ts["review_timestamp_est"].dt.year
racialised_ts["review_month"] = racialised_ts["review_timestamp_est"].dt.month
racialised_ts["meteorological_season"] = racialised_ts["review_month"].map(SEASON_MAP)

def get_pool_season_year(row):
    """Indoor season spans Oct-Apr across a year boundary, so Jan-Apr belongs to
    the indoor block that started the PREVIOUS October, not a fresh one."""
    month, year = row["review_month"], row["review_year"]
    if 5 <= month <= 9:
        return year, "outdoor"
    elif month in (10, 11, 12):
        return year, "indoor"
    else:
        return year - 1, "indoor"

racialised_ts[["pool_year", "pool_season"]] = racialised_ts.apply(
    lambda r: pd.Series(get_pool_season_year(r)), axis=1
)

CUTOFF_YEAR = 2018  # racialised subset (N~5,693-7,768 depending on filters) is much smaller than the overall social set
THIN_THRESHOLD = 30
SEASON_ORDER = ["Winter", "Spring", "Summer", "Autumn"]

CONNOTATION_LABELS = ["Negative", "Neutral", "Positive"]
CONTEXT_LABELS = ["generalised", "incident"]
APPROACH_LABELS = ["Explicit", "Implicit", "Coded"]


def label_match(cell, label):
    """Multi-label-safe matching — a review can carry >1 label per dimension, stored as ndarray."""
    if isinstance(cell, (list, tuple, set, np.ndarray)):
        return label in cell
    if isinstance(cell, str):
        return cell == label
    return False

def prevalence_with_ci_by_group(df, group_col, label_col, labels):
    """Generic: for each value of group_col, compute share (+ Wilson CI) of rows carrying each label."""
    rows = []
    for group_val, g in df.groupby(group_col):
        n = len(g)
        row = {group_col: group_val, "n_reviews": n}
        for lab in labels:
            k = g[label_col].apply(lambda cell: label_match(cell, lab)).sum()
            rate, ci_lo, ci_hi = wilson(k, n)
            row[f"{lab}_rate"] = rate
            row[f"{lab}_ci_lo"] = ci_lo
            row[f"{lab}_ci_hi"] = ci_hi
        rows.append(row)
    return pd.DataFrame(rows).set_index(group_col)

def trim_trailing_thin(df, n_col, threshold=THIN_THRESHOLD):
    n_trim = 0
    while df[n_col].iloc[-(n_trim + 1)] < threshold:
        n_trim += 1
    if n_trim > 0:
        print(f"dropping {n_trim} trailing thin bucket(s):")
        print(df.iloc[-n_trim:])
        df = df.iloc[:-n_trim].copy()
    return df

def draw_ticks_and_gridlines(ax_top, ax_bottom, tick_positions, tick_labels):
    ax_bottom.set_xticks(tick_positions)
    ax_bottom.set_xticklabels(tick_labels, rotation=45, ha="right")
    ax_bottom.tick_params(axis="x", which="major", bottom=True, top=False,
                           direction="out", length=8, width=1.2, color="black")
    ymin, ymax = ax_bottom.get_ylim()
    tick_len = (ymax - ymin) * 0.04
    for pos in tick_positions:
        ax_bottom.plot([pos, pos], [ymin - tick_len, ymin], color="black", linewidth=1.2,
                        clip_on=False, zorder=10)
        ax_top.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)
        ax_bottom.axvline(pos, color="gray", linestyle="--", linewidth=0.6, alpha=0.4, zorder=0)

def plot_prevalence_bands(df, labels, title, colors=None, xlabel="", n_threshold=None,
                           ylim=None, figsize=(15, 7.5), max_ticks=20, n_bar_color="#4C72B0"):
    """One reusable function for every prevalence-over-time plot: line + CI band on top,
    n_reviews bars below. If n_threshold is set, thin buckets are dropped and x_pos renumbered."""
    if n_threshold is not None:
        dropped = df[df["n_reviews"] < n_threshold]
        if len(dropped) > 0:
            print(f"excluding {len(dropped)} bucket(s) with n < {n_threshold}:")
            print(dropped[["label", "n_reviews"]].to_string())
        df = df[df["n_reviews"] >= n_threshold].copy()
        df["x_pos"] = range(len(df))
        title = f"{title} (excluding n < {n_threshold})"

    if colors is None:
        palette = plt.cm.tab10.colors
        colors = {lab: palette[i % len(palette)] for i, lab in enumerate(labels)}

    fig, (ax_rate, ax_n) = plt.subplots(2, 1, figsize=figsize, gridspec_kw={"height_ratios": [2.5, 1]})

    for lab in labels:
        rate, ci_lo, ci_hi = df[f"{lab}_rate"], df[f"{lab}_ci_lo"], df[f"{lab}_ci_hi"]
        ax_rate.plot(df["x_pos"], rate, marker="o", markersize=3, linewidth=1, label=lab, color=colors[lab])
        ax_rate.fill_between(df["x_pos"], ci_lo, ci_hi, color=colors[lab], alpha=0.15, linewidth=0)

    if ylim is None:
        ci_lo_cols = [f"{s}_ci_lo" for s in labels]
        ci_hi_cols = [f"{s}_ci_hi" for s in labels]
        y_floor = max(0, df[ci_lo_cols].min().min() - 0.05)
        y_ceil = min(1.02, df[ci_hi_cols].max().max() + 0.03)
        ylim = (y_floor, y_ceil)

    ax_rate.set_ylabel("share of racialised reviews")
    ax_rate.set_title(title)
    ax_rate.set_ylim(*ylim)
    ax_rate.legend(fontsize=8)
    ax_rate.grid(True, alpha=0.3)

    ax_n.bar(df["x_pos"], df["n_reviews"], color=n_bar_color, width=0.8)
    ax_n.set_ylabel("n racialised reviews")
    ax_n.set_xlabel(xlabel)

    step = max(1, len(df) // max_ticks)
    tick_pos = df["x_pos"].iloc[::step].tolist()
    tick_labels = df["label"].iloc[::step].tolist()
    draw_ticks_and_gridlines(ax_rate, ax_n, tick_pos, tick_labels)

    fig.tight_layout()
    fig.subplots_adjust(bottom=0.2, hspace=0.15)
    plt.savefig(FIGURES_DIR / f"bands_{slugify(title)}.png", dpi=150, bbox_inches="tight")
    plt.show()

# quick sanity check on label storage (run once to confirm assumptions above)
for col, expected_labels in [("connotation", CONNOTATION_LABELS), ("context", CONTEXT_LABELS), ("approach", APPROACH_LABELS)]:
    print(f"\n=== {col} ===")
    print("cell types:", racialised_ts[col].apply(type).value_counts().to_dict())
    print("sample values:", racialised_ts[col].head(3).tolist())

In [ ]:
# ============================================================================
# SECTION 4 — POOL SEASON (indoor/outdoor), chronological, corrected year boundary
# ============================================================================
def make_pool_bucket(row):
    if row["pool_year"] < CUTOFF_YEAR:
        return (f"pre-{CUTOFF_YEAR}", "pool")
    return (str(row["pool_year"]), row["pool_season"])

racialised_ts["pool_bucket_key"] = racialised_ts.apply(make_pool_bucket, axis=1)

n_by_pool = racialised_ts.groupby("pool_bucket_key").size().reset_index(name="n_reviews")
n_by_pool["year_part"] = n_by_pool["pool_bucket_key"].apply(lambda k: k[0])
n_by_pool["season_part"] = n_by_pool["pool_bucket_key"].apply(lambda k: k[1])
n_by_pool["sort_year"] = n_by_pool["year_part"].apply(lambda y: -1 if y == f"pre-{CUTOFF_YEAR}" else int(y))
n_by_pool["sort_season"] = n_by_pool["season_part"].apply(lambda s: 0 if s in ("outdoor", "pool") else 1)
n_by_pool = n_by_pool.sort_values(["sort_year", "sort_season"]).set_index("pool_bucket_key")
n_by_pool["label"] = n_by_pool.apply(
    lambda r: f"pre-{CUTOFF_YEAR}" if r["year_part"] == f"pre-{CUTOFF_YEAR}"
    else (f"{r['year_part']}-outdoor" if r["season_part"] == "outdoor" else f"{r['year_part']}/{int(r['year_part'])+1}-indoor"),
    axis=1
)
n_by_pool = trim_trailing_thin(n_by_pool, "n_reviews")
n_by_pool["x_pos"] = range(len(n_by_pool))

print(f"\npool-season buckets: {len(n_by_pool)} — NOTE small racialised subset means these are likely THIN;")
print(n_by_pool[["label", "n_reviews"]].to_string())

TOP_THEMES = racialised_df["theme"].explode().value_counts().head(5).index.tolist()
pool_connotation = n_by_pool.join(prevalence_with_ci_by_group(racialised_ts, "pool_bucket_key", "connotation", CONNOTATION_LABELS).drop(columns=["n_reviews"]))
pool_context = n_by_pool.join(prevalence_with_ci_by_group(racialised_ts, "pool_bucket_key", "context", CONTEXT_LABELS).drop(columns=["n_reviews"]))
pool_approach = n_by_pool.join(prevalence_with_ci_by_group(racialised_ts, "pool_bucket_key", "approach", APPROACH_LABELS).drop(columns=["n_reviews"]))
pool_theme = n_by_pool.join(prevalence_with_ci_by_group(racialised_ts, "pool_bucket_key", "theme", TOP_THEMES).drop(columns=["n_reviews"]))

plot_prevalence_bands(pool_connotation, CONNOTATION_LABELS,
                       f"Connotation prevalence by indoor/outdoor season (racialised subset, N={len(racialised_ts):,})",
                       colors={"Negative": "#C44E52", "Neutral": "#DD8452", "Positive": "#55A868"},
                       xlabel="season block (chronological)", ylim=(0, 1), figsize=(16, 6.5))
plot_prevalence_bands(pool_context, CONTEXT_LABELS,
                       f"Context prevalence by indoor/outdoor season (racialised subset, N={len(racialised_ts):,})",
                       colors={"generalised": "#4C72B0", "incident": "#DD8452"},
                       xlabel="season block (chronological)", ylim=(0, 1), figsize=(16, 6.5))
plot_prevalence_bands(pool_approach, APPROACH_LABELS,
                       f"Approach prevalence by indoor/outdoor season (racialised subset, N={len(racialised_ts):,})",
                       colors={"Explicit": "#55A868", "Implicit": "#DD8452", "Coded": "#8172B2"},
                       xlabel="season block (chronological)", ylim=(0, 1), figsize=(16, 6.5))
plot_prevalence_bands(pool_theme, TOP_THEMES,
                       f"Top {TOP_THEMES} themes prevalence by indoor/outdoor season (racialised subset, N={len(racialised_ts):,})",
                       xlabel="season block (chronological)", ylim=(0, 1), figsize=(16, 6.5))

In [ ]:
# ============================================================================
# SECTION 6 — SUMMER vs OTHER MONTHS, chronological
# ============================================================================

racialised_post_cutoff = racialised_ts[racialised_ts["review_year"] >= CUTOFF_YEAR].copy()
SEASON_MONTH_ORDER = {"Winter": 0, "Spring": 1, "Summer": 2, "Autumn": 3}

n_by_season = racialised_post_cutoff.groupby(["review_year", "meteorological_season"]).size().reset_index(name="n_reviews")
n_by_season["season_order"] = n_by_season["meteorological_season"].map(SEASON_MONTH_ORDER)
n_by_season = n_by_season.sort_values(["review_year", "season_order"]).reset_index(drop=True)
n_by_season["label"] = n_by_season["review_year"].astype(str) + "-" + n_by_season["meteorological_season"]
n_by_season["season_bucket_key"] = list(zip(n_by_season["review_year"], n_by_season["meteorological_season"]))
n_by_season = n_by_season.set_index("season_bucket_key")
n_by_season["x_pos"] = range(len(n_by_season))

print(f"\nseason buckets: {len(n_by_season)} — check n_reviews for thin early-year cells:")
print(n_by_season[["label", "n_reviews"]].to_string())


racialised_post_cutoff["summer_vs_other"] = np.where(
    racialised_post_cutoff["meteorological_season"] == "Summer",  "Summer", "Other months"
)

n_by_summer = racialised_post_cutoff.groupby(["review_year", "summer_vs_other"]).size().reset_index(name="n_reviews")
GROUP_ORDER = {"Summer": 0, "Other months": 1}
n_by_summer["group_order"] = n_by_summer["summer_vs_other"].map(GROUP_ORDER)
n_by_summer = n_by_summer.sort_values(["review_year", "group_order"]).reset_index(drop=True)
n_by_summer["label"] = n_by_summer["review_year"].astype(str) + "-" + n_by_summer["summer_vs_other"]
n_by_summer["summer_bucket_key"] = list(zip(n_by_summer["review_year"], n_by_summer["summer_vs_other"]))
n_by_summer = n_by_summer.set_index("summer_bucket_key")
n_by_summer["x_pos"] = range(len(n_by_summer))

print(f"\nsummer-vs-other buckets: {len(n_by_summer)}")
print(n_by_summer[["label", "n_reviews"]].to_string())

racialised_post_cutoff["summer_bucket_key"] = list(zip(racialised_post_cutoff["review_year"], racialised_post_cutoff["summer_vs_other"]))

summer_connotation = n_by_summer.join(prevalence_with_ci_by_group(racialised_post_cutoff, "summer_bucket_key", "connotation", CONNOTATION_LABELS).drop(columns=["n_reviews"]))
summer_context = n_by_summer.join(prevalence_with_ci_by_group(racialised_post_cutoff, "summer_bucket_key", "context", CONTEXT_LABELS).drop(columns=["n_reviews"]))
summer_approach = n_by_summer.join(prevalence_with_ci_by_group(racialised_post_cutoff, "summer_bucket_key", "approach", APPROACH_LABELS).drop(columns=["n_reviews"]))
summer_theme = n_by_summer.join(prevalence_with_ci_by_group(racialised_post_cutoff, "summer_bucket_key", "theme", TOP_THEMES).drop(columns=["n_reviews"]))

plot_prevalence_bands(summer_connotation, CONNOTATION_LABELS,
                       f"Connotation prevalence: Summer vs Other months (racialised subset, post-{CUTOFF_YEAR})",
                       colors={"Negative": "#C44E52", "Neutral": "#DD8452", "Positive": "#55A868"},
                       xlabel="year-group", ylim=(0, 1), figsize=(16, 6.5))
plot_prevalence_bands(summer_context, CONTEXT_LABELS,
                       f"Context prevalence: Summer vs Other months (racialised subset, post-{CUTOFF_YEAR})",
                       colors={"generalised": "#4C72B0", "incident": "#DD8452"},
                       xlabel="year-group", ylim=(0, 1), figsize=(16, 6.5))
plot_prevalence_bands(summer_approach, APPROACH_LABELS,
                       f"Approach prevalence: Summer vs Other months (racialised subset, post-{CUTOFF_YEAR})",
                       colors={"Explicit": "#55A868", "Implicit": "#DD8452", "Coded": "#8172B2"},
                       xlabel="year-group", ylim=(0, 1), figsize=(16, 6.5))
plot_prevalence_bands(summer_theme, TOP_THEMES,
                       f"Top {TOP_THEMES} themes prevalence: Summer vs Other months (racialised subset, post-{CUTOFF_YEAR})",
                       xlabel="year-group", ylim=(0, 1), figsize=(16, 6.5))

In [ ]:
rdf = corpus.loc[corpus["racialised"] == "yes"].copy()
rdf = rdf[rdf["review_timestamp"].notna()].copy()  # actual, non-estimated timestamps only
rdf["review_year"] = rdf["review_timestamp_est"].dt.year
rdf["review_month"] = rdf["review_timestamp_est"].dt.month
rdf["meteorological_season"] = rdf["review_month"].map(SEASON_MAP)
rdf["pool_season"] = np.where(rdf["review_month"].between(5, 9), "outdoor", "indoor")
rdf["summer_vs_other"] = np.where(rdf["meteorological_season"] == "Summer", "Summer", "Other")
print(f"racialised, actual-timestamp subset for these tests: {len(rdf):,}")

def label_match(cell, label):
    return label in cell if isinstance(cell, (list, tuple, set, np.ndarray)) else cell == label

DIM_LABELS = {
    "theme": TOP_THEMES,  # top-5, as established earlier in this section
    "context": ["generalised", "incident"],
    "connotation": ["Negative", "Neutral", "Positive"],
    "approach": ["Explicit", "Implicit", "Coded"],
}

post_df = rdf[rdf["review_year"].between(2021, 2025)].copy()
post_df["year_c"] = post_df["review_year"] - post_df["review_year"].mean()

records = []
for dim, labels in DIM_LABELS.items():
    for lab in labels:
        rdf[f"_has_{lab}"] = rdf[dim].apply(lambda c: label_match(c, lab)).astype(int)
        post_df[f"_has_{lab}"] = post_df[dim].apply(lambda c: label_match(c, lab)).astype(int)

        m = smf.logit(f"_has_{lab} ~ year_c", data=post_df).fit(disp=0)
        trend_coef, trend_p = m.params["year_c"], m.pvalues["year_c"]

        ct1 = pd.crosstab(rdf["pool_season"], rdf[f"_has_{lab}"])
        chi1, p1, _, _ = stats.chi2_contingency(ct1)
        v1 = np.sqrt(chi1 / (ct1.values.sum() * (min(ct1.shape) - 1)))

        ct2 = pd.crosstab(rdf["summer_vs_other"], rdf[f"_has_{lab}"])
        chi2_, p2, _, _ = stats.chi2_contingency(ct2)
        v2 = np.sqrt(chi2_ / (ct2.values.sum() * (min(ct2.shape) - 1)))

        records.append({
            "dim": dim, "label": lab,
            "trend_coef_2021_25": trend_coef, "trend_p": trend_p,
            "indoor_pct": ct1.loc["indoor", 1] / ct1.loc["indoor"].sum() * 100,
            "outdoor_pct": ct1.loc["outdoor", 1] / ct1.loc["outdoor"].sum() * 100,
            "io_p": p1, "io_V": v1,
            "summer_pct": ct2.loc["Summer", 1] / ct2.loc["Summer"].sum() * 100,
            "other_pct": ct2.loc["Other", 1] / ct2.loc["Other"].sum() * 100,
            "so_p": p2, "so_V": v2,
        })

sig_table = pd.DataFrame(records)
pd.set_option("display.width", 160)
print(sig_table.round(4).to_string(index=False))

print("\nsignificant (p<0.05) post-2021 trend, sorted by |coefficient| (steepest first):")
print(sig_table[sig_table["trend_p"] < 0.05].sort_values("trend_coef_2021_25", key=abs, ascending=False)
      [["dim", "label", "trend_coef_2021_25", "trend_p"]].to_string(index=False))

**Takeaway.** There is a visible, not-flat pattern: summer months (Jun/Jul/Aug: 1.8-2.0%) run clearly above winter/spring/autumn (1.2-1.3%), and the meteorological-season cut confirms it — Summer sits at **1.92%** vs. 1.21-1.29% for the other three seasons. The simplified `pool_season` split points the same way (outdoor season ~1.80% vs. indoor season ~1.19%). This is plausibly a real seasonal effect — outdoor public pools draw much larger and more diverse crowds in summer, which is exactly the kind of situation the `Crowd`/`Etiquette` themes (see the dimension-content-overview section later in the notebook) describe — though it's still a review-*posting*-date pattern, not a confirmed visit-date one (see the `pool_season` caveat above).

**Note.** `Crowd` (the single largest theme -- see the dimension-content-overview section later) declines steadily from ~60% of racialised reviews in 2018-2019 to ~40% by 2025, while `Staff` and `Clothing` drift upward — a shift in which theme dominates, not just noise around a flat share, though see the cross-dimension check below (§1.4f) before reading too much into it in isolation.

**Note.** `incident`-framed reviews (a specific event, as opposed to a `generalised` complaint) rise steadily from ~27% in 2018-2019 to ~49% by 2025 — a fairly steady trend, not a noisy one.


**Note.** `Negative` connotation share rises from ~85% (2018-2020) to ~92-94% by 2025-2026 — a modest but fairly steady climb, at the expense of both `Neutral` and `Positive`.


**Caveat.** `approach` has near-chance-level human-human inter-annotator agreement (κ≈0-0.09, see `inter_annotator_agreement.ipynb`) — treat any pattern in the approach breakdown below as exploratory, not evidentiary, more so than the other dimensions on this page.


**Note.** `Explicit` share drifts down somewhat (~82% in 2018-2019 to ~75-76% by 2024-2025) with `Implicit` and `Coded` picking up the difference, but the year-to-year movement is noisier than `context`'s or `connotation`'s — consistent with the low-IAA caveat above; treat this one as the least trustworthy of the four series here.



## Formal significance tests: is the trend, and the season effect, real?

Everything above has been shown via Wilson confidence-interval bands or point estimates -- convincing, but not the same as an actual hypothesis test with a p-value. We close WHEN's headline-trend thread with the formal versions of the yearly trend, indoor/outdoor, and summer/other questions raised above, then repeat all three per dimensional label.

**Takeaway -- yes, all three are real, formally significant effects.**

- **Yearly trend:** 2016-2025 is highly significant (coef=0.0824 log-odds/year, p=6.8e-65); restricted to 2021-2025 it's **2.7x steeper** (0.2216, p=5.4e-92) -- a genuine acceleration, not just the same trend zoomed in. Most quotable: **0.77% (2021) -> 2.30% (2025)** (z=-18.16, p=1.2e-73).
- **Indoor vs. outdoor:** significant, chi2=313.8, p=3.3e-70 (rates as above).
- **Summer vs. other:** significant, chi2=358.7, p=5.5e-80 (rates as above).
- **Caveat applying to all three:** Cramér's V for both season tests is tiny (0.025-0.027) -- with over a million reviews, even a trivial association clears significance. The p-values confirm these effects are *real*; the seasonal effect's *size* is modest, while the year-on-year trend is both significant and substantively large (rate roughly tripled).

### The same three tests, per dimensional label -- which ones stand out?

We repeat all three tests -- post-2021 trend, indoor/outdoor, summer/other -- for every `theme` (restricted to the top-5, per the established convention above), `context`, `connotation`, and `approach` label, on the actually-parsed-timestamp racialised subset (N=5,693, same restriction as the month/season analysis above, for the same estimation-artifact reason). This directly answers "are there dimensional labels that stand out": rather than reading 12 separate line charts, we rank every label by effect size across all three tests at once.


**Takeaway -- `Clothing` is still the clear standout, but the significant-label set changed underneath it.**

- **`Clothing` (theme):** one of the largest post-2021 coefficients of all 13 labels tested (+0.212, p=6.5e-12), just behind `connotation`'s `Negative` (+0.216, p=1.8e-09); also the largest effect size on both season tests (summer vs. other V=0.088, the largest in the whole table; indoor vs. outdoor V=0.070, largest among theme labels). All three point the same way: `Clothing`-themed discourse is rising faster than almost any other label since 2021, concentrated in summer/outdoor months (19.4% of summer racialised reviews carry it vs. 12.9% in other months, ~1.5x gap).
- **Beyond `Clothing`:** `Crowd` declining (-0.066, p=0.004); `context`'s `incident`/`generalised` shift continuing (+0.111/-0.097, both p<0.001); `connotation` continuing to polarize toward `Negative` (+0.216) at the expense of `Neutral` (-0.177) and `Positive` (-0.129), all p<0.02. `Belonging`, previously a borderline riser, is **no longer significant** (p=0.084) -- dropped from the standout list rather than repeated uncritically.
- **The one real change: all three `approach` labels are now significant** (`Coded` +0.103 p=0.007, `Implicit` +0.071 p=0.018, `Explicit` -0.070 p=0.009) -- previously none were. Given `approach`'s near-chance human-human agreement (κ≈0-0.09), a newly-significant trend here is more likely the corpus growing large enough to make label noise "significant" than a real shift -- read with *more* caution, not less.
- **Season splits:** `Etiquette` is significantly more prevalent indoors (33.6% vs. 28.6%, p<0.0001, as before), cutting against a naive "crowded summer = more etiquette complaints" assumption; now also `Explicit` approach (80.1% vs. 76.9%, p=0.001), with `Implicit` shifting the other way (p<0.001) -- consistent opposite movement, but read through the low-IAA caveat above.

## Are all constituencies contributing to the post-2021 rise?

One more question the trend above raises directly: is the 2021-2025 increase a broad, nationwide shift, or is it concentrated in a handful of constituencies that simply didn't have any racialised reviews until recently (which would make it look like a national trend while really being a geographic one)? We check this by classifying every constituency by *when* its racialised reviews fall relative to 2021.


In [ ]:
rdf_all_years = corpus.loc[corpus["racialised"] == "yes"].copy()
rdf_all_years["review_year"] = rdf_all_years["review_timestamp_est"].dt.year
print(f"racialised reviews with a resolvable year (actual + estimated): "
      f"{rdf_all_years['review_year'].notna().sum():,} / {len(rdf_all_years):,}")

contrib = rdf_all_years.groupby("constituency_code")["review_year"].agg(
    n="size", n_pre2021=lambda s: (s < 2021).sum(), n_post2021=lambda s: (s >= 2021).sum(),
    min_year="min", max_year="max",
)
contrib["has_pre"] = contrib["n_pre2021"] > 0
contrib["has_post"] = contrib["n_post2021"] > 0

n_zero = 299 - len(contrib)
n_both = ((contrib["has_pre"]) & (contrib["has_post"])).sum()
n_pre_only = ((contrib["has_pre"]) & (~contrib["has_post"])).sum()
n_post_only = ((~contrib["has_pre"]) & (contrib["has_post"])).sum()

print(f"\nconstituencies with zero racialised reviews ever: {n_zero} / 299")
print(f"constituencies with racialised reviews BOTH before and from 2021 onward: {n_both} / 299 ({n_both/299:.1%})")
print(f"constituencies with racialised reviews ONLY before 2021: {n_pre_only}")
print(f"constituencies with racialised reviews ONLY from 2021 onward: {n_post_only}")

# const_agg (built in WHERE) doesn't exist yet at this point in the notebook -- pull
# constituency/state from `political`, already loaded in Section 0, instead
meta_cols = ["constituency", "state"]
political_meta = socio_political.set_index("constituency_code")[meta_cols]
pre_only = contrib[(contrib["has_pre"]) & (~contrib["has_post"])].join(political_meta)
post_only = contrib[(~contrib["has_pre"]) & (contrib["has_post"])].join(political_meta)

print("\nconstituencies with racialised reviews ONLY before 2021:")
print(pre_only[["constituency", "state", "n", "min_year", "max_year"]].to_string())
print("\nconstituencies with racialised reviews ONLY from 2021 onward:")
print(post_only[["constituency", "state", "n", "min_year", "max_year"]].to_string())


**Takeaway -- yes, essentially every constituency contributes to both periods; the rise is nationwide, not a handful of newly-appearing places.**

- **288/299 (96.3%)** constituencies have racialised reviews both before and from 2021 onward.
- **1 constituency** (`Berlin-Pankow`) has reviews exclusively before 2021 -- a single 2020 review, already the thinnest constituency in the corpus.
- **10 constituencies** have reviews only from 2021 onward -- all thin (2-14 reviews), scattered across 7 states with no shared pattern; only 2 (`Lübeck`, `Trier`) clear the `reliable_n` bar.
- **No constituency has zero racialised reviews ever.**

This rules out the obvious concerns: the 2021-2025 rise isn't an artifact of a few constituencies suddenly appearing, and it isn't concentrated in one region -- a broad, essentially universal pattern, which makes the trend test above more trustworthy as a genuine national shift rather than a compositional artifact of *which* constituencies happen to have data in which years.

## how does constituency racialised rate vary for pre 2021 and post 2021

The section above shows the post-2021 rise is nationwide rather than a handful of newly-appearing constituencies -- but *contributing reviews in both periods* isn't the same as *the rate itself moving the same way everywhere*. Here we look directly at how each constituency's own racialised rate changed from pre-2021 to post-2021, restricted to constituencies with enough racialised reviews in *both* windows to compare reliably.


In [ ]:
# ============================================================================
# Constituency racialised rate: pre-2021 vs. post-2021
# ============================================================================
ts_all_years = corpus.loc[social_mask & corpus["review_timestamp_est"].notna()].copy()
ts_all_years["review_year"] = ts_all_years["review_timestamp_est"].dt.year
ts_all_years["period"] = np.where(ts_all_years["review_year"] < 2021, "pre_2021", "post_2021")

period_counts = ts_all_years.groupby(["constituency_code", "period"]).agg(
    social_reviews=("racialised", "size"),
    racialised_reviews=("racialised", lambda s: (s == "yes").sum()),
).reset_index()

rate_pivot = period_counts.pivot(index="constituency_code", columns="period",
                                  values=["social_reviews", "racialised_reviews"])
rate_pivot.columns = ["_".join(c) for c in rate_pivot.columns]
rate_pivot = rate_pivot.fillna(0)

for period in ["pre_2021", "post_2021"]:
    rate_pivot[f"rate_{period}"] = rate_pivot.apply(
        lambda r: wilson(r[f"racialised_reviews_{period}"], r[f"social_reviews_{period}"])[0], axis=1
    )

# corpus already carries constituency/state per row (from the Section 0 spatial join) --
# use that directly rather than `political` (only built later, in WHERE)
const_meta = corpus[["constituency_code", "constituency", "state"]].drop_duplicates(
    subset="constituency_code"
).set_index("constituency_code")
rate_pivot = rate_pivot.join(const_meta)

RELIABLE_N = 10  # same reliable_n bar used throughout WHERE
reliable_change = rate_pivot[
    (rate_pivot["racialised_reviews_pre_2021"] >= RELIABLE_N)
    & (rate_pivot["racialised_reviews_post_2021"] >= RELIABLE_N)
].copy()
reliable_change["rate_change_pp"] = (reliable_change["rate_post_2021"] - reliable_change["rate_pre_2021"]) * 100

print(f"constituencies with >= {RELIABLE_N} racialised reviews in BOTH periods: "
      f"{len(reliable_change)} / 299 ({len(reliable_change) / 299:.1%})")
print(f"mean rate: pre-2021 {reliable_change['rate_pre_2021'].mean():.2%}, "
      f"post-2021 {reliable_change['rate_post_2021'].mean():.2%}")

n_up = (reliable_change["rate_change_pp"] > 0).sum()
n_down = (reliable_change["rate_change_pp"] < 0).sum()
print(f"increased: {n_up} ({n_up / len(reliable_change):.1%}), "
      f"decreased: {n_down} ({n_down / len(reliable_change):.1%})")

wstat, wp = stats.wilcoxon(reliable_change["rate_pre_2021"], reliable_change["rate_post_2021"])
print(f"Wilcoxon signed-rank (paired pre vs. post): stat={wstat:.1f}, p={wp:.3f}")

rho, rho_p = stats.spearmanr(reliable_change["rate_pre_2021"], reliable_change["rate_post_2021"])
print(f"Spearman rank correlation (does pre-2021 rank predict post-2021 rank?): rho={rho:.3f}, p={rho_p:.2e}")

print("\ntop 8 largest increases:")
print(reliable_change.nlargest(8, "rate_change_pp")[
    ["constituency", "state", "rate_pre_2021", "rate_post_2021", "rate_change_pp",
     "racialised_reviews_pre_2021", "racialised_reviews_post_2021"]
].to_string())
print("\ntop 8 largest decreases:")
print(reliable_change.nsmallest(8, "rate_change_pp")[
    ["constituency", "state", "rate_pre_2021", "rate_post_2021", "rate_change_pp",
     "racialised_reviews_pre_2021", "racialised_reviews_post_2021"]
].to_string())

fig, ax = plt.subplots(figsize=(8, 8))
ax.scatter(reliable_change["rate_pre_2021"] * 100, reliable_change["rate_post_2021"] * 100,
           alpha=0.6, color="#4C72B0")
lim = max(reliable_change["rate_pre_2021"].max(), reliable_change["rate_post_2021"].max()) * 100 * 1.1
ax.plot([0, lim], [0, lim], color="gray", linestyle="--", linewidth=1, label="no change")
ax.set_xlabel("pre-2021 racialised rate (%)")
ax.set_ylabel("post-2021 racialised rate (%)")
ax.set_title(f"Constituency racialised rate: pre- vs. post-2021\n"
             f"(n={len(reliable_change)} constituencies with >= {RELIABLE_N} racialised reviews each period)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "constituency_rate_pre_post_2021.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway -- the national rise is real, but it is not existing hotspots getting worse; it is breadth, not intensification.**

- Only **65/299 (21.7%)** constituencies have enough racialised reviews (>=10) in *both* periods to compare reliably -- skewed toward the largest, already-highest-baseline places (Duisburg II, the Berlin constituencies, Cologne).
- Among those 65: **34 increased, 31 decreased**, mean rate essentially flat (2.78% pre vs. 2.76% post); a paired Wilcoxon test finds **no significant shift** (p=0.52).
- This doesn't contradict the headline 0.77%->2.30% national trend -- it narrows *where* it comes from. This 65-constituency set is biased-high by construction (it needs substantial pre-2021 volume, when racialised discourse was rarer overall), so a flat average here is consistent with the rise being driven by the other 234 constituencies, including the 10 that only start appearing from 2021 onward. The rise looks like *more of the country* generating racialised discourse at all, not the same hotspots discoursing more intensely.
- One structural pattern still holds: pre- and post-2021 rates are strongly rank-correlated (**Spearman rho=0.761, p=1.96e-13**) -- a constituency relatively high before 2021 is still very likely relatively high after; not a random reshuffle.
- Biggest movers: `Duisburg II` drops from the single highest pre-2021 rate (10.8%) to 5.9% post -- still among the highest post-2021, i.e. regression toward the mean from an extreme start, not a reversal. `Chemnitz` (+3.0pp), `Lörrach-Müllheim` (+2.4pp), `Berlin-Friedrichshain-Kreuzberg` (+2.2pp) show the largest genuine increases, none of which were pre-2021 outliers -- new constituencies entering the elevated range, not old ones climbing further.

## Constituencies contributing over time, and per-constituency rate trajectories

The pre/post-2021 cut above answers "did already-established constituencies shift" -- it can't say anything about *breadth* (how many places contribute at all, each year) or about *shape* (did a given constituency's own rate climb smoothly, or jump). We check both directly: first, how many distinct constituencies contribute at least one racialised review each calendar year; second, for constituencies with enough data, the actual year-over-year change in rate, ranked by size -- rather than forcing each constituency into a single "gradual vs. sudden" label, which turns out to be unreliable with only a handful of time points per constituency.


In [ ]:
# ============================================================================
# How many distinct constituencies contribute >=1 racialised review each year?
# ============================================================================
rdf_year = corpus.loc[corpus["racialised"] == "yes"].copy()
rdf_year["review_year"] = rdf_year["review_timestamp_est"].dt.year
rdf_year = rdf_year[rdf_year["review_year"].notna() & (rdf_year["review_year"] < 2026)].copy()
rdf_year["review_year"] = rdf_year["review_year"].astype(int)

contributors_per_year = rdf_year.groupby("review_year")["constituency_code"].nunique()
print("contributing constituencies per year (out of 299):")
print(contributors_per_year)

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(contributors_per_year.index.astype(str), contributors_per_year.values, color="#4C72B0")
ax.set_ylabel("constituencies with >= 1 racialised review")
ax.set_xlabel("review year")
ax.set_title("Breadth: how many constituencies contribute any racialised review, per year")
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "constituency_contributors_per_year.png", dpi=150, bbox_inches="tight")
plt.show()

# --- per-constituency-year counts, for the qualifying-set logic below ---
cy_counts = rdf_year.groupby(["constituency_code", "review_year"]).size().rename("n").reset_index()
year_pivot = cy_counts.pivot(index="constituency_code", columns="review_year", values="n").fillna(0).astype(int)

total_n = year_pivot.sum(axis=1)
n_distinct_years = (year_pivot > 0).sum(axis=1)
max_year_share = year_pivot.max(axis=1) / total_n

const_meta = corpus[["constituency_code", "constituency", "state"]].drop_duplicates(
    subset="constituency_code"
).set_index("constituency_code")

const_year_summary = pd.DataFrame({
    "total_n": total_n, "n_distinct_years": n_distinct_years, "max_year_share": max_year_share,
}).join(const_meta)

QUALIFY_MIN_TOTAL = 20
QUALIFY_MIN_YEARS = 4
const_year_summary["qualify"] = (
    (const_year_summary["total_n"] >= QUALIFY_MIN_TOTAL)
    & (const_year_summary["n_distinct_years"] >= QUALIFY_MIN_YEARS)
)
print(f"\nqualify (total >= {QUALIFY_MIN_TOTAL} racialised reviews AND present in "
      f">= {QUALIFY_MIN_YEARS} distinct years): {const_year_summary['qualify'].sum()} / 299")

In [ ]:
# --- for constituencies that DON'T qualify but still have a reasonable total (>=10),
# check their year-by-year spread directly -- is the shortfall due to genuine
# concentration in one year (a spike), or just too few total reviews spread thinly? ---
near_miss = const_year_summary[
    (const_year_summary["total_n"] >= 10) & (~const_year_summary["qualify"])
].sort_values("total_n", ascending=False)

print(f"near-miss (total >= 10 but doesn't qualify): {len(near_miss)}")
print(f"  -- of these, genuinely concentrated (n_distinct_years < {QUALIFY_MIN_YEARS}): "
      f"{(near_miss['n_distinct_years'] < QUALIFY_MIN_YEARS).sum()}")
print(f"  -- of these, spread is fine, just short of the total-{QUALIFY_MIN_TOTAL} bar: "
      f"{(near_miss['n_distinct_years'] >= QUALIFY_MIN_YEARS).sum()}")

print("\nnear-miss summary (sorted by total_n):")
print(near_miss[["constituency", "state", "total_n", "n_distinct_years", "max_year_share"]].to_string())

In [ ]:
# ============================================================================
# Year-over-year rate change per constituency (2018+, so every denominator is
# large enough to trust -- pre-2018 is already flagged sparse elsewhere in this
# notebook). Rather than forcing each constituency into a "gradual"/"sudden"
# label, we compute the actual year-to-year change directly and rank by size.
# ============================================================================
ts_diff = corpus.loc[social_mask & corpus["review_timestamp_est"].notna()].copy()
ts_diff["review_year"] = ts_diff["review_timestamp_est"].dt.year
ts_diff = ts_diff[(ts_diff["review_year"] >= 2018) & (ts_diff["review_year"] < 2026)].copy()
ts_diff["review_year"] = ts_diff["review_year"].astype(int)

qualify_ids = const_year_summary[const_year_summary["qualify"]].index
cy_diff = ts_diff[ts_diff["constituency_code"].isin(qualify_ids)].groupby(
    ["constituency_code", "review_year"]
).agg(
    social_reviews=("racialised", "size"),
    racialised_reviews=("racialised", lambda s: (s == "yes").sum()),
).reset_index()
cy_diff["rate"] = cy_diff["racialised_reviews"] / cy_diff["social_reviews"]

print("social_reviews per constituency-year (2018+, qualifying set only):")
print(cy_diff["social_reviews"].describe())

# only strictly-consecutive-year pairs within a constituency count as one "step"
rows = []
for cid, g in cy_diff.groupby("constituency_code"):
    g = g.sort_values("review_year").reset_index(drop=True)
    for i in range(1, len(g)):
        if g.loc[i, "review_year"] - g.loc[i - 1, "review_year"] == 1:
            rows.append({
                "constituency_code": cid, "year": g.loc[i, "review_year"],
                "rate_prev": g.loc[i - 1, "rate"] * 100, "rate_curr": g.loc[i, "rate"] * 100,
                "diff_pp": (g.loc[i, "rate"] - g.loc[i - 1, "rate"]) * 100,
                "social_prev": g.loc[i - 1, "social_reviews"], "social_curr": g.loc[i, "social_reviews"],
            })
year_diffs = pd.DataFrame(rows).join(const_meta, on="constituency_code")

print(f"\nyear-over-year diff pairs: {len(year_diffs)} (across {year_diffs['constituency_code'].nunique()} "
      f"of {len(qualify_ids)} qualifying constituencies)")
print(year_diffs["diff_pp"].describe())

n_gt2 = (year_diffs["diff_pp"].abs() > 2).sum()
n_gt5 = (year_diffs["diff_pp"].abs() > 5).sum()
print(f"\ntransitions with |change| > 2pp: {n_gt2} ({n_gt2 / len(year_diffs):.1%})")
print(f"transitions with |change| > 5pp: {n_gt5} ({n_gt5 / len(year_diffs):.1%})")
print("\nlarge (|change| > 5pp) transitions, by year and constituency -- checking these aren't all one shock year:")
big_by_year = year_diffs.loc[year_diffs["diff_pp"].abs() > 5].sort_values(["year", "diff_pp"])
for yr, g in big_by_year.groupby("year"):
    print(f"  {yr} ({len(g)}):")
    for _, r in g.iterrows():
        print(f"    {r['constituency']} ({r['state']}): {r['diff_pp']:+.1f}pp")

print("\ntop 12 largest single-year jumps (either direction):")
print(year_diffs.reindex(year_diffs["diff_pp"].abs().sort_values(ascending=False).index).head(12)[
    ["constituency", "state", "year", "rate_prev", "rate_curr", "diff_pp", "social_prev", "social_curr"]
].to_string())

fig, ax = plt.subplots(figsize=(10, 6))
big = year_diffs["diff_pp"].abs() > 5
ax.scatter(year_diffs.loc[~big, "year"], year_diffs.loc[~big, "diff_pp"],
           color="#4C72B0", alpha=0.5, s=25, label="|change| <= 5pp")
ax.scatter(year_diffs.loc[big, "year"], year_diffs.loc[big, "diff_pp"],
           color="#C44E52", alpha=0.85, s=45, label="|change| > 5pp")
ax.axhline(0, color="gray", linestyle="--", linewidth=1)
ax.set_ylabel("year-over-year change in racialised rate (pp)")
ax.set_xlabel("year (change from year-1 to year)")
ax.set_title(f"Year-over-year rate change per constituency\n"
             f"(n={len(qualify_ids)} qualifying constituencies, 2018-2025)")
ax.legend(fontsize=9)
for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)
# label each big (red) point with its constituency, so the plot answers
# "which constituency is this?" directly instead of requiring the printed table
for _, r in year_diffs.loc[big].iterrows():
    ax.annotate(r["constituency"], (r["year"], r["diff_pp"]),
                fontsize=6.5, rotation=30, ha="left", va="bottom" if r["diff_pp"] >= 0 else "top",
                xytext=(3, 2 if r["diff_pp"] >= 0 else -2), textcoords="offset points")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "constituency_rate_yoy_change.png", dpi=150, bbox_inches="tight")
plt.show()

2019: Bochum I -6.1pp, Berlin-Neukölln +8.3pp

2020: Berlin-Neukölln -11.1pp, Bochum I -5.6pp, Recklinghausen I -5.5pp, Augsburg-Land +5.7pp, Duisburg II +5.8pp, Offenburg +6.5pp

2021: Duisburg II -12.4pp, Offenburg -11.5pp, Gelsenkirchen -7.2pp, Berlin-Friedrichshain-Kreuzberg -5.1pp, Mettmann II -5.1pp, Berlin-Mitte +5.2pp, Chemnitz +5.7pp

2022: Duisburg II +5.2pp, Bochum I +6.0pp, Offenburg +6.3pp, Essen II +7.1pp, Berlin-Friedrichshain-Kreuzberg +7.6pp

2023: Neustadt – Speyer +5.3pp

2024: Offenburg -6.1pp, Ludwigshafen/Frankenthal +6.8pp, Essen II +8.2pp, Recklinghausen I +15.0pp

2025: Essen II -9.0pp, Recklinghausen I -6.5pp, Regensburg -6.5pp, Essen III +5.0pp, Chemnitz +5.5pp, Lörrach – Müllheim +5.9pp, Offenburg +6.2pp, Hamburg-Mitte +8.2pp


**Takeaway -- most year-to-year movement is small, but real, sizeable single-year jumps do happen, at solid sample sizes, and they aren't concentrated in one shock year.**

- 2018+, 124 qualifying constituencies -> **868 year-over-year transitions**, all with denominators >=25 social reviews per constituency-year, so no small-N artifacts.
- Typical change is tiny: median **+0.13pp**, middle 50% within about -0.7pp to +1.1pp -- most constituencies, most years, barely move.
- But **22.1% of transitions exceed +/-2pp**, and **3.8% exceed +/-5pp** -- spread across 2019-2025 (2-8 per year), not piled into a single COVID-reopening-style shock year.
- Biggest jumps: `Recklinghausen I` +15.0pp in 2024 (0%->15%, n=59->60); `Duisburg II` -12.4pp in 2021 (14.7%->2.3%, n=184->130 -- the same constituency flagged as the largest pre/post-2021 faller above, now pinned to a specific year); `Berlin-Neukölln` swings both directions (+8.3pp 2019, then -11.1pp 2020); `Essen II` spikes in 2024 (+8.2pp) and gives most of it back in 2025 (-9.0pp), a one-year blip.
- Sharper than the biennium trajectories above: ranking actual year-over-year changes shows large single-year moves are real and not rare (1 in 4 transitions exceeds 2pp), but most are either one-off blips that partly reverse (`Essen II`) or a step coinciding with the already-identified pre/post-2021 shift (`Duisburg II`), not evidence of a distinct "sudden-riser" category.

# WHERE

How the racialised-review rate is distributed across German constituencies: physical
coverage (pools/reviews), the sharper reliability question of how many *racialised* reviews
back each rate, prevalence with confidence intervals, choropleths, venue-level concentration
and volume-vs-rate sensitivity checks, and a geometric border-constituency proxy.

## Coverage: pools and text reviews per constituency and state

We build the constituency-level aggregate table now (`const_agg`) and reuse it for the rest of Part 1 — one canonical table rather than recomputing aggregates ad hoc in each section. Critically, we start from the **full list of 299 constituencies** (not just the ones that happen to appear in the corpus) so constituencies with zero pools show up as zero, not as missing rows — a left join here, not an inner join, is the whole point: it's the only way to see the coverage gap at all.

In [ ]:
# canonical 299-constituency backbone comes from the political CSV (already verified 299 rows,
# consistent naming with the shapefile/join in section 1)
backbone = socio_political[["state", "constituency_code", "constituency"]].copy()

agg = (
    corpus.groupby("constituency_code")
    .agg(
        pool_count=("unique_identifier", "nunique"),
        total_text_reviews=("review_id", "count"),
        social_reviews=("social", lambda s: (s == "yes").sum()),
        racialised_reviews=("racialised", lambda s: (s == "yes").sum()),
    )
    .reset_index()
)

const_agg = backbone.merge(agg, on="constituency_code", how="left")
for col in ["pool_count", "total_text_reviews", "social_reviews", "racialised_reviews"]:
    const_agg[col] = const_agg[col].fillna(0).astype(int)

# pools that ever got a social review -- the correct denominator for "share of pools with a
# racialised review", since racialised reviews are a subset of social ones (a pool with zero
# social reviews can never register as racialised).
social_pools = (
    corpus.loc[corpus["social"] == "yes"]
    .groupby("constituency_code")["unique_identifier"].nunique()
    .rename("pools_with_social")
)
const_agg = const_agg.merge(social_pools, on="constituency_code", how="left")
const_agg["pools_with_social"] = const_agg["pools_with_social"].fillna(0).astype(int)

n_zero_pool = (const_agg["pool_count"] == 0).sum()
n_zero_racialised = (const_agg["racialised_reviews"] == 0).sum()
print(f"constituencies with zero pools: {n_zero_pool} / 299")
print(f"constituencies with zero racialised reviews: {n_zero_racialised} / 299")
print()
print("pool_count per constituency:")
print(const_agg["pool_count"].describe())
print()
print("total_text_reviews per constituency:")
print(const_agg["total_text_reviews"].describe())

In [ ]:
# Coverage thresholds: how many constituencies would be usable at different minimum-N bars?
# This is purely descriptive (physical venue/review coverage) — it does NOT define what counts
# as a "reliable" racialised-rate estimate. That's a distinct question, addressed properly in
# the racialised-reviews-coverage subsection below using social_reviews/racialised_reviews
# directly, since pool_count and total_text_reviews
# are only loose proxies for what actually drives the rate's confidence interval.
for thresh in [1, 2, 5, 10, 20]:
    n = (const_agg["pool_count"] >= thresh).sum()
    print(f"constituencies with >= {thresh:>2} pools: {n:>3} / 299 ({n / 299:.1%})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(const_agg["pool_count"], bins=40, color="#4C72B0")
axes[0].set_title("Pools per constituency")
axes[0].set_xlabel("pool_count")
axes[1].hist(const_agg["total_text_reviews"], bins=40, color="#DD8452")
axes[1].set_title("Text reviews per constituency")
axes[1].set_xlabel("total_text_reviews")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "pools_and_text_reviews_per_constituency_hist.png", dpi=150, bbox_inches="tight")
plt.show()


In [ ]:
# State-level is a natural fallback grain wherever constituency-level is too thin
state_agg = (
    const_agg.groupby("state")
    .agg(
        n_constituencies=("constituency_code", "count"),
        pool_count=("pool_count", "sum"),
        total_text_reviews=("total_text_reviews", "sum"),
        social_reviews=("social_reviews", "sum"),
        racialised_reviews=("racialised_reviews", "sum"),
    )
    .sort_values("pool_count", ascending=False)
)
state_agg


**Takeaway.** Coverage is better than we assumed going in: every one of the 299 constituencies has at least 4 pools and at least 678 text reviews — there are **no true zero-coverage constituencies** anywhere in Germany. Pool coverage in particular is close to a non-issue (every constituency clears even a 4-pool floor), so it is not the binding constraint on what's analytically reliable — that question is about the racialised-review counts specifically, addressed next in the following subsection. At state level, coverage ranges from Bremen (2 constituencies, 25 pools) to Nordrhein-Westfalen (64 constituencies, 809 pools) — city-states and small states will naturally have noisier rate estimates and this should be kept in mind whenever state is used as the unit of comparison.

## Coverage of racialised reviews specifically

The coverage subsection above showed pool/text-review coverage is essentially a non-issue — every constituency clears even a low bar. But that's the wrong thing to gate on: the spatial join already guarantees every constituency has *some* data, and downstream (in the prevalence-with-CIs subsection onward) we're not estimating "does this constituency have pools", we're estimating a **rate** — racialised / social reviews — whose Wilson confidence interval width is driven directly by `social_reviews` (the denominator, i.e. the effective sample size for the proportion) and `racialised_reviews` (the numerator, i.e. how many actual "successes" the estimate rests on). A constituency can have plenty of pools and text reviews and still have a rate estimate resting on only 1-2 racialised reviews, which is a fundamentally different (and worse) reliability problem than "not enough pools." We check that directly here, and **redefine `reliable_n` on that basis** — replacing the pool/text-review-based version from the coverage subsection above, which we now drop since it wasn't measuring the thing that actually matters.

In [ ]:
print("social_reviews per constituency (the rate's denominator):")
print(const_agg["social_reviews"].describe())
print()
print("racialised_reviews per constituency (the rate's numerator):")
print(const_agg["racialised_reviews"].describe())
print()

for thresh in [1, 2, 5, 10, 20]:
    n = (const_agg["racialised_reviews"] >= thresh).sum()
    print(f"constituencies with >= {thresh:>2} racialised reviews: {n:>3} / 299 ({n / 299:.1%})")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(const_agg["social_reviews"], bins=40, color="#55A868")
axes[0].set_title("Social reviews per constituency (rate denominator)")
axes[0].set_xlabel("social_reviews")
axes[1].hist(const_agg["racialised_reviews"], bins=range(0, const_agg["racialised_reviews"].max() + 2), color="#C44E52")
axes[1].set_title("Racialised reviews per constituency (rate numerator)")
axes[1].set_xlabel("racialised_reviews")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "social_and_racialised_reviews_per_constituency_hist.png", dpi=150, bbox_inches="tight")
plt.show()

# reliability now defined on what actually drives the rate's CI width, not on pools/total text reviews.
# MIN_SOCIAL_REVIEWS: a floor on the denominator. MIN_RACIALISED_REVIEWS: revised upward from the
# standard rare-event rule-of-thumb minimum of >=5 "successes" to >=10, for a meaningfully tighter
# Wilson CI on the rate estimate (see the '1.2 addendum' markdown cell below for the exclusion count).
MIN_SOCIAL_REVIEWS = 100
MIN_RACIALISED_REVIEWS = 10

const_agg["reliable_n"] = (
    (const_agg["social_reviews"] >= MIN_SOCIAL_REVIEWS)
    & (const_agg["racialised_reviews"] >= MIN_RACIALISED_REVIEWS)
)
n_reliable = const_agg["reliable_n"].sum()
print(f"\nconstituencies meeting the revised reliability bar (>={MIN_SOCIAL_REVIEWS} social reviews AND "
      f">={MIN_RACIALISED_REVIEWS} racialised reviews): {n_reliable} / 299 ({n_reliable / 299:.1%})")

# how different is this from the old, pool/text-review-based bar?
old_reliable = (const_agg["pool_count"] >= 5) & (const_agg["total_text_reviews"] >= 100)
print(f"old (pool/text-review-based) bar would have flagged: {old_reliable.sum()} / 299 as reliable")
print(f"constituencies where the two bars disagree: {(old_reliable != const_agg['reliable_n']).sum()}")

n_below_bar = (~const_agg["reliable_n"]).sum()
print(f"\nMIN_SOCIAL_REVIEWS never actually binds here: minimum observed social_reviews is "
      f"{const_agg['social_reviews'].min()}, already well above {MIN_SOCIAL_REVIEWS}. The revised bar's "
      f"{n_below_bar} exclusions are driven entirely by too few racialised_reviews, not by denominator size.")

print(f"\nThe {n_below_bar} constituencies below the revised bar (<{MIN_RACIALISED_REVIEWS} racialised reviews):")
print(const_agg.loc[~const_agg["reliable_n"], ["constituency", "state", "social_reviews", "racialised_reviews"]]
      .sort_values("racialised_reviews").to_string(index=False))


**Takeaway.** This confirms the concern directly: `social_reviews` was never actually the binding constraint (minimum is 307, far above any reasonable denominator floor), but `racialised_reviews` is a real bottleneck for a genuinely different set of constituencies. At the original `>=5` racialised-reviews bar, 11/299 (3.7%) constituencies fell short, led by `Berlin-Pankow` with just **1** racialised review out of 382 social reviews. We revise this bar upward to `>=10` (code above) for a meaningfully tighter Wilson CI on the rate estimate — see the reliability-bar revision directly below for the exclusion count and named examples at the revised bar. The old pool/text-review-based bar and the racialised/social-review-based bar (at the revised `>=10` cut) **disagree on 48 constituencies** — it was neither a subset nor a superset of the right answer, it was measuring a different thing that happened to overlap only partially. From here on, `reliable_n` in this notebook reflects this racialised/social-review-based definition at `>=10`, not the pool-count one — anywhere a choropleth or robustness check flags "unreliable," it now means "too few racialised reviews to trust this rate," which is the caveat that actually matters for everything built on top of `racialised_rate`.

## Revising the reliability bar to `>=10` racialised reviews

Raising `MIN_RACIALISED_REVIEWS` from 5 to 10 moves the exclusion count from **11/299 (3.7%)** to **52/299 (17.4%)** — the sensitivity sweep two cells above already previewed this (the `>=10` row). The 41 newly-excluded constituencies (5-9 racialised reviews, previously counted as reliable) include some places that are otherwise well-covered by reviews overall, e.g. `Münster` (5 racialised / 634 social), `Ulm` (7 / 984), `Stuttgart I` (9 / 785), `Würzburg` (9 / 1,102), and `Rosenheim` (9 / 1,595) — a reminder that `social_reviews` volume and `racialised_reviews` volume are genuinely different constraints (as the cell above already notes; it just bites harder at the tighter bar). Everything downstream (choropleth hatching in the Choropleths subsection, the `is_land_border_constituency` flag in the
border-constituency subsection, and the final saved `const_agg.parquet`) now uses this `>=10` definition.


## Prevalence, with confidence intervals

A bare proportion (racialised / social reviews) is misleading on its own for a rare event measured on constituencies with anywhere from ~700 to ~21,000 reviews — the same point estimate means something very different backed by 700 reviews vs. 21,000. We use the **Wilson score interval** rather than the normal approximation: it stays inside [0, 1] and is far better behaved for small samples and proportions near zero, both of which apply here (racialised rate is ~0.5-2%, and several constituencies have modest review counts). We define the rate as **racialised / social**, i.e. among reviews already judged to be about the social experience of the venue (rather than / total reviews, which would dilute the rate with reviews about e.g. opening hours or facilities that were never in scope for racialised-discourse classification).

In [ ]:
# wilson() and its proportion_confint import are hoisted into the shared Part 1 setup
# above (WHEN's temporal cells need it too, and now run before this section).
overall_rate, overall_lo, overall_hi = wilson(
    (corpus["racialised"] == "yes").sum(), (corpus["social"] == "yes").sum()
)
print(f"overall racialised rate (of social=yes reviews): {overall_rate:.3%}  "
      f"[95% CI {overall_lo:.3%}, {overall_hi:.3%}]")

# per-state rates with CI
state_rate = state_agg.apply(
    lambda r: pd.Series(wilson(r["racialised_reviews"], r["social_reviews"]), index=["rate", "ci_lo", "ci_hi"]),
    axis=1,
)
state_rate = state_agg.join(state_rate).sort_values("rate", ascending=False)
state_rate[["n_constituencies", "social_reviews", "racialised_reviews", "rate", "ci_lo", "ci_hi"]]


In [ ]:
# constituency-level rate + CI, added onto the canonical const_agg table
rates = const_agg.apply(
    lambda r: pd.Series(wilson(r["racialised_reviews"], r["social_reviews"]), index=["racialised_rate", "ci_lo", "ci_hi"]),
    axis=1,
)
const_agg = pd.concat([const_agg, rates], axis=1)
const_agg["ci_width"] = const_agg["ci_hi"] - const_agg["ci_lo"]

print("Widest 5 CIs (least reliable rate estimates):")
print(const_agg.sort_values("ci_width", ascending=False)
      [["constituency", "state", "social_reviews", "racialised_reviews", "racialised_rate", "ci_width"]]
      .head(5).to_string(index=False))

print("\nNarrowest 5 CIs (most reliable rate estimates):")
print(const_agg.sort_values("ci_width")
      [["constituency", "state", "social_reviews", "racialised_reviews", "racialised_rate", "ci_width"]]
      .head(5).to_string(index=False))

      
FULL_W = 12.0    # match \textwidth (confirm your class)
FONT = 10

plot_df = const_agg.sort_values("racialised_rate").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(FULL_W, FULL_W * 5 / 14))   # keeps your original 14:5 aspect
colors = np.where(plot_df["reliable_n"], "#4C72B0", "#C44E52")
ax.errorbar(
    range(len(plot_df)), plot_df["racialised_rate"],
    yerr=[plot_df["racialised_rate"] - plot_df["ci_lo"], plot_df["ci_hi"] - plot_df["racialised_rate"]],
    fmt="none", ecolor=colors, alpha=0.5, linewidth=1,
)
ax.scatter(range(len(plot_df)), plot_df["racialised_rate"], c=colors, s=10, zorder=3)
ax.axhline(overall_rate, color="black", linestyle="--", linewidth=1, label=f"overall rate ({overall_rate:.2%})")
ax.set_xlabel("constituencies, sorted by racialised rate", fontsize=FONT, fontweight="bold")
ax.set_ylabel("racialised rate (of social reviews)", fontsize=FONT, fontweight="bold")
ax.tick_params(axis="both", labelsize=FONT-1, width=0.8, length=3)
ax.set_title(f"Racialised rate per constituency with 95% Wilson CI\n(red = below reliability bar: <{MIN_RACIALISED_REVIEWS} racialised reviews)", fontsize=FONT)
leg = ax.legend(fontsize=FONT-1, edgecolor="black", facecolor="white", frameon=True)
leg.get_frame().set_linewidth(0.8)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "racialised_rate_per_constituency_wilson_ci.png", dpi=300, bbox_inches="tight")
plt.show()


**Takeaway.** The overall racialised rate is **1.54% of social reviews [95% CI 1.51%, 1.58%]** — a tight interval at this scale, so the headline prevalence number is solid. State-level rates range roughly 4x, from Mecklenburg-Vorpommern (0.81%) up to Saarland/Berlin/Bremen (~3.0-3.1%), and these top three are small states/city-states, so their CIs are wider — worth naming as "elevated, but on a smaller base" rather than a flat ranking. At constituency level the widest CIs cluster in Ruhr-area NRW constituencies (Essen II, Bochum I, Recklinghausen I) and Berlin-Neukölln — but these aren't just noisy from small N; several (e.g. Bochum I at 526 reviews, 5.7% rate) have *both* reasonable sample size and a genuinely high rate, which is a more interesting signal than sampling noise and worth flagging as a candidate area for closer qualitative follow-up. The narrowest, most reliable estimates sit in the 0.25-0.7% range in mid-size rural/semi-rural constituencies.

## Choropleths

Three separate maps onto the constituency polygons from Section 1: pool coverage
(`pools_with_social`), the social-review rate (`social_reviews / total_text_reviews`), and the
racialised rate itself. For the rate map specifically, constituencies below our `reliable_n` bar
(<10 racialised reviews, revised upward from an earlier <5 cut — see the reliability-bar
subsection above) are hatched rather than removed — removing them would imply "no data" when the
honest message is "data exists but the estimate is noisy."

In [ ]:
# national pool coverage: how many of the 5,528 pools ever get a social/racialised review at all?
n_pools_total = corpus["unique_identifier"].nunique()
n_pools_with_social = corpus.loc[social_mask, "unique_identifier"].nunique()
n_pools_with_racialised = racialised_df["unique_identifier"].nunique()
print(f"pools nationally: {n_pools_total:,}")
print(f"pools with >=1 social review: {n_pools_with_social:,} ({n_pools_with_social / n_pools_total:.1%})")
print(f"pools with >=1 racialised review: {n_pools_with_racialised:,} ({n_pools_with_racialised / n_pools_total:.1%})")

map_df = constituencies[["WKR_NR", "geometry"]].rename(columns={"WKR_NR": "constituency_code"}).merge(
    const_agg, on="constituency_code", how="left"
)


In [ ]:
import textwrap

def format_legend(numbered_names, width=20):
    lines = []
    for num, name in numbered_names:
        prefix = f"{num}: "
        wrapped = textwrap.wrap(name, width=width)
        lines.append(prefix + wrapped[0])
        for cont in wrapped[1:]:
            lines.append(" " * len(prefix) + cont)
    return "\n".join(lines)

fig, ax = plt.subplots(figsize=(7, 8))
map_df.plot(column="pools_with_social", cmap="Blues", legend=True, ax=ax, edgecolor="white", linewidth=0.1)
state_boundaries = map_df.dissolve(by="state")
state_boundaries.boundary.plot(ax=ax, color="black", linewidth=0.9)

top8_pools = map_df.sort_values("pools_with_social", ascending=False).head(8).reset_index(drop=True)
for num, row in top8_pools.iterrows():
    pt = row.geometry.representative_point()
    ax.annotate(str(num + 1), xy=(pt.x, pt.y), fontsize=8, fontweight="bold", color="black",
                ha="center", va="center",
                bbox=dict(boxstyle="circle,pad=0.25", fc="white", ec="black", linewidth=0.6))

legend_text = format_legend([(i + 1, row["constituency"]) for i, row in top8_pools.iterrows()])
ax.text(0.85, 0.32, legend_text, transform=ax.transAxes, fontsize=6.5, ha="left", va="bottom",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black", linewidth=0.6))

ax.axis("off")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "pools_with_social_choropleth.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
map_df["social_rate"] = map_df["social_reviews"] / map_df["total_text_reviews"]
import textwrap

def format_legend(numbered_names, width=20):
    # right-aligned box (ha="right"): matplotlib right-aligns each line independently,
    # so continuation lines need no manual indent -- ha handles the alignment.
    lines = []
    for num, name in numbered_names:
        wrapped = textwrap.wrap(name, width=width)
        lines.append(f"{num}: {wrapped[0]}")
        lines.extend(wrapped[1:])
    return "\n".join(lines)

fig, ax = plt.subplots(figsize=(7, 8))
map_df.plot(column="social_rate", cmap="Greens", legend=True, ax=ax, edgecolor="white", linewidth=0.1)
state_boundaries = map_df.dissolve(by="state")
state_boundaries.boundary.plot(ax=ax, color="black", linewidth=0.9)

top8_social = map_df.sort_values("social_rate", ascending=False).head(8).reset_index(drop=True)
for num, row in top8_social.iterrows():
    pt = row.geometry.representative_point()
    ax.annotate(str(num + 1), xy=(pt.x, pt.y), fontsize=8, fontweight="bold", color="black",
                ha="center", va="center",
                bbox=dict(boxstyle="circle,pad=0.25", fc="white", ec="black", linewidth=0.6))

legend_text = format_legend([(i + 1, row["constituency"]) for i, row in top8_social.iterrows()])
ax.text(0.98, 0.28, legend_text, transform=ax.transAxes, fontsize=6.5, ha="right", va="bottom",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black", linewidth=0.6))

ax.axis("off")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "social_rate_choropleth.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Every constituency actually named in the WHERE-section prose (not just top-N by rate, which
# would miss cases like Berlin-Pankow, Trier, Mansfeld that are discussed for other reasons --
# coverage edge case, post-2021-only, dominant-pool outlier -- so a reader can locate each one
# spatially. Numbered markers + an on-map legend box, bottom corner, long names wrapped.
named_constituencies = [
    "Offenburg", "Duisburg II", "Gelsenkirchen", "Lörrach – Müllheim", "Essen II",
    "Berlin-Pankow", "Lübeck", "Trier", "Mansfeld",
]

missing = set(named_constituencies) - set(map_df["constituency"])
if missing:
    print(f"WARNING: not found in map_df, check spelling: {missing}")

import textwrap

def format_legend(numbered_names, width=20):
    # right-aligned box (ha="right"): matplotlib right-aligns each line independently,
    # so continuation lines need no manual indent -- ha handles the alignment.
    lines = []
    for num, name in numbered_names:
        wrapped = textwrap.wrap(name, width=width)
        lines.append(f"{num}: {wrapped[0]}")
        lines.extend(wrapped[1:])
    return "\n".join(lines)

fig, ax = plt.subplots(figsize=(7, 8))
map_df.plot(column="racialised_rate", cmap="Reds", legend=True, ax=ax, edgecolor="white", linewidth=0.1)
unreliable = map_df.loc[~map_df["reliable_n"]]
unreliable.plot(ax=ax, facecolor="none", edgecolor="black", hatch="///", linewidth=0.6)
state_boundaries = map_df.dissolve(by="state")
state_boundaries.boundary.plot(ax=ax, color="black", linewidth=0.9)

for num, name in enumerate(named_constituencies, start=1):
    row = map_df.loc[map_df["constituency"] == name].iloc[0]
    pt = row.geometry.representative_point()
    ax.annotate(str(num), xy=(pt.x, pt.y), fontsize=8, fontweight="bold", color="black",
                ha="center", va="center",
                bbox=dict(boxstyle="circle,pad=0.25", fc="white", ec="black", linewidth=0.6))

legend_text = format_legend([(i, name) for i, name in enumerate(named_constituencies, start=1)])
ax.text(0.98, 0.28, legend_text, transform=ax.transAxes, fontsize=6.5, ha="right", va="bottom",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="black", linewidth=0.6))

ax.axis("off")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "rr_choropleth_labeled.png", dpi=300, bbox_inches="tight")
plt.show()

**Takeaway.** Three maps, three layers of the same underlying data. Pool coverage
(`pools_with_social`) is patchy but not sparse anywhere critical — the most pool-dense
constituencies (`Nordfriesland – Dithmarschen Nord`, 53; `Ansbach`, 48; `Erzgebirgskreis I`, 48)
are large rural areas accumulating many small venues, not data errors. The social-review rate
map shows which constituencies' reviews skew toward the social-experience content this whole
analysis depends on — highest in `Bottrop – Recklinghausen III` (57.0% of 2,785 reviews),
`Berlin-Marzahn-Hellersdorf` (53.1%), and `Dahme-Spreewald – Teltow-Fläming III` (51.6% of
21,490) — with `Duisburg II` also appearing here (49.1%), an early hint that its high racialised
rate isn't a small-sample artifact. The racialised-rate map is the substantive one: it is **not**
a diffuse, noisy-looking pattern — it shows a concentrated cluster of high, well-supported rates
in western Germany, led by `Offenburg` (8.6% of 1,724 social reviews), `Duisburg II` (7.6% of
2,316), `Gelsenkirchen` (6.0%), and `Lörrach – Müllheim` (5.7% of 3,975 reviews) — all several
times the 1.54% national rate, and all with sample sizes large enough that this isn't sampling
noise. Combined with the Ruhr-area constituencies flagged in the prevalence-with-CIs subsection
above, there's a recognisable geography here (Ruhr valley NRW + southwestern Baden-Württemberg
border constituencies) that is a genuine spatial pattern, not scattered outliers.

## Spatial autocorrelation: is the clustering in the map above real?

Moved here from what was originally drafted as a Part 2 section: testing this needs only `racialised_rate` and constituency geometry, both already available at this point, not the socioeconomic/political data Part 2 introduces -- no reason to make a reader wait for Part 2 to find out whether the racialised-rate choropleth above is showing a real pattern or just a suggestive color scale. The earlier ICWSM prototype hand-rolled Moran's I with an ad hoc distance-weight threshold, despite already importing `pysal`/`esda` for exactly this purpose. We use the actual library this time: **Queen contiguity weights** (constituencies that share a border or a single point are neighbours) built directly from the Wahlkreis polygons, row-standardized, with **Global Moran's I** tested via permutation (999 random relabellings) rather than a normal-theory p-value, since we can't be sure the normality assumption holds for a rate this skewed.

In [ ]:
from libpysal.weights import Queen
from esda.moran import Moran, Moran_Local

moran_gdf = map_df.dropna(subset=["racialised_rate"]).reset_index(drop=True)

w = Queen.from_dataframe(moran_gdf, use_index=False)
n_islands = len(w.islands)
print(f"constituencies with zero contiguity neighbours (islands): {n_islands}")
if n_islands:
    print(moran_gdf.loc[w.islands, "constituency"].to_list())
w.transform = "r"

moran = Moran(moran_gdf["racialised_rate"].values, w, permutations=999)
print(f"\nGlobal Moran's I = {moran.I:.4f}")
print(f"expected I under spatial randomness = {moran.EI:.4f}")
print(f"p-value (999 permutations) = {moran.p_sim:.4f}")
print(f"z-score = {moran.z_sim:.2f}")


In [ ]:
# Local Moran's I (LISA) to identify *which* constituencies drive the global clustering
from matplotlib.patches import Patch

lisa = Moran_Local(moran_gdf["racialised_rate"].values, w, permutations=999, seed=42)
sig = lisa.p_sim < 0.05

cluster_labels = np.array(["Not significant"] * len(moran_gdf), dtype=object)
quadrant_names = {1: "High-High (hotspot)", 2: "Low-High", 3: "Low-Low (coldspot)", 4: "High-Low"}
for q, name in quadrant_names.items():
    cluster_labels[(lisa.q == q) & sig] = name
moran_gdf["lisa_cluster"] = cluster_labels

print(moran_gdf["lisa_cluster"].value_counts())
print("\nHigh-High (hotspot) constituencies:")
print(moran_gdf.loc[moran_gdf["lisa_cluster"] == "High-High (hotspot)", ["constituency", "state", "racialised_rate"]]
      .sort_values("racialised_rate", ascending=False).to_string(index=False))

fig, ax = plt.subplots(figsize=(7, 8))
palette = {
    "High-High (hotspot)": "#C44E52", "Low-Low (coldspot)": "#4C72B0",
    "Low-High": "#8FBADE", "High-Low": "#E8A9AC", "Not significant": "#E8E8E8",
}
for label, color in palette.items():
    subset = moran_gdf[moran_gdf["lisa_cluster"] == label]
    if len(subset):
        subset.plot(ax=ax, color=color, edgecolor="white", linewidth=0.2)
ax.legend(handles=[Patch(facecolor=c, label=l) for l, c in palette.items()], loc="lower left", fontsize=8)
ax.set_title("LISA cluster map: racialised rate\n(Queen contiguity, 999 permutations, p<0.05)")
ax.axis("off")
plt.tight_layout()
fig.savefig(FIGURES_DIR / "lisa_clusters.png", dpi=130)
plt.show()


**Takeaway.** The clustering is real, not an artifact of the choropleth's color scale: **Global Moran's I = 0.28** (expected value under spatial randomness ≈ 0, z = 7.91, p = 0.001 over 999 permutations) — a moderate, highly significant positive spatial autocorrelation. This confirms what the map above suggested visually and replaces the earlier prototype's ad hoc, hand-rolled Moran's I with the proper library implementation.

The LISA breakdown identifies exactly two coherent **High-High hotspot clusters**, both matching the constituencies already flagged above: a **Ruhr-area NRW cluster** (`Gelsenkirchen`, `Bochum I`, `Essen II`/`III`, `Duisburg I`, `Mülheim – Essen I`, `Herne – Bochum II`, `Oberhausen – Wesel III`, `Mettmann II`, and neighbours) and a **Berlin cluster** (`Berlin-Neukölln`, `Berlin-Mitte`, `Berlin-Tempelhof-Schöneberg`, `Berlin-Friedrichshain-Kreuzberg`, `Berlin-Treptow-Köpenick`, `Berlin-Charlottenburg-Wilmersdorf`, plus the adjoining Brandenburg constituency `Dahme-Spreewald – Teltow-Fläming III`), with a smaller pair (`Rastatt`, `Emmendingen – Lahr`) extending the earlier Offenburg/Lörrach signal in southwestern Baden-Württemberg. **Low-Low coldspots** are broad and diffuse — most of Schleswig-Holstein/Mecklenburg-Vorpommern in the north, a Saxony-Anhalt/Thuringia block in the center-east, and most of Bavaria south of Nuremberg — i.e. the "low" side of the pattern is a general background rate across large, mostly rural regions, while the "high" side is concentrated in specific dense, post-industrial or major-metropolitan constituencies. This is a genuinely paper-worthy geographic finding on its own. **Part 2** (`02_socioeconomic_political_relations.ipynb`) later connects these same hotspots to the deprivation/political predictors it introduces — the hotspots sit in exactly the kind of higher-deprivation, higher-foreign-population-share urban constituencies Part 2 finds correlated with the rate.

## Concentration: how many *pools*, not just reviews, drive a constituency's racialised rate?

The reliability-bar subsection above checked whether a constituency has enough racialised reviews to trust its rate — but not *where within the constituency* those reviews sit. Two constituencies can have identical `racialised_reviews` counts and tell completely different stories: one where the reviews are spread across most of the constituency's pools (a pattern that plausibly reflects something about the *area*), and one where they all sit at one or two specific venues (a pattern that may really be about those *venues*, not the constituency as a whole). This is a sharper, venue-level version of the ecological-fallacy problem: if a "high-rate constituency" is really "one pool with a bad pattern," reading it as evidence about the surrounding area is even shakier than the aggregate framing already implies.

For each constituency we compute: how many of its pools have *at least one* racialised review (`pools_with_racialised`, and as a share of `pools_with_social` — pools that could plausibly have a racialised review at all, since a racialised review is by definition also a social one), and how concentrated the racialised reviews are within it (`top_pool_share_of_racialised` — the share of a constituency's racialised reviews contributed by its single most-implicated pool). A low share of affected pools + a high top-pool share together would mean "one venue is doing most of the work"; a high share of affected pools + a low top-pool share would mean "this is genuinely spread across the area."

In [ ]:
# racialised_df already built earlier (shared Part 1 setup); reused here for pool-level counts
# pool-level racialised counts, then aggregated up to constituency level
pool_racialised = (
    racialised_df.groupby(["constituency_code", "unique_identifier"]).size().reset_index(name="pool_racialised_count")
)
pool_agg = pool_racialised.groupby("constituency_code").agg(
    pools_with_racialised=("unique_identifier", "nunique"),
    top_pool_racialised_count=("pool_racialised_count", "max"),
)

const_agg = const_agg.merge(pool_agg, on="constituency_code", how="left")
const_agg["pools_with_racialised"] = const_agg["pools_with_racialised"].fillna(0).astype(int)
# denominator is pools_with_social, not pool_count: a pool with zero social reviews can never
# have a racialised review, so it shouldn't count against a constituency's "share affected".
const_agg["pct_pools_with_racialised"] = (
    const_agg["pools_with_racialised"] / const_agg["pools_with_social"].replace(0, np.nan)
)
const_agg["top_pool_share_of_racialised"] = (
    const_agg["top_pool_racialised_count"] / const_agg["racialised_reviews"]
)

print("\nShare of a constituency's social-active pools that have >=1 racialised review:")
print(const_agg["pct_pools_with_racialised"].describe())
print()
print("Share of a constituency's racialised reviews contributed by its single most-implicated pool:")
print(const_agg["top_pool_share_of_racialised"].describe())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(const_agg["pct_pools_with_racialised"], bins=30, color="#4C72B0")
axes[0].set_title("Share of social-active pools with >=1 racialised review")
axes[0].set_xlabel("pct_pools_with_racialised")
axes[0].set_ylabel("number of constituencies")
axes[1].hist(const_agg["top_pool_share_of_racialised"], bins=30, color="#C44E52")
axes[1].set_title("Top-pool share of a constituency's racialised reviews")

plt.tight_layout()
plt.savefig(FIGURES_DIR / "pool_share_and_top_pool_share_racialised_hist.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# racialised_rate (Wilson point estimate) now already exists on const_agg -- computed in the
# Prevalence-with-CIs subsection above, which now runs before this one.
rho_pct, p_pct = stats.spearmanr(const_agg["racialised_rate"], const_agg["pct_pools_with_racialised"])
rho_top, p_top = stats.spearmanr(const_agg["racialised_rate"], const_agg["top_pool_share_of_racialised"])
print(f"racialised rate vs. pct_pools_with_racialised: rho={rho_pct:.3f}, p={p_pct:.2e}")
print(f"racialised rate vs. top_pool_share_of_racialised: rho={rho_top:.3f}, p={p_top:.2e}")

# the specific high-rate constituencies flagged in the prevalence/choropleth subsections above
# (checked here by name)
hotspot_names = [
    "Offenburg", "Duisburg II", "Gelsenkirchen", "Lörrach – Müllheim", "Bochum I",
    "Berlin-Neukölln", "Essen II", "Recklinghausen I",
]
cols = ["constituency", "state", "pool_count", "racialised_reviews",
        "pools_with_racialised", "pct_pools_with_racialised", "top_pool_share_of_racialised", "racialised_rate"]
print("\nConcentration profile of the flagged high-rate constituencies:")
print(const_agg[const_agg["constituency"].isin(hotspot_names)][cols]
      .sort_values("racialised_rate", ascending=False).to_string(index=False))

**Takeaway.** Concentration is real but the pattern is reassuring, not worrying. On average, only **40% of a constituency's social-active pools have any racialised review at all** (median 37.5%), and its single most-implicated pool typically accounts for **~48% of its racialised reviews** (median 47%, ranging all the way to 100% for a few constituencies with very few affected pools) — so no, this was not a negligible detail: a meaningful share of a constituency's racialised-review count often does rest on one or two specific venues, exactly the scenario in your hypothetical.

But the key relationship goes the *reassuring* way: **racialised rate correlates strongly with how many pools are affected** (ρ=0.65, p≪0.001), not with how concentrated the top pool is (ρ=0.09, not significant). In other words, **high-rate constituencies tend to have racialised discourse spread across many of their pools, not concentrated in one "bad venue"** — checking the specific hotspots named earlier confirms this directly: `Gelsenkirchen` (90% of its pools affected) and `Essen II` (83%) are genuinely broad-based patterns, and even the more concentrated-looking cases (`Duisburg II`, `Essen II` at 65-71% top-pool share) have very few pools in total (6), so one venue naturally carries a large share simply because there isn't much to spread across — that's a small-denominator effect, not evidence of a single outlier driving an otherwise-quiet constituency.

## Is the dominant pool unusual in *rate*, or just in *volume*?

The concentration subsection above showed concentration exists but isn't what drives high constituency rates. One more check in the same spirit: when one pool contributes the largest share of a constituency's racialised reviews, is that because the pool has an unusually **elevated racialised rate** (something specific to that venue), or simply because it's the **biggest, most-reviewed pool** in the constituency (so it naturally contributes more of everything, racialised included, without being disproportionately "worse")? These have very different implications — the first is a venue-specific finding, the second is just an artifact of review volume.

For each constituency's single most-implicated ("dominant") pool (the same one identified in the concentration check above, which on average contributes ~48% of a constituency's racialised reviews), we compare it directly against **the rest of its own constituency**, not against pools nationally: its own rate (racialised/social) vs. the combined rate of every other pool in the same constituency, tested with a two-proportion test, and its share of the constituency's total social-review volume. This answers the actual question at the right grain — is *this* pool unusual for *its own area* — without pulling in unrelated pools from other constituencies as a reference population, and without inventing a second reliability threshold: we reuse the existing `reliable_n` bar (247/299 constituencies) already established above.

In [ ]:
# Comparing the dominant pool against pools nationally answered a different question -- "is this
# pool unusual among all German pools" -- and needed an arbitrary reliability floor to keep that
# reference population meaningful, which excluded 62% of constituencies. The real question is
# whether the dominant pool is unusual *for its own constituency*: compare it directly against the
# rest of that constituency's pools combined, reusing the existing reliable_n bar (247/299) instead
# of inventing a pool-level one.
from statsmodels.stats.proportion import proportions_ztest

# the actual dominant pool (not just its count) for each constituency
dominant_idx = pool_racialised.groupby("constituency_code")["pool_racialised_count"].idxmax()

pool_social = corpus.loc[corpus["social"] == "yes"].groupby("unique_identifier").size().rename("dominant_social_reviews")

dominant_pools = pool_racialised.loc[dominant_idx].set_index("constituency_code")
dominant_pools = dominant_pools.join(pool_social, on="unique_identifier")

reliable_const = const_agg.loc[
    const_agg["reliable_n"], ["constituency_code", "constituency", "state", "social_reviews", "racialised_reviews"]
].set_index("constituency_code")
dp = dominant_pools.join(reliable_const, how="inner")

dp["dominant_rate"] = dp["pool_racialised_count"] / dp["dominant_social_reviews"]
dp["rest_racialised"] = dp["racialised_reviews"] - dp["pool_racialised_count"]
dp["rest_social"] = dp["social_reviews"] - dp["dominant_social_reviews"]

n_single_pool = (dp["rest_social"] <= 0).sum()
print(f"reliable constituencies analysed: {len(dp)}")
print(f"single-pool constituencies (no 'rest of constituency' to compare against, excluded): {n_single_pool}")
dp = dp.loc[dp["rest_social"] > 0].copy()

dp["rest_rate"] = dp["rest_racialised"] / dp["rest_social"]
dp["volume_share"] = dp["dominant_social_reviews"] / dp["social_reviews"]

def two_prop_p(row):
    return proportions_ztest(
        [row["pool_racialised_count"], row["rest_racialised"]],
        [row["dominant_social_reviews"], row["rest_social"]],
    )[1]

dp["p_value"] = dp.apply(two_prop_p, axis=1)
dp["significantly_elevated"] = (dp["p_value"] < 0.05) & (dp["dominant_rate"] > dp["rest_rate"])
dp["big"] = dp["volume_share"] > 0.5

# Wilson CIs on each side (the `wilson()` helper defined earlier in Part 1), reported alongside
# the z-test for interpretability -- note these are two different checks: non-overlapping CIs
# implies significance, but overlapping CIs does NOT imply non-significance (CI-overlap is the
# more conservative, weaker test). The z-test above remains the actual significance criterion;
# this just shows how much stricter a bare CI-overlap check would be.
dp["dominant_ci_lo"], dp["dominant_ci_hi"] = zip(*dp.apply(
    lambda r: wilson(r["pool_racialised_count"], r["dominant_social_reviews"])[1:], axis=1))
dp["rest_ci_lo"], dp["rest_ci_hi"] = zip(*dp.apply(
    lambda r: wilson(r["rest_racialised"], r["rest_social"])[1:], axis=1))
dp["ci_non_overlapping"] = dp["dominant_ci_lo"] > dp["rest_ci_hi"]

print(f"\n  of the {dp['significantly_elevated'].sum()} z-test-significant dominant pools, "
      f"{(dp['significantly_elevated'] & dp['ci_non_overlapping']).sum()} "
      f"also have non-overlapping Wilson CIs (the stricter check)")

print(f"\nof {len(dp)} constituencies with a comparable 'rest of constituency':")
print(f"  dominant pool's rate significantly higher than the rest of its own constituency: "
      f"{dp['significantly_elevated'].sum()} ({dp['significantly_elevated'].mean():.1%})")
print(f"  dominant pool contributes >50% of the constituency's social-review volume: "
      f"{dp['big'].sum()} ({dp['big'].mean():.1%})")

quadrants = pd.Series(
    np.select(
        [dp["big"] & dp["significantly_elevated"], dp["big"] & ~dp["significantly_elevated"],
         ~dp["big"] & dp["significantly_elevated"], ~dp["big"] & ~dp["significantly_elevated"]],
        ["big pool, significantly elevated rate", "big pool, not significantly elevated",
         "small pool, significantly elevated rate", "small pool, not significantly elevated"],
        default="unknown",
    )
).value_counts()
print("\nWhich kind of pool is 'dominant' in its constituency?")
print(quadrants)

small_elevated = dp.loc[~dp["big"] & dp["significantly_elevated"]].sort_values("dominant_rate", ascending=False)
print(f"\nThe {len(small_elevated)} 'small pool, significantly elevated rate' constituencies:")
print(small_elevated[["constituency", "state", "pool_racialised_count", "dominant_social_reviews",
                       "dominant_rate", "dominant_ci_lo", "dominant_ci_hi", "rest_rate", "volume_share",
                       "p_value", "ci_non_overlapping"]].to_string(index=False))

fig, ax = plt.subplots(figsize=(6, 5))
colors = np.where(dp["significantly_elevated"], "#C44E52", "#4C72B0")
ax.errorbar(
    dp["rest_rate"] * 100, dp["dominant_rate"] * 100,
    yerr=[(dp["dominant_rate"] - dp["dominant_ci_lo"]) * 100, (dp["dominant_ci_hi"] - dp["dominant_rate"]) * 100],
    xerr=[(dp["rest_rate"] - dp["rest_ci_lo"]) * 100, (dp["rest_ci_hi"] - dp["rest_rate"]) * 100],
    fmt="none", ecolor=colors, alpha=0.3, linewidth=0.8,
)
ax.scatter(dp["rest_rate"] * 100, dp["dominant_rate"] * 100, c=colors, alpha=0.6, s=20, zorder=3)
lims = [0, max(dp["dominant_ci_hi"].max(), dp["rest_ci_hi"].max()) * 100 * 1.05]
ax.plot(lims, lims, color="grey", linestyle="--", linewidth=0.8)
ax.set_xlabel("rate of the rest of the constituency's pools (%)")
ax.set_ylabel("dominant pool's own rate (%)")
ax.set_title("Dominant pool's rate vs. the rest of its own constituency\n(red = significantly elevated, p<0.05)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dominant_pool_vs_rest_of_constituency_rate.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway.** Comparing each constituency's dominant pool directly against the rest of its own constituency (a two-proportion test, `reliable_n` constituencies only, n=247 -- no single-pool constituencies to exclude) gives a cleaner and more concerning picture than either earlier version.

**40% of dominant pools (99/247) have a rate that is statistically significantly higher than the rest of their own constituency's pools combined.** Splitting by volume share: **54/247 (22%) of dominant pools carry more than half their constituency's social-review volume** ("big"); of those, only 9/247 (3.6%) are *also* significantly elevated -- for most big dominant pools, their size alone explains their contribution, consistent with the earlier finding. But **90/247 (36%) are dominant pools that carry a minority of the volume and are still significantly elevated** -- a real venue-specific effect, not a volume artifact, and a much larger share than either earlier pass suggested. `Lübeck` and `Mansfeld` (5/21 = 23.8% each) are legitimately on this list: a proper significance test says a 23.8% rate against a ~0.7% constituency baseline is extreme regardless of the small raw count (p≈2e-18) -- the earlier percentile-based versions either flagged this kind of case on a shaky basis (no minimum-count floor) or excluded it altogether (an arbitrary national floor); testing directly against the constituency's own baseline handles both problems at once, with no extra threshold needed. Cross-checked against Wilson CIs (the same interval used throughout Part 1): 60 of the 99 z-test-significant dominant pools also have non-overlapping CIs against the rest of their constituency -- the z-test is the correct primary criterion here (non-overlapping CIs implies significance, but overlapping CIs doesn't imply the reverse), so the other 39 are not spurious, just below the more conservative CI-overlap bar. `Lübeck`/`Mansfeld` clear both: their dominant-pool CI (10.6-45.1%) sits entirely above the rest-of-constituency rate.

In [ ]:
# flag these constituencies on const_agg for downstream robustness checks, without recomputing
# pool-level stats. Definition updated to match the corrected
# within-constituency test: dominant pool carries a minority of constituency volume AND is
# significantly elevated vs. the rest of the constituency (was: national percentile-based).
const_agg["dominant_pool_is_small_high_rate"] = const_agg["constituency_code"].isin(small_elevated.index)
print(f"flagged on const_agg: {const_agg['dominant_pool_is_small_high_rate'].sum()} constituencies")

### Draft: paper write-up table (WHERE section findings)

One table: constituency-level correlation/significance tests (pool-concentration, dominant-pool test, border distance). Trimmed to what the prose actually cites -- dropped the old prevalence/reliability summary table (redundant with Fig. 1 + the opening paragraph) and the uncorrected "national boundary incl. coastline" border-distance rows (not referenced in text; only the corrected land-neighbour measure is). Source file: `where_section_findings_tables.tex` (session scratchpad, not reachable from your side -- this cell is the copy to actually use). Move into the paper repo when ready.

```latex
% Requires: \usepackage{booktabs} \usepackage{array}
% All numbers pulled directly from descriptive_analysis.ipynb's WHERE section
% takeaway cells (Concentration / Is the dominant pool unusual / Geometric proxy subsections).
% Single-column table. Test column is a wrapping p{} column (not plain l) and labels are
% shortened -- the previous version's long Test strings (e.g. "Dominant pool sig. elevated
% vs. rest of constituency") forced the table past \linewidth in a two-column layout.

\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{3pt}
\caption{WHERE section: correlation and significance tests (constituency level; reliable\_n = $\geq$10 racialised reviews, 247/299).}
\label{tab:where-correlations}
\begin{tabular}{@{} p{3.0cm} r l r c @{}}
\toprule
Test & $\rho$ & $p$ & $n$ & Sig. \\
\midrule
Rate vs.\ \% pools w/ racialised review$^\dagger$ & 0.65 & $\ll 0.001$ & 299 & *** \\
Rate vs.\ top-pool share$^\P$ & 0.09 & n.s. & 299 & ns \\
Dominant pool sig.\ elevated$^\ddagger$ & -- & -- & 247 & 99/247 (40\%) \\
Border dist.$^\S$, all constituencies & $-0.055$ & 0.341 & 299 & ns \\
Border dist.$^\S$, reliable\_n only & $-0.000$ & 0.996 & 247 & ns \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
$^{***}p<0.001$, ns = not significant. $^\dagger$Share of a constituency's social-active pools (not all pools) with $\geq$1 racialised review. $^\P$Share of a constituency's racialised reviews contributed by its single most-implicated pool. $^\ddagger$Per-constituency two-proportion $z$-test vs.\ rest of constituency, not a single correlation; of these 99, 54/247 (22\%) overall also carry $>$50\% of the constituency's review volume (``big''), while 90/247 (36\%) are minority-volume pools that are still significantly elevated -- see text. $^\S$Distance to nearest land-neighbour country.
\end{table}
```

## Geometric proxy: how close is each constituency to a neighbouring country's border?

Using constituency polygons from Section 1 and actual country polygons (Natural Earth, bundled
with this environment's `pyogrio` install) for Germany's 9 land neighbours — France, Poland,
Austria, Switzerland, Luxembourg, Belgium, Netherlands, Denmark, Czechia — we compute
`distance_to_neighbor_country_km` (straight-line distance to the nearest neighbouring-country
polygon) and the binary `is_land_border_constituency` (within 150m of one). We deliberately
measure distance to a neighbouring country's polygon, not to Germany's own external boundary
line: the latter also includes the North Sea and Baltic coastlines, which would incorrectly put
a coastal-but-inland constituency like `Bremen II – Bremerhaven` at the same 0km as a genuine
cross-border constituency like `Offenburg`. Both columns are saved to `const_agg`. This is a bivariate, purely descriptive check — it does
not control for anything (state, foreign-population share, urban/rural, etc.).

In [ ]:
world_path = "/home/ethel/Desktop/swimming_pool_project/.venv/lib/python3.12/site-packages/pyogrio/tests/fixtures/naturalearth_lowres/naturalearth_lowres.shp"
world = gpd.read_file(world_path)
NEIGHBOURS = ["France", "Poland", "Austria", "Switzerland", "Luxembourg", "Belgium", "Netherlands", "Denmark", "Czechia"]
neighbour_gdf = world[world["name"].isin(NEIGHBOURS)].to_crs(constituencies.crs)
print(f"neighbouring countries matched: {sorted(neighbour_gdf['name'].tolist())} ({len(neighbour_gdf)}/9)")

neighbour_union = neighbour_gdf.union_all()
dist_by_wkr = constituencies.set_index("WKR_NR").geometry.distance(neighbour_union) / 1000
const_agg["distance_to_neighbor_country_km"] = const_agg["constituency_code"].map(dist_by_wkr)
const_agg["is_land_border_constituency"] = const_agg["distance_to_neighbor_country_km"] < 0.15

n_land_border = const_agg["is_land_border_constituency"].sum()
print(f"land-border constituencies: {n_land_border} / 299 ({n_land_border / 299:.1%})")

# sanity check: known hotspots on the France/Switzerland border should be True and near 0km; a
# genuinely coastal-but-inland seat and clearly interior constituencies should be False
check_names = ["Offenburg", "Lörrach – Müllheim", "Bremen II – Bremerhaven", "Frankfurt am Main I", "Kassel"]
print(const_agg.loc[const_agg["constituency"].isin(check_names),
                     ["constituency", "state", "is_land_border_constituency", "distance_to_neighbor_country_km"]]
      .to_string(index=False))

# Does racialised_rate actually vary smoothly with distance to a land border? Spearman rather
# than Pearson, for the same reason as elsewhere in this notebook: racialised_rate is a skewed
# proportion, not a normally-distributed quantity.
rho_all, p_all = stats.spearmanr(const_agg["distance_to_neighbor_country_km"], const_agg["racialised_rate"])
rho_rel, p_rel = stats.spearmanr(
    const_agg.loc[const_agg["reliable_n"], "distance_to_neighbor_country_km"],
    const_agg.loc[const_agg["reliable_n"], "racialised_rate"],
)
print(f"\ndistance_to_neighbor_country_km vs. racialised_rate: all 299 rho={rho_all:.3f} p={p_all:.3f} | "
      f"reliable_n only (n={const_agg['reliable_n'].sum()}) rho={rho_rel:.3f} p={p_rel:.3f}")

# coarse near/mid/far split on fixed, round-number breaks (a plain quantile split doesn't work
# cleanly here: the land-border constituencies all sit near 0km, which collapses qcut's first
# two bin edges together)
const_agg["dist_band"] = pd.cut(
    const_agg["distance_to_neighbor_country_km"], bins=[-0.1, 30, 70, 300],
    labels=["<=30km", "30-70km", ">70km"],
)
band_rates = const_agg.groupby("dist_band", observed=True).apply(
    lambda g: pd.Series(wilson(g["racialised_reviews"].sum(), g["social_reviews"].sum()),
                         index=["rate", "ci_lo", "ci_hi"])
)
band_rates["n_constituencies"] = const_agg.groupby("dist_band", observed=True).size()
print("\nracialised rate by distance-to-land-border band (pooled numerator/denominator, not averaged per-constituency):")
print(band_rates)

fig, ax = plt.subplots(figsize=(8, 5))
colors = np.where(const_agg["reliable_n"], "#4C72B0", "#C44E52")
ax.scatter(const_agg["distance_to_neighbor_country_km"], const_agg["racialised_rate"], c=colors, s=18, alpha=0.7)
for name in ["Offenburg", "Lörrach – Müllheim", "Duisburg II", "Gelsenkirchen"]:
    row = const_agg.loc[const_agg["constituency"] == name]
    if len(row):
        ax.annotate(name, xy=(row["distance_to_neighbor_country_km"].iloc[0], row["racialised_rate"].iloc[0]),
                    fontsize=7, xytext=(4, 4), textcoords="offset points")
ax.set_xlabel("distance to nearest land-neighbour country (km)")
ax.set_ylabel("racialised rate (of social reviews)")
ax.set_title(f"Racialised rate vs. distance to land border (rho={rho_all:.2f}, p={p_all:.2f}, n=299)\n"
             "blue = reliable_n, red = below reliability bar")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "racialised_rate_vs_distance_to_land_border_scatter.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway.** Racialised rate does not track distance to a land-neighbour border: the
correlation is weak and not significant (all 299 constituencies: rho=-0.055, p=0.341;
`reliable_n` only: rho=-0.000, p=0.996). The banded view shows why it isn't monotonic:
`<=30km` sits near the national rate, `30-70km` runs *above* it, `>70km` runs *below* it —
because the two single highest-rate constituencies nationally, `Duisburg II` (7.6%) and
`Gelsenkirchen` (6.0%), are Ruhr-area NRW constituencies far from any land border (28-36km
inland), while the constituencies genuinely closest to a border nationally (Schleswig-Holstein,
near Denmark) sit at or below the national rate. Different border segments pull in opposite
directions and cancel out in an aggregate correlation — the high-rate cluster shown in the
choropleth (`Offenburg`, `Lörrach – Müllheim`, near France/Switzerland) is specific to that
stretch, not a general border-proximity effect. `is_land_border_constituency` /
`distance_to_neighbor_country_km` are saved to `const_agg` as covariates, not as a validated
standalone explanation.

# WHO

Before anything ethnicity- or language-based: a look at reviewer identity itself -- how much
of the corpus comes from repeat reviewers (and whether racialised reviews are structurally
different there), and how completely the name-to-country classifier actually covers the
reviewer names it was given, since every comparison in this section rests on that coverage.
Then the reviewer-ethnicity name-origin signal proper: a data-quality check on `org_review_lang`,
then composition of racialised=yes vs. racialised=no reviews using a refined name-origin
categorization built once and reused throughout -- a genuine "Other" separated from a genuine
"Unknown", plus an auditable "Arab League" aggregate -- extended to a three-way comparison
across the entire corpus / social=yes / racialised=yes grains, then a review-language facet,
a German vs. non-German cut against the four content dimensions
(`theme`/`context`/`connotation`/`approach`) refined further by specific language (within
Non-German reviewers) and by top ethnicity, then consolidated into four theoretically-motivated
groups (`Germany`/`Arab`/`Türkiye`/`Europe`) and checked against East/West/Berlin geography.
A closing section then brings this together with the constituency-level rate from WHERE to ask
what's actually driving it.

## Reviewer identity structure: how unique are reviewers and names, and how well does the classifier cover them?

Two things worth establishing before any ethnicity/language analysis: `reviewer_id` is not
equally "one row per person" in every grain -- some reviewers post many reviews, and that
repeat-reviewer share could differ between racialised and non-racialised discourse in a way
that would bias composition comparisons. And `ethnicity_country_name` comes from a name-level
lookup (reviewer names were deduplicated before being sent to the prediction tool, so it was
never asked to classify the same string twice) -- worth knowing not just the overall
classified/unclassified split, but whether that coverage differs for names that repeat across
multiple reviewers versus names that appear once, and whether racialised reviews look any
different from the corpus at large on either front.

In [ ]:
def reviewer_id_breakdown(label, df):
    g = df.dropna(subset=["reviewer_id"]).groupby("reviewer_id").size()
    singleton = (g == 1).sum()
    repeat_counts = g[g > 1]
    excess_row_share = 1 - len(g) / len(df)
    print(f"--- {label} ---")
    print(f"unique reviewer_id: {len(g):,} / {len(df):,} rows")
    print(f"  singleton (1 review): {singleton:,} ({singleton / len(g):.1%})")
    print(f"  repeat (>1 review):    {len(repeat_counts):,} ({len(repeat_counts) / len(g):.1%}), "
          f"{repeat_counts.sum():,} rows ({repeat_counts.sum() / len(df):.1%} of rows)")
    print(f"  excess-row share (rows beyond one-per-id): {excess_row_share:.1%}")
    if len(repeat_counts):
        print(f"  spread among repeat reviewers: median={repeat_counts.median():.0f}, mean={repeat_counts.mean():.1f}, "
              f"90th pct={repeat_counts.quantile(0.9):.0f}, max={repeat_counts.max():,}")
    print()
    return g

id_corpus = reviewer_id_breakdown("Entire corpus", corpus)
id_social = reviewer_id_breakdown("Social = yes", corpus.loc[corpus["social"] == "yes"])
id_racialised = reviewer_id_breakdown("Racialised = yes", corpus.loc[corpus["racialised"] == "yes"])

**Takeaway.** Racialised reviews are overwhelmingly one-off: **98.4% of racialised reviewer_ids have exactly one review** (only 1.6% repeat, accounting for just 3.4% of racialised rows), compared to 86.9% singleton at social grain and 78.8% at corpus grain. The excess-row share (rows beyond one-per-id) drops from 27.1% (corpus) to 15.8% (social) to **1.8%** (racialised) -- repeat-reviewer structure shrinks sharply and monotonically as you move toward the subset that actually matters for this analysis. Where repeat reviewers do exist, the spread is heavily right-skewed (median exactly 2 reviews even at corpus grain, a long thin tail of power-reviewers rather than a bulk of prolific ones) -- so the handful of reviewers with many reviews are not a meaningful confound for racialised-specific comparisons, since that grain barely has any repeat reviewers to begin with.

In [ ]:
def name_classification_breakdown(label, df):
    g = df.dropna(subset=["reviewer_name"]).groupby("reviewer_name").agg(
        n_rows=("reviewer_name", "size"),
        classified=("ethnicity_country_name", lambda s: s.notna().any()),
    )
    singleton = g[g["n_rows"] == 1]
    repeating = g[g["n_rows"] > 1]
    print(f"--- {label} ---")
    print(f"unique names: {len(g):,}")
    print(f"  singleton names: {len(singleton):,} -- classified {singleton['classified'].mean():.1%} "
          f"({singleton['classified'].sum():,} of {len(singleton):,})")
    print(f"  repeating names: {len(repeating):,} -- classified {repeating['classified'].mean():.1%} "
          f"({repeating['classified'].sum():,} of {len(repeating):,})")
    print()
    return g

names_corpus = name_classification_breakdown("Entire corpus", corpus)
names_social = name_classification_breakdown("Social = yes", corpus.loc[corpus["social"] == "yes"])
names_racialised = name_classification_breakdown("Racialised = yes", corpus.loc[corpus["racialised"] == "yes"])

**Takeaway.** Classification coverage sits around 95% for both singleton and repeating names at corpus grain (95.2%/95.3%) and social grain (95.0%/94.1%) -- being shared across multiple reviewers doesn't make a name meaningfully harder to classify at these grains. Racialised grain is lower overall (90.2%) and the gap between singleton (91.1%) and repeating (68.4%) names is much wider -- but repeating names are a tiny slice there (269 of 7,389 unique names), so this is a small-N pattern worth explaining directly rather than a robust trend on its own -- see the next two checks.

In [ ]:
rac = corpus.loc[corpus["racialised"] == "yes"]
g = rac.dropna(subset=["reviewer_name"]).groupby("reviewer_name")["reviewer_id"].agg(["nunique", "size"])
repeating = g[g["size"] > 1]
same_id = repeating[repeating["nunique"] == 1]
diff_id = repeating[repeating["nunique"] > 1]

print(f"racialised repeating names: {len(repeating)}")
print(f"  same reviewer_id every time (genuine repeat reviewer): {len(same_id)} names, {same_id['size'].sum()} rows")
print(f"  different reviewer_ids (coincidental name collision):  {len(diff_id)} names, {diff_id['size'].sum()} rows")
print()
print("most-shared names in the racialised set (by number of distinct reviewer_ids using that name):")
print(diff_id.sort_values("nunique", ascending=False).head(10))

**Takeaway.** Racialised "repeating names" are mostly *not* the same person reviewing twice: 161 of the 269 repeating names (414 of 648 rows) are **coincidental collisions** -- different real reviewers who happen to share a generic name -- versus 108 names (234 rows) that really are one reviewer posting multiple times. The collision list is dominated by bare initials (`M`, shared by 9 different reviewer_ids; `M M`, `S K`, `S` by 6-7 each) and common first-name-only entries (`Chris`, `Tom`, `Michael`, `Ben`), plus one literal placeholder -- `Vorname Nachname`, German for "Firstname Lastname" -- used by 6 different reviewer_ids who apparently never entered a real name. This is consistent with, not contradictory to, the reviewer-ID finding above: racialised reviewers are almost all one-off individually, and the small "repeating name" pool is mostly an artifact of generic/placeholder names, not evidence of a repeat-reviewer subpopulation.

In [ ]:
def unclassified_name_nature(label, df):
    g = df.dropna(subset=["reviewer_name"]).groupby("reviewer_name")["ethnicity_country_name"].apply(lambda s: s.notna().any())
    unclassified_names = g[~g].index.to_series()
    n = len(unclassified_names)
    word_count = unclassified_names.str.split().str.len()
    char_len = unclassified_names.str.len()
    single_word = (word_count == 1).sum()
    non_ascii = unclassified_names.apply(lambda s: any(ord(c) > 127 for c in s)).sum()
    print(f"--- {label}: unclassified names (n={n:,}) ---")
    print(f"  single-word (no space): {single_word:,} ({single_word / n:.1%})")
    print(f"  contains non-ASCII char: {non_ascii:,} ({non_ascii / n:.1%})")
    print(f"  avg word count: {word_count.mean():.2f}, avg char length: {char_len.mean():.1f}")
    top = df[df["reviewer_name"].isin(unclassified_names)]["reviewer_name"].value_counts().head(8)
    print(f"  most common unclassified name-strings:")
    print(top.to_string())
    print()

unclassified_name_nature("Entire corpus", corpus)
unclassified_name_nature("Social = yes", corpus.loc[corpus["social"] == "yes"])
unclassified_name_nature("Racialised = yes", corpus.loc[corpus["racialised"] == "yes"])

**Takeaway.** The classifier fails on the same kind of name everywhere -- this is not a racialised-specific problem. At every grain the most common unclassified strings are bare initials (`M`, `M K`, `A. S.`, `S. B.`) that simply can't be mapped to a country of origin, regardless of how the review is classified. If anything, racialised unclassified names are a *milder* case of the same pattern: less often single-word (37.1% vs. 52.9% corpus, 51.0% social) and less often non-ASCII (4.7% vs. 7.2% corpus, 6.7% social) -- just a much smaller absolute set (720 names) than the other two grains. The only genuinely distinct entries in the racialised unclassified list are one-offs (a concatenated no-space name, a gaming handle) rather than a pattern -- there's no separate failure mode specific to racialised reviews, just the general limit of name-to-country prediction on initials-only names, showing up proportionally less here than elsewhere.

### Draft: paper write-up tables (reviewer-ID structure + name classification)

Two small tables: (1) reviewer_id singleton/repeat structure by grain, (2) name classification coverage by grain, singleton vs. repeating names. Source cells: `whoid-code1`/`whoid-take1` (table 1), `whoid-code2`/`whoid-code3`/`whoid-take2`/`whoid-take3` (table 2).

```latex
% Requires: \usepackage{booktabs}

% ------------------------------------------------------------------
% Table 1 -- reviewer_id structure
% ------------------------------------------------------------------
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\caption{Reviewer-\texttt{id} structure: singleton vs.\ repeat reviewers, by grain.}
\label{tab:reviewer-id-structure}
\begin{tabular}{@{} l r r r r @{}}
\toprule
Grain & Unique IDs & Singleton & Repeat & Excess-row \\
\midrule
Entire corpus     & 923{,}168 & 78.8\% & 21.2\% & 27.1\% \\
Social = yes      & 424{,}420 & 86.9\% & 13.1\% & 15.8\% \\
Racialised = yes  &   7{,}627 & 98.4\% &  1.6\% &  1.8\% \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
Singleton/Repeat = share of unique reviewer\_ids with exactly one review vs.\ more than one. Excess-row = rows beyond one-per-\texttt{id}, as a share of all rows in that grain. Spread among repeat reviewers is heavily right-skewed at every grain (median 2 reviews throughout); 90th percentile / max: corpus 4 / 155, social 3 / 89, racialised 3 / 5.
\end{table}

% ------------------------------------------------------------------
% Table 2 -- name classification coverage
% ------------------------------------------------------------------
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\caption{Name-to-country classification coverage, singleton vs.\ repeating reviewer names, by grain.}
\label{tab:name-classification-coverage}
\begin{tabular}{@{} l r r r @{}}
\toprule
Grain & Unique names & Singleton class.\ & Repeating class.\ \\
\midrule
Entire corpus     & 758{,}558 & 95.2\% & 95.3\% \\
Social = yes      & 356{,}220 & 95.0\% & 94.1\% \\
Racialised = yes  &   7{,}389 & 91.1\% & 68.4\%$^\dagger$ \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
Singleton/Repeating class.\ = \% of unique names in that category with a country prediction. $^\dagger$Only 269 repeating names at racialised grain (vs.\ 194,909 corpus-wide) -- of these, 161 (414 rows) are coincidental collisions between different reviewers sharing a generic name (e.g.\ bare initials, or the literal placeholder \textit{Vorname Nachname}), not genuine repeat reviewers (108 names, 234 rows). Unclassified names follow the same pattern at every grain regardless -- dominated by bare-initials entries (\textit{M}, \textit{M K}, \textit{A. S.}) the classifier cannot map to a country; racialised unclassified names are if anything less often single-word (37.1\% vs.\ 52.9\% corpus) or non-ASCII (4.7\% vs.\ 7.2\% corpus), just a smaller absolute set (720 names).
\end{table}
```

## Reviewer ethnicity facet

**Key question:** does reviewer name-origin composition differ between racialised and non-racialised discourse, and does that shift differ by grain (entire corpus / social / racialised)?

`ethnicity_country_name` is a **name-to-country prediction**, not a verified identity — per the corpus README, 51,935 reviewer names were shared by more than one reviewer ID (affecting 313,200 rows), and age/gender predictions were intentionally excluded as unreliable. We treat this strictly as a name-origin signal for descriptive comparison, not as ground truth about who wrote which review.

Rather than a simple top-10-country cut with everything else dumped into one "Other/unknown" label, we build the categorization we actually want to read from the start: the top-10 non-Arab countries kept individually, an **Arab League aggregate** (individual Arab-League member-state name-origins are too small to clear a top-10 cut on their own, but scattered across ~20 countries they may still be a meaningful minority signal), a genuine **Other** (a real, non-top-10, non-Arab country), and a genuine **Unknown** (a missing prediction) — kept distinct rather than conflated into one catch-all. The top-10 cut is computed on the full `social == 'yes'` population (504,072 rows), not the much smaller racialised=yes subset, so a rare country's ranking doesn't wobble on small-sample noise.

We look at this three ways, in order: first the same categories across all three natural grains, to confirm filtering to `social == 'yes'` doesn't itself skew composition; then a direct test of whether composition differs specifically between racialised=yes and racialised=no (within social=yes); then, holding each group's own size fixed, whether a review from a given group is itself more likely to be racialised at all. This same categorization is reused, unchanged, later in this section for the language facet.

In [ ]:
# --- Arab League aggregate + Other/Unknown split: built once here (on the full social=yes
# population, so the categorization is stable enough to reuse everywhere else in this section)
ARAB_LEAGUE_VARIANTS = {
    "Algeria": ["Algeria", "People's Democratic Republic of Algeria"],
    "Bahrain": ["Bahrain", "Kingdom of Bahrain"],
    "Comoros": ["Comoros", "Union of the Comoros"],
    "Djibouti": ["Djibouti", "Republic of Djibouti"],
    "Egypt": ["Egypt", "Egypt, Arab Republic of"],
    "Iraq": ["Iraq", "Republic of Iraq"],
    "Jordan": ["Jordan", "Hashemite Kingdom of Jordan"],
    "Kuwait": ["Kuwait", "State of Kuwait"],
    "Lebanon": ["Lebanon", "Lebanese Republic"],
    "Libya": ["Libya", "Libyan Arab Jamahiriya", "State of Libya"],
    "Mauritania": ["Mauritania", "Islamic Republic of Mauritania"],
    "Morocco": ["Morocco", "Kingdom of Morocco"],
    "Oman": ["Oman", "Sultanate of Oman"],
    "Palestine": ["Palestine, State of", "State of Palestine", "Palestinian Territory, Occupied"],
    "Qatar": ["Qatar", "State of Qatar"],
    "Saudi Arabia": ["Saudi Arabia", "Kingdom of Saudi Arabia"],
    "Somalia": ["Somalia", "Federal Republic of Somalia"],
    "Sudan": ["Sudan", "Republic of the Sudan"],
    "Syria": ["Syrian Arab Republic", "Syria"],
    "Tunisia": ["Tunisia", "Tunisian Republic"],
    "United Arab Emirates": ["United Arab Emirates"],
    "Yemen": ["Yemen", "Republic of Yemen"],
}

def build_ethnicity_final(country_series, n_top=10):
    """Top-N non-Arab countries, kept individually, + an Arab-League aggregate + explicit
    Other (real, non-top-N, non-Arab country) / Unknown (missing prediction)."""
    present_values = set(country_series.dropna().unique())
    arab_value_map = {
        v: country
        for country, variants in ARAB_LEAGUE_VARIANTS.items()
        for v in variants
        if v in present_values
    }
    arab_mask = country_series.isin(arab_value_map.keys())

    non_arab_mask = ~arab_mask & country_series.notna()
    top_countries = country_series.loc[non_arab_mask].value_counts().head(n_top).index

    final = pd.Series("Other", index=country_series.index)
    final = final.mask(country_series.isin(top_countries), country_series)
    final = final.mask(arab_mask, "Arab")
    final = final.mask(country_series.isna(), "Unknown")
    return final, arab_value_map, list(top_countries)

social_df = corpus.loc[social_mask].copy()
n_ethnicity_missing = social_df["ethnicity_country_name"].isna().mean()
print(f"social=yes reviews missing an ethnicity_country_name prediction: {n_ethnicity_missing:.1%}")

social_df["ethnicity_final"], arab_value_map, top_countries = build_ethnicity_final(social_df["ethnicity_country_name"])
found_arab_countries = sorted(set(arab_value_map.values()))
print(f"Arab League countries matched in ethnicity_country_name: {len(found_arab_countries)}/22")

print(f"\ncategory counts (social=yes, N={len(social_df):,}):")
print(social_df["ethnicity_final"].value_counts())

### Top ethnicities: entire corpus vs. social vs. racialised reviews

Before comparing racialised=yes against racialised=no directly, worth seeing the same `ethnicity_final` categories at all three natural grains side by side: the **entire corpus** (1,265,806 reviews, including the ~40% not judged to be about the social experience at all), **social=yes** (504,072), and **racialised=yes** (7,768). This shows how much composition shifts as we narrow from "everyone reviewing a pool" down to "reviews flagged as racialised" -- the entire-vs-social columns serve as a sanity check that filtering to `social == 'yes'` doesn't itself skew composition much, so any real shift should show up between social and racialised specifically, tested directly in the next subsection.

We reuse the exact category scheme already fit above (`top_countries`, `arab_value_map`, both fit on the `social == 'yes'` population) and simply apply it to the full corpus too, rather than refitting a separate top-10 cut on 1,265,806 rows -- refitting here would risk a third, subtly different category list for the same underlying scheme.

In [ ]:
def apply_ethnicity_final(country_series, top_countries, arab_value_map):
    """Applies an already-fit ethnicity_final scheme (top_countries + arab_value_map, both fit
    on the social=yes population above) to another population, without refitting a new top-10 cut."""
    arab_mask = country_series.isin(arab_value_map.keys())
    final = pd.Series("Other", index=country_series.index)
    final = final.mask(country_series.isin(top_countries), country_series)
    final = final.mask(arab_mask, "Arab")
    final = final.mask(country_series.isna(), "Unknown")
    return final

corpus["ethnicity_final"] = apply_ethnicity_final(corpus["ethnicity_country_name"], top_countries, arab_value_map)
racialised_df["ethnicity_final"] = social_df.loc[racialised_df.index, "ethnicity_final"]

three_way = pd.DataFrame({
    "entire_corpus_pct": corpus["ethnicity_final"].value_counts(normalize=True) * 100,
    "social_pct": social_df["ethnicity_final"].value_counts(normalize=True) * 100,
    "racialised_pct": racialised_df["ethnicity_final"].value_counts(normalize=True) * 100,
}).fillna(0).sort_values("racialised_pct", ascending=False)

print(f"entire corpus: N={len(corpus):,}  |  social=yes: N={len(social_df):,}  |  racialised=yes: N={len(racialised_df):,}\n")
print(three_way.round(2))

three_way.plot(kind="barh", figsize=(8, 7))
plt.xlabel("% of reviews in that population")
plt.title("Ethnicity name-origin composition: entire corpus vs. social=yes vs. racialised=yes")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ethnicity_composition_corpus_social_racialised.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** The entire-corpus and social=yes columns are nearly identical (Germany 39.58% vs. 39.62%, United States 13.49% vs. 14.00%, Other 24.30% vs. 23.27%) -- confirming that filtering down to `social == 'yes'` does not itself introduce a meaningful compositional shift in reviewer name-origin. The real shift happens entirely between social and racialised -- confirmed formally in the next subsection, which tests it directly against racialised=no. Anchored against the full corpus too: **Germany drops sharply** to 27.63% of racialised reviews (vs. ~39.6% at the other two grains), **Other** actually rises slightly (23.27% -> 25.54%) to become nearly as large a category as Germany, **Unknown** rises from 7.74% to 11.07%, and **Arab** (1.92% -> 5.95%, ~3.1x) and **Türkiye** (1.81% -> 4.78%, ~2.6x) roughly triple. Every other top-10 country (Russian Federation, France, Indonesia, Poland, Philippines, Italy, Brazil) sits within a point or two of its social-grain share at all three grains -- the racialised-vs-not shift is concentrated in Germany, Arab, Türkiye, and (to a lesser extent) Unknown, not spread evenly across the whole category list.

### Does composition differ specifically between racialised=yes and racialised=no reviews?

The three-way comparison above shows *where* the shift concentrates (between social and racialised, not between corpus and social); this tests it formally, within `social == 'yes'`, with a chi-square test of independence -- review-level, not deduplicated by reviewer, since we care about the composition of discourse instances, not the reviewer population.

In [ ]:
comp = (
    social_df.groupby("racialised")["ethnicity_final"]
    .value_counts(normalize=True)
    .rename("share")
    .reset_index()
    .pivot(index="ethnicity_final", columns="racialised", values="share")
    .fillna(0)
    .sort_values("yes", ascending=False)
)
print("share of each racialised group's own reviews (each column sums to 1):")
print(comp)

# association test: is ethnicity_final distribution different for racialised=yes vs no?
ct = pd.crosstab(social_df["ethnicity_final"], social_df["racialised"])
chi2, p, dof, _ = stats.chi2_contingency(ct)
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"\nchi2 = {chi2:.1f}, dof = {dof}, p = {p:.2e}, Cramer's V = {cramers_v:.4f}")

comp.plot(kind="barh", figsize=(8, 6))
plt.xlabel("share of reviews social=yes, by racialised=yes vs. no")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ethnicity_composition_racialised_yes_vs_no.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway.** The association is statistically significant (χ² = 1573.1, p ≪ .001) but the effect size is small (Cramér's V = 0.056) — with ~500K reviews, even a small compositional shift clears significance easily, so **the p-value alone is not the interesting number here; the category-level shifts are.** Three shifts stand out: reviewers with a Germany name-origin signal make up 39.8% of racialised=no reviews but only 27.6% of racialised=yes reviews; **Arab** rises from 1.9% to 5.9% (~3.2x) and **Türkiye** rises from 1.8% to 4.8% (~2.7x) — now that Arab is its own auditable category rather than scattered across ~20 individually-tiny countries, its elevation is visibly comparable in magnitude to the already-known Türkiye shift, not a footnote to it. Separating Other from Unknown also clarifies the picture: **Unknown** (missing prediction) is a modest 7.7% of all social reviews and rises only slightly for racialised=yes (7.7% -> 11.1%), while **Other** (a real, non-top-10, non-Arab country) is the largest single category after Germany in both groups (23.2% -> 25.5%) — so a conflated "Other/unknown" label would have mostly been describing real long-tail country signal, not missing data. We deliberately avoid interpreting this as "who racialised discourse targets" or "who writes it" — the ethnicity signal is a name-origin prediction with known collision issues, not a per-review reviewer identity — but the magnitude of the Arab and Türkiye shifts specifically looks like it's worth a closer, qualitative look in the dimension-level work already planned (`approach`/`context` labels), rather than stopping at this aggregate association. One methodological note already put to use above: computing the top-10 cut from the full social population instead of the small racialised-only subset is not just a stability preference — it changes the actual category list (e.g. `India`, which edges out `Brazil` in the 7,768-row racialised subset by a handful of reviews, drops out of the top-10 once ranked against the full 504,072-row population, where `Brazil` is clearly the more common name-origin) — reusing the subset-derived ranking later would have made the two grains of this facet inconsistent with each other.


### Data-quality check: reviews with a missing `org_review_lang`

Before the language facet later in this section, one loose end is worth checking: `org_review_lang` is null for a non-trivial slice of the corpus. The likely explanation is mechanical, not missing data -- Google Maps only populates `trans_review` (and, presumably, its language detector) when it detects the review is written in a language *different* from the display locale used at scrape time. If a review is already in English (the scrape's apparent display locale) there may be nothing to detect or translate, leaving `org_review_lang` null even though the review plainly has a language. We check this directly before relying on `org_review_lang` for anything below, rather than silently letting `pd.crosstab`/`value_counts` drop these rows later (which is what was happening implicitly until now).


In [ ]:
null_mask = corpus["org_review_lang"].isnull()
print(f"corpus rows with null org_review_lang: {null_mask.sum():,} / {len(corpus):,} ({null_mask.mean():.2%})")

# if org_review_lang is null because "nothing needed translating," org_review and trans_review
# should be identical -- check that directly rather than assuming it
same_text = corpus.loc[null_mask, "org_review"] == corpus.loc[null_mask, "trans_review"]
print(f"org_review == trans_review among null-lang rows: {same_text.sum():,} / {null_mask.sum():,} ({same_text.mean():.1%})")

# spot-check the text itself: a simple English-vs-German marker-word scan (not a real
# language detector, just a sanity check) confirms these are English writing, not a missing
# detection on some other language
import re
def has_marker(s, words):
    return any(re.search(rf"\b{w}\b", s, re.IGNORECASE) for w in words)
sample_text = corpus.loc[null_mask, "org_review"].dropna()
german_hit = sample_text.apply(lambda s: has_marker(s, ["der", "die", "das", "und", "sehr", "nicht", "ist"]))
english_hit = sample_text.apply(lambda s: has_marker(s, ["the", "and", "very", "not", "is", "we", "are", "good", "great"]))
print(f"share with a German marker word: {german_hit.mean():.1%}")
print(f"share with an English marker word: {english_hit.mean():.1%} (many of the rest are single-word reviews like \"Top\"/\"Super\")")

print(f"\nsocial=yes reviews affected: {(social_df['org_review_lang'].isnull()).sum():,}")
print(f"racialised=yes reviews affected: {(racialised_df['org_review_lang'].isnull()).sum():,} "
      "(these were previously silently dropped from the ethnicity x language crosstab below, "
      "since pd.crosstab excludes NaN by default)")

# --- correction: null org_review_lang -> "English" ---
corpus["org_review_lang"] = corpus["org_review_lang"].fillna("English")
social_df["org_review_lang"] = corpus.loc[social_df.index, "org_review_lang"]
racialised_df["org_review_lang"] = corpus.loc[racialised_df.index, "org_review_lang"]

print(f"\nnull org_review_lang after correction: {corpus['org_review_lang'].isnull().sum()}")
print(f"'English' now recorded for {(corpus['org_review_lang']=='English').sum():,} corpus-wide, "
      f"{(social_df['org_review_lang']=='English').sum():,} social=yes, "
      f"{(racialised_df['org_review_lang']=='English').sum():,} racialised=yes")


**Takeaway.** The hypothesis holds up completely: **100% of the 65,908 null-`org_review_lang` rows (5.21% of the corpus) have `org_review == trans_review`** -- i.e. Google Maps never ran a translation on them, consistent with them already being in the scrape's display language. The marker-word scan confirms it's specifically English text, not a missing detection on some other language: essentially no German markers (0.2%) vs. a clear English signature (48.1% contain an unambiguous English marker word; most of the remainder are short single-word reviews like "Top"/"Super"/"Must have" that are consistent with English but too short to trigger a multi-word regex). We correct this by filling `org_review_lang` with `"English"` wherever it was null, across `corpus`, `social_df`, and `racialised_df`.

This isn't just a cosmetic fix. It recovers **710 social=yes reviews and 17 racialised=yes reviews** that previously had no language recorded at all -- and since `pd.crosstab` (used throughout the language facet below) silently drops rows with a NaN in either dimension, those 17 racialised reviews weren't just uncategorized, they were **entirely absent** from the ethnicity x language contingency table before this fix, without any visible warning. English now appears as its own (small) category throughout the language facet below.

### Within-group racialised rate: is a review from a given ethnicity group more likely to be racialised at all?

The three-way comparison above answers "of racialised reviews, what share come from each
group" -- a composition question, sensitive to how big each group is in the corpus to begin
with. The complementary question is the individual-level analogue of the constituency-level
ethnicity-share correlation found later in this notebook (Bringing WHO and WHERE together):
**taking each `ethnicity_final` group's own total reviews as the denominator, what share of
*those* reviews are social=yes, and, of the social ones, what share are racialised=yes?** This
holds group size fixed and asks about the group's own base rate directly, rather than its share
of an already-rare event.

In [ ]:
group_stats = corpus.groupby("ethnicity_final").agg(
    total_reviews=("review_id", "size"),
    social_reviews=("social", lambda s: (s == "yes").sum()),
)
group_stats["pct_social"] = group_stats["social_reviews"] / group_stats["total_reviews"] * 100

social_only = corpus.loc[corpus["social"] == "yes"]
rac_by_group = social_only.groupby("ethnicity_final").agg(
    racialised_reviews=("racialised", lambda s: (s == "yes").sum()),
)
group_stats = group_stats.join(rac_by_group)

rates = group_stats.apply(
    lambda r: pd.Series(wilson(r["racialised_reviews"], r["social_reviews"]),
                         index=["racialised_rate", "ci_lo", "ci_hi"]),
    axis=1,
)
group_stats = group_stats.join(rates)
for col in ["racialised_rate", "ci_lo", "ci_hi"]:
    group_stats[col] = group_stats[col] * 100
group_stats = group_stats.rename(columns={"racialised_rate": "racialised_rate_pct"})

overall_rate_pct = (corpus["racialised"] == "yes").sum() / (corpus["social"] == "yes").sum() * 100
group_stats = group_stats.sort_values("racialised_rate_pct", ascending=False)

print(f"baseline (all reviewers combined) racialised rate: {overall_rate_pct:.3f}%\n")
print(group_stats[["total_reviews", "social_reviews", "pct_social",
                    "racialised_reviews", "racialised_rate_pct", "ci_lo", "ci_hi"]].round(3))

fig, ax = plt.subplots(figsize=(8, 5))
colors = np.where(group_stats["racialised_rate_pct"] > overall_rate_pct, "#C44E52", "#4C72B0")
y = np.arange(len(group_stats))
ax.barh(y, group_stats["racialised_rate_pct"], color="#4C72B0",
        xerr=[group_stats["racialised_rate_pct"] - group_stats["ci_lo"],
              group_stats["ci_hi"] - group_stats["racialised_rate_pct"]])
ax.set_yticks(y)
ax.set_yticklabels(group_stats.index)
ax.axvline(overall_rate_pct, color="black", linestyle="--", linewidth=1,
           label=f"overall rate ({overall_rate_pct:.2f}%)")
ax.set_xlabel("racialised rate within this group's own social reviews (%, 95% Wilson CI)")
# ax.set_title("Within-group racialised rate, by ethnicity_final")
ax.invert_yaxis()
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "within_group_racialised_rate.png", dpi=150, bbox_inches="tight")
plt.show()

**Takeaway -- yes, there is a real in-group effect, and it lines up exactly with the composition finding above, not just as an artifact of group size.** Against a baseline racialised rate of 1.541% (all reviewers pooled), **`Arab`-name-origin reviewers have their own reviews flagged racialised at 4.78%** (95% CI 4.38-5.23%) and **`Türkiye` at 4.06%** (3.68-4.49%) -- both roughly 3-4x the baseline, with confidence intervals nowhere near overlapping it. `Germany` sits at the *opposite* end, the single lowest group at **1.08%** (1.03-1.12%), clearly below baseline. Everything else in between forms a gradient rather than a sharp cutoff: `France` (2.44%), `Indonesia` (2.22%), `Unknown` (2.20%) sit moderately above baseline; `Poland`, `Russian Federation`, `Other`, `Philippines`, `Italy` cluster just above or around it (1.49-1.78%); `Brazil` and `United States` sit at or slightly below it (1.27-1.29%).

This directly answers the question the composition view alone couldn't: **it is not simply that `Arab`/`Türkiye` are large enough groups to dominate racialised-review composition -- a review actually written under an `Arab` or `Türkiye` name-origin is intrinsically several times more likely to be racialised than one written under a `Germany` name-origin**, holding each group's own size fixed. The ordering here (`Arab` > `Türkiye` > ... > `Germany` lowest) exactly matches the composition shift found in the three-way table above, which is reassuring: two different ways of asking the question (share of racialised reviews vs. own-group base rate) land on the same story, not two different ones.

### Top languages: entire corpus vs. social vs. racialised reviews

The three-way ethnicity comparison above raises the natural follow-up: does *review language* shift the same way across the same three grains, and is any shift statistically significant? We apply the same method as the ethnicity three-way table -- fit a top-10-language cut once (on `social == 'yes'`, for the same stability reason as before), then apply that fixed category list to all three grains rather than refitting per grain.


In [ ]:
N_TOP_LANG = 10
top_langs = social_df["org_review_lang"].value_counts().head(N_TOP_LANG).index.tolist()
print("top 10 languages (fit on social=yes):", top_langs)

def apply_lang_grouped(series, top_langs):
    return np.where(series.isin(top_langs), series, "Other language")

corpus["lang_grouped_all"] = apply_lang_grouped(corpus["org_review_lang"], top_langs)
social_df["lang_grouped_all"] = apply_lang_grouped(social_df["org_review_lang"], top_langs)
racialised_df["lang_grouped_all"] = apply_lang_grouped(racialised_df["org_review_lang"], top_langs)

three_way_lang = pd.DataFrame({
    "entire_corpus_pct": pd.Series(corpus["lang_grouped_all"]).value_counts(normalize=True) * 100,
    "social_pct": pd.Series(social_df["lang_grouped_all"]).value_counts(normalize=True) * 100,
    "racialised_pct": pd.Series(racialised_df["lang_grouped_all"]).value_counts(normalize=True) * 100,
}).fillna(0).sort_values("racialised_pct", ascending=False)

print(f"\nentire corpus: N={len(corpus):,}  |  social=yes: N={len(social_df):,}  |  racialised=yes: N={len(racialised_df):,}\n")
print(three_way_lang.round(2))

three_way_lang.plot(kind="barh", figsize=(8, 6))
plt.xlabel("% of reviews in that population")
plt.title("Review language composition: entire corpus vs. social=yes vs. racialised=yes")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "language_composition_corpus_social_racialised.png", dpi=150, bbox_inches="tight")
plt.show()

# significance: language x racialised(yes/no), within social=yes -- same design as the ethnicity test above
ct = pd.crosstab(social_df["lang_grouped_all"], social_df["racialised"])
chi2, p, dof, _ = stats.chi2_contingency(ct)
n = ct.values.sum()
cv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"\nlanguage x racialised(yes/no), within social=yes: chi2={chi2:.1f}, dof={dof}, p={p:.2e}, Cramer's V={cv:.4f}")


**Takeaway.** Language behaves differently from ethnicity across these three grains -- worth flagging directly since it's the opposite of what the ethnicity three-way table showed. Entire-corpus and social=yes are **not** near-identical here: `German` rises from 88.58% (entire) to 95.19% (social) and `Other language` (which absorbs `English` -- not in the top-10 fit on social=yes, since it's overwhelmingly a *non-social* review language, per the earlier data-quality fix) drops from 6.44% to 0.83%. This makes sense mechanically: many non-social reviews are short, generic, non-German snippets ("Nice", "Clean", "Good") that get filtered out once you require a review to be about the venue's social experience specifically.

The social-to-racialised shift reproduces the ethnicity-facet finding from a different angle: `German` drops from 95.19% to 90.69% (-4.5pp), while `French` rises 0.91%->3.14% (~3.4x), `Arabic` 0.23%->0.68% (~3x), `Czech` 0.22%->0.62% (~2.8x), and `Polish` 0.54%->0.88% (~1.6x) -- the same France/Arab/Czech/Polish elevation already documented in the ethnicity x language cross-tab, now shown as a direct three-way comparison. The shift is **statistically significant** (language x racialised, within social=yes: chi2=645.4, dof=10, p=3.3e-132) but with a small effect size (Cramér's V=0.036) -- the now-familiar "large N makes a real but modest shift easily detectable" pattern seen throughout this notebook.

### Ethnicity signal x review language

If `ethnicity_country_name` reflects genuine reviewer-identity signal, we might expect some
correspondence with the language a review is actually written in — e.g. a Turkish name-origin
reviewer writing disproportionately more often in Turkish than the corpus baseline. We test this
directly: cross-tabulate the final ethnicity category (built above) against `org_review_lang`, and quantify
the association with Cramér's V (via a chi-square test of independence).

`org_review_lang` has 37 distinct values, most vanishingly rare (`review_id`-level review-language
tags on 1-7 reviews each). A contingency table with that many sparse columns is uninterpretable, so
we collapse any language with fewer than 10 reviews in this subset into `"Other language"` --
the same "don't force detail the data can't support" logic as the Other/Unknown split above.

In [ ]:
df = racialised_df.copy()  # ethnicity_final already attached via the three-way comparison above

LANG_MIN_N = 10

lang_counts = df["org_review_lang"].value_counts()
print("org_review_lang counts:")
print(lang_counts)

kept_langs = lang_counts[lang_counts >= LANG_MIN_N].index
df["lang_grouped"] = np.where(df["org_review_lang"].isin(kept_langs), df["org_review_lang"], "Other language")
print(f"\nlanguages kept individually (n >= {LANG_MIN_N}): {list(kept_langs)}")
print(f"\ngrouped language counts:")
print(df["lang_grouped"].value_counts())


In [ ]:
ct = pd.crosstab(df["ethnicity_final"], df["lang_grouped"])
print("ethnicity_final x lang_grouped contingency table:")
print(ct)

chi2, p, dof, _ = stats.chi2_contingency(ct)
n = ct.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
print(f"\nchi2 = {chi2:.1f}, dof = {dof}, p = {p:.2e}, Cramer's V = {cramers_v:.4f}")

# row-normalised: within each ethnicity category, what % of reviews are in each language
row_pct = ct.div(ct.sum(axis=1), axis=0)
print("\nrow-normalised (share of each ethnicity category's reviews, by language):")
print((row_pct * 100).round(1))


In [ ]:
row_order = [c for c in ct.sum(axis=1).sort_values(ascending=False).index]
row_pct_pct = row_pct.loc[row_order] * 100

fig, ax = plt.subplots(figsize=(11, 7))
sns.heatmap(
    row_pct_pct,
    annot=row_pct_pct.round(1).values,
    fmt=".1f",
    cmap="Greens",
    ax=ax,
    cbar_kws={"label": "row % (share of that ethnicity category's reviews)"},
)
ax.set_xlabel("review language (org_review_lang, grouped)")
ax.set_ylabel("ethnicity category (ethnicity_final)")
ax.set_title("Ethnicity signal x review language (% share of each ethnicity category's own reviews)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "ethnicity_x_language_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** This is *not* a mechanical artifact — the association is both statistically significant and, once judged against baseline rates rather than raw dominance, substantively real (Cramér's V = 0.240, a moderate effect on a 13x15 table; chi2 = 5349.8, dof = 168, p ≈ 0). German is the majority language in almost every row simply because this is a corpus of reviews on German pools, so raw row-dominance by German is expected and uninformative on its own — the meaningful comparison is each category's *own-language share relative to that language's corpus-wide baseline* (German 90.7%, French 3.1%, Russian 1.2%, Polish 0.9%, Arabic 0.7%, Turkish 0.3% of all 7,768 reviews). Against that baseline, several categories show a dramatic, genuine elevation: **France** reviews are 49.7% French-language (vs. 3.1% baseline, and nearly a 50/50 split with German rather than German-dominant), **Poland** reviews are 28.8% Polish (vs. 0.9%, ~33x), and **Russian Federation** reviews are 25.5% Russian (vs. 1.2%, ~22x) — these three categories alone are doing most of the work behind the Cramér's V. **Arab** (8.4% Arabic vs. 0.7% baseline, ~12x) and **Türkiye** (3.5% Turkish vs. 0.3%, ~13x) show the same direction of effect, smaller in absolute share but still a meaningful multiple of baseline. By contrast, categories with no obvious home language in Europe — Brazil, Indonesia, Philippines, United States, Unknown, and the residual Other — sit at 83-99% German, essentially indistinguishable from the corpus baseline, i.e. no detectable own-language elevation there (there is no single "Portuguese" or "Indonesian" tag with enough reviews in this subset to register, so this is partly a construct-availability limit, not necessarily evidence of no signal). **Read honestly: this looks like genuine identity signal for the European/Slavic/Turkish/Arab categories where a corresponding language actually exists in `org_review_lang`, concentrated in a handful of categories rather than uniform across all of them** — not a broad mechanical artifact, but also not a story that holds evenly across every category in the table.

**English, now that it's been recovered.** The data-quality fix earlier in this section (correcting the null-`org_review_lang` rows that were actually English but had never been detected as any language) adds English as a real, if small, category here: 17 of the 7,768 racialised reviews (0.2%). It doesn't change the story above -- 0.2% is nowhere near enough to move the Cramér's V or challenge any of the category-level findings -- but it matters for correctness: before the fix, these 17 rows were not just uncategorized, `pd.crosstab` silently dropped them from this entire table, so the contingency table above was quietly built on 7,751 rows instead of the full 7,768.


## How does racialised discourse differ between German and non-German name-origin reviewers?

**Key question:** does *who* the racialised review's name-origin signal points to relate to *what* the review actually says? We split the racialised=yes subset into **German** (`ethnicity_country_name == "Germany"`, the largest single category throughout this section) vs. **Non-German** (any other known name-origin -- Arab League members, Türkiye, the US, etc., pooled together) vs. **Unknown** (missing prediction, excluded from the comparison below since we can't classify what we don't know). This is a coarser cut than the full `ethnicity_final` scheme above -- deliberately so, since the four content dimensions (`theme`/`context`/`connotation`/`approach`) are already thin at N=7,768, and splitting further than a binary German/non-German cut would leave individual dimension labels (e.g. `Security`, a small `theme` category) too sparse per cell to read.

We look at this two ways: **individually** (does each of the four dimensions, on its own, differ by German-status?), then **collectively** (specific label *combinations* across dimensions -- e.g. `theme == Clothing` reviews framed with `approach == Explicit` vs. `Coded` -- since a real pattern can be invisible in each dimension's marginal distribution and only show up once you condition on a second dimension).


In [ ]:
german_mask = df["ethnicity_country_name"] == "Germany"
unknown_mask = df["ethnicity_country_name"].isna()
non_german_mask = ~german_mask & ~unknown_mask

df["german_status"] = np.select(
    [german_mask, non_german_mask, unknown_mask],
    ["German", "Non-German", "Unknown"],
    default="Unknown",
)
print("german_status counts (of all 7,768 racialised reviews):")
print(df["german_status"].value_counts())
print((df["german_status"].value_counts(normalize=True) * 100).round(1))

# Unknown (missing ethnicity_country_name prediction) can't be classified German/non-German --
# excluded from every comparison below, the same logic as excluding it from a country-level
# comparison in the ethnicity subsections above.
known = df[df["german_status"] != "Unknown"].copy()
baseline_german_pct = (known["german_status"] == "German").mean() * 100
print(f"\nknown-ethnicity racialised reviews used below: {len(known):,} / {len(df):,}")
print(f"baseline German share of known-ethnicity racialised reviews: {baseline_german_pct:.1f}%")


### Individually: does each dimension differ by German-status on its own?

For each dimension, we explode multi-label reviews (same approach as the theme x connotation check in the dimension-content-overview section below), cross-tabulate against `german_status`, and compare the **column-normalised %** (share *within* German reviews vs. share *within* non-German reviews) rather than raw counts, since the two groups are unequal in size (2,146 vs. 4,762). Cramér's V lets us compare effect size across the four dimensions directly, on the same 0-1 scale.


In [ ]:
DIMENSIONS = ["approach", "theme", "connotation", "context"]

dim_results = {}
for dim in DIMENSIONS:
    exploded = known[[dim, "german_status"]].explode(dim)
    ct = pd.crosstab(exploded[dim].to_numpy(), exploded["german_status"].to_numpy())
    chi2, p, dof_, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    cv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    col_pct = ct.div(ct.sum(axis=0), axis=1) * 100
    dim_results[dim] = {"ct": ct, "col_pct": col_pct, "chi2": chi2, "p": p, "cramers_v": cv}
    gap = (col_pct["German"] - col_pct["Non-German"]).sort_values(key=abs, ascending=False)
    print(f"\n--- {dim}: chi2={chi2:.1f}, p={p:.2e}, Cramer's V={cv:.3f} ---")
    print(col_pct.round(1))
    print("biggest German - Non-German percentage-point gaps:")
    print(gap.round(1))

print("\n" + "=" * 60)
print("Cramer's V by dimension, largest effect first:")
print(pd.Series({d: r["cramers_v"] for d, r in dim_results.items()}).sort_values(ascending=False).round(3))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
for ax, dim in zip(axes.flat, DIMENSIONS):
    col_pct, cv, p = dim_results[dim]["col_pct"], dim_results[dim]["cramers_v"], dim_results[dim]["p"]
    gap = (col_pct["German"] - col_pct["Non-German"]).sort_values()
    colors = np.where(gap > 0, "#4C72B0", "#C44E52")
    ax.barh(gap.index.astype(str), gap.values, color=colors)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.set_title(f"{dim} (Cramer's V={cv:.3f}, p={p:.1e})")
    ax.set_xlabel("German % - Non-German % (pct points)\nblue = more German, red = more Non-German")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "german_vs_nongerman_gap_by_dimension.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** Of the 7,768 racialised reviews, **2,146 (27.6%) have a German name-origin signal, 4,762 (61.3%) a non-German one, and 860 (11.1%) are Unknown** (missing prediction, excluded below) -- giving a baseline German share of **31.1%** among the 6,908 reviews with a known origin.

`theme` is doing almost all of the work here. Cramér's V by dimension, largest first: **theme = 0.180**, context = 0.085, approach = 0.050, connotation = 0.032 -- all four clear p<.05 (unsurprising at N=6,908), but effect size makes clear this is really a `theme`/`context` story, not a four-way one. The direction is coherent across both: **German-authored racialised reviews skew toward `Crowd`** (32.6% vs. 23.7% of non-German, +8.9pp) **and `Etiquette`** (21.5% vs. 14.9%, +6.6pp) **themes, framed `generalised`** rather than incident-specific (72.9% vs. 64.2%). **Non-German-authored racialised reviews skew toward `Staff`** (17.3% vs. 9.5% German, i.e. -7.9pp) **and `Access`** (6.2% vs. 2.1%, -4.0pp) **themes, framed as a specific `incident`** more often (35.8% vs. 27.1%). `connotation` is nearly flat (86.8% vs. 87.0% Negative) -- both groups are overwhelmingly negative regardless of name-origin signal, so ethnicity doesn't track *how negative* the discourse is, only *what it's about* and *how concretely it's framed*.

Read plainly: German-origin racialised reviews lean toward broad, situational complaints ("too crowded", "people not following the rules") described in general terms, while non-German-origin racialised reviews lean toward complaints about a specific staff interaction or being restricted/denied access, described as a specific incident. That asymmetry -- generalised/environmental vs. incident/interpersonal -- is a more specific and more interesting story than "both groups write similarly negative reviews," and is worth carrying into the qualitative RQ3 work.

### Collectively: do specific dimension *combinations* stand out?

A single dimension's marginal distribution can hide a sharper pattern that only appears once you condition on a second dimension -- e.g. German-status might look unremarkable for `approach` overall, yet be sharply skewed for one specific `theme`. We cross every pair of the four dimensions (6 pairs total) against `german_status`, keep only label combinations with at least 20 reviews (thinner cells are too noisy for a stable percentage), and rank every surviving combination by how far its German-share deviates from the `baseline_german_pct` computed above -- this is an exploratory scan for "what's interesting," not a pre-registered set of hypotheses, so we report both directions (most non-German-skewed and most German-skewed) rather than a single test.


In [ ]:
from itertools import combinations

MIN_COMBO_N = 20
records = []
for d1, d2 in combinations(DIMENSIONS, 2):
    exploded = known[[d1, d2, "german_status"]].explode(d1).explode(d2)
    grp = exploded.groupby([d1, d2])["german_status"].agg(
        n="size", german_pct=lambda s: (s == "German").mean() * 100
    )
    grp = grp[grp["n"] >= MIN_COMBO_N].reset_index()
    grp = grp.rename(columns={d1: "label_1", d2: "label_2"})
    grp["dim_pair"] = f"{d1} x {d2}"
    records.append(grp)

combo_df = pd.concat(records, ignore_index=True)
combo_df["deviation"] = combo_df["german_pct"] - baseline_german_pct

print(f"combinations tested (n>={MIN_COMBO_N} each) across all 6 dimension pairs: {len(combo_df)}\n")
print(f"Most non-German-skewed combinations (baseline = {baseline_german_pct:.1f}% German):")
print(combo_df.sort_values("german_pct").head(10)[["dim_pair", "label_1", "label_2", "n", "german_pct", "deviation"]].to_string(index=False))
print(f"\nMost German-skewed combinations:")
print(combo_df.sort_values("german_pct", ascending=False).head(10)[["dim_pair", "label_1", "label_2", "n", "german_pct", "deviation"]].to_string(index=False))


In [ ]:
# approach x theme stood out above as the richest pairing (Coded approach skews German across
# most themes) -- shown as a full heatmap rather than just a top-10 list, so the pattern (or lack
# of one) is visible across every theme at once, not just the extremes.
exploded_at = known[["approach", "theme", "german_status"]].explode("approach").explode("theme")
at_pct = exploded_at.groupby(["approach", "theme"])["german_status"].agg(
    n="size", german_pct=lambda s: (s == "German").mean() * 100
)
at_pct = at_pct[at_pct["n"] >= MIN_COMBO_N].reset_index()
pivot = at_pct.pivot(index="approach", columns="theme", values="german_pct")
n_pivot = at_pct.pivot(index="approach", columns="theme", values="n")

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(
    pivot, annot=True, fmt=".0f", cmap="RdBu_r", center=baseline_german_pct, ax=ax,
    cbar_kws={"label": "German % of reviews with this approach x theme combination"},
)
ax.set_title(f"German share by approach x theme (baseline={baseline_german_pct:.1f}%, blank = n<{MIN_COMBO_N})")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "german_share_by_approach_x_theme_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

print("cell sizes (n):")
print(n_pivot)


**Takeaway.** 95 dimension-pair combinations cleared the `n>=20` bar across the 6 possible pairs. The clearest, most repeatable pattern in the scan: reviews using the **`Coded` approach skew German across nearly every theme they co-occur with** -- Clothing (44.7%), Hygiene (41.2%), Etiquette (39.6-40.9%), Belonging (40.4%), Crowd (40.1%), and Safety (37.6%) all sit well above the 31.1% baseline, visible as a near-solid warm row in the `approach x theme` heatmap above. `Access` and `Staff` are the clearest counter-pattern: both stay non-German-skewed **regardless of approach** (Access: Explicit 13.7%, Implicit 11.8%; Staff: Implicit 15.4%) -- consistent with `Access`/`Staff` already being the two most non-German-skewed themes on their own (individual-dimension subsection above), and confirming that's not just an `Explicit`-approach artifact.

The single most extreme cells: **Clothing reviews with a `Positive` connotation are the most non-German-skewed combination found** (10.6% German, n=47, -20.4pp from baseline), and `Access` clusters at the non-German extreme almost regardless of what else co-occurs with it (paired with `incident` context, `Negative` connotation, or `Implicit` approach, all land at 12-18% German). At the other extreme, `Coded` + `Neutral` connotation (46.4% German, n=28) and `Implicit` + `Hygiene` (45.5%, n=55) are the two most German-skewed combinations found.

**Caveat.** Several of the most extreme single combinations sit at n=25-60 -- thin enough that individual percentages are better read as leads for the qualitative RQ3 work than as stable estimates. The `approach x theme` pattern is the one result here with enough independently-repeated, same-direction cells (6 of 9 visible themes landing above baseline for `Coded`) to treat as a genuine pattern rather than a handful of noisy extremes -- worth flagging directly for the `theme`/`approach` qualitative read planned for RQ3.

### Do non-German reviewers use different languages for different labels?

Within German-origin reviews, language isn't a useful further split -- the ethnicity x language table earlier in this section already showed `Germany` reviews are 98.9% German-language. Non-German reviewers, though, genuinely split across many languages (French, Arabic, Turkish, Russian, Polish, German-as-a-second-language, etc.), so it's worth asking a sharper question than the German-status cut above: **does the specific language a non-German-origin reviewer writes in predict which dimension labels show up, beyond what their name-origin alone predicts?** We reuse the `lang_grouped` column already fit earlier in this section (same `LANG_MIN_N=10` cut, on the full racialised subset) rather than refitting it on the smaller Non-German-only slice, restricted here to the 4,762 Non-German rows of `known`.


In [ ]:
non_german = known[known["german_status"] == "Non-German"].copy()
print(f"Non-German known-ethnicity racialised reviews: {len(non_german):,}")
print("\nlang_grouped distribution within this subset (languages this thin should be read cautiously below):")
print(non_german["lang_grouped"].value_counts())


In [ ]:
lang_results = {}
for dim in DIMENSIONS:
    exploded = non_german[[dim, "lang_grouped"]].explode(dim)
    ct = pd.crosstab(exploded[dim].to_numpy(), exploded["lang_grouped"].to_numpy())
    chi2, p, dof_, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    cv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    col_pct = ct.div(ct.sum(axis=0), axis=1) * 100
    lang_results[dim] = {"ct": ct, "col_pct": col_pct, "chi2": chi2, "p": p, "cramers_v": cv}
    print(f"\n--- {dim}: chi2={chi2:.1f}, dof={dof_}, p={p:.2e}, Cramer's V={cv:.3f} ---")

print("Cramer's V by dimension (language, within Non-German), largest first:")
print(pd.Series({d: r["cramers_v"] for d, r in lang_results.items()}).sort_values(ascending=False).round(3))
print("\n(for comparison, Cramer's V by dimension for German-status itself, from the subsection above): "
      f"theme={dim_results['theme']['cramers_v']:.3f}, context={dim_results['context']['cramers_v']:.3f}, "
      f"approach={dim_results['approach']['cramers_v']:.3f}, connotation={dim_results['connotation']['cramers_v']:.3f}")


In [ ]:
# connotation has the largest language effect above -- plotted directly, restricted to languages
# with a reasonably interpretable N (>=40 within this Non-German subset); thinner ones are listed
# but not plotted, consistent with this notebook's usual thin-cell caveat
conn_pct = lang_results["connotation"]["col_pct"]
lang_n = non_german["lang_grouped"].value_counts()
plot_langs = [l for l in conn_pct.columns if lang_n.get(l, 0) >= 40]
print(f"languages shown (n>=40): {plot_langs}")
print(f"languages excluded from the plot for thin N (<40): {[l for l in conn_pct.columns if l not in plot_langs]}")

neg_share = conn_pct.loc["Negative", plot_langs].sort_values()
overall_non_german_negative = (non_german["connotation"].explode() == "Negative").mean() * 100

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(neg_share.index, neg_share.values, color="#C44E52")
ax.axvline(overall_non_german_negative, color="black", linestyle="--", linewidth=1,
           label=f"Non-German overall ({overall_non_german_negative:.1f}%)")
ax.set_xlabel("% Negative connotation")
ax.set_title("Negative-connotation share by review language (Non-German reviewers, n>=40 only)")
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "negative_connotation_share_by_language_nongerman.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** Language, within Non-German reviewers, tells a genuinely different story than name-origin does -- and in one dimension, a stronger one. Cramér's V by dimension for **language** (within the 4,762 Non-German reviews): **connotation = 0.183**, approach = 0.095, context = 0.073, theme = 0.067. Compare that to Cramér's V for **German-status itself** (subsection above): theme = 0.180, context = 0.085, approach = 0.050, connotation = 0.032. The ranking essentially flips: `connotation` -- the dimension name-origin barely touches -- is the dimension language explains best, while `theme` -- name-origin's strongest dimension -- is language's weakest.

The pattern itself is a clean, near-monotonic gradient in the chart above (restricted to the 8 languages with n>=40, out of 4,762): **German-language reviews are the most negative at 89.8%**, then French (80.1%), Dutch (72.7%), Arabic (70.0%), Polish (64.5%), Russian (62.5%), Other language (62.2%), down to **Czech at 56.8%, the least negative**. In other words, among Non-German-origin reviewers, *writing in German specifically* -- not just being a non-German name-origin reviewer generally -- goes with the harshest tone; writing in one's own likely home language goes with a milder one. This is a plausible, substantive lead (language-of-writing as a proxy for something like audience/integration, distinct from and stronger than name-origin for tone specifically), not just noise -- though it's a correlational read, not a causal one.

**Caveat.** Several languages sit right at or just above the `LANG_MIN_N=10` floor (Turkish n=19, Ukrainian n=19, Spanish n=17, English n=16, Italian n=14, Hungarian n=11, Romanian n=10) and were deliberately excluded from the chart for that reason -- e.g. Hungarian (41.7% Negative / 41.7% Neutral) and Ukrainian (36.8% Negative / 36.8% Positive) look like striking outliers in the printed table above, but at n=11 and n=19 these are single-digit-count differences, not stable estimates.

### Breaking "Non-German" into its top ethnicities

The German-status binary above averages together every non-German name-origin into one bucket -- worth checking whether that hides real differences between, say, `Arab`/`Türkiye` and `United States`/`France`. We repeat the same per-dimension test using the full `ethnicity_final` scheme already built earlier in this section (`Germany` + the top-9 individual countries + `Arab` + `Other`, excluding `Unknown`), rather than collapsing everything non-German into one group.


In [ ]:
known_eth = df[df["ethnicity_final"] != "Unknown"].copy()
print(f"known-ethnicity racialised reviews (ethnicity_final scheme): {len(known_eth):,}")
print(known_eth["ethnicity_final"].value_counts())


In [ ]:
eth_results = {}
for dim in DIMENSIONS:
    exploded = known_eth[[dim, "ethnicity_final"]].explode(dim)
    ct = pd.crosstab(exploded[dim].to_numpy(), exploded["ethnicity_final"].to_numpy())
    chi2, p, dof_, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    cv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    col_pct = ct.div(ct.sum(axis=0), axis=1) * 100
    eth_results[dim] = {"ct": ct, "col_pct": col_pct, "chi2": chi2, "p": p, "cramers_v": cv}
    print(f"\n--- {dim}: chi2={chi2:.1f}, dof={dof_}, p={p:.2e}, Cramer's V={cv:.3f} ---")

print("\nCramer's V by dimension (full ethnicity_final scheme), largest first:")
print(pd.Series({d: r["cramers_v"] for d, r in eth_results.items()}).sort_values(ascending=False).round(3))


In [ ]:
# for every (dimension, ethnicity, label) cell with n>=20, how far does that ethnicity's share of
# that label deviate from the grand mean across all known-ethnicity racialised reviews? surfaces
# which specific ethnicities are driving each dimension's overall association, rather than just
# reporting an aggregate Cramer's V.
grand_mean = {dim: known_eth[[dim]].explode(dim)[dim].value_counts(normalize=True) * 100 for dim in DIMENSIONS}

records = []
for dim in DIMENSIONS:
    col_pct = eth_results[dim]["col_pct"]
    n_per_col = eth_results[dim]["ct"].sum(axis=0)
    for eth in col_pct.columns:
        if n_per_col[eth] < 20:
            continue
        for label in col_pct.index:
            records.append({
                "dim": dim, "ethnicity": eth, "label": label,
                "pct": col_pct.loc[label, eth], "grand_mean_pct": grand_mean[dim].get(label, 0),
                "dev": col_pct.loc[label, eth] - grand_mean[dim].get(label, 0), "n_eth": int(n_per_col[eth]),
            })
dev_df = pd.DataFrame(records)
dev_df = dev_df.reindex(dev_df["dev"].abs().sort_values(ascending=False).index)

print("biggest single (dimension, ethnicity, label) deviations from the grand mean, n>=20 only:")
print(dev_df.head(20).to_string(index=False))

# theme x ethnicity heatmap of the same deviation, for a full picture beyond the top-20 list
theme_pivot = eth_results["theme"]["col_pct"].sub(grand_mean["theme"], axis=0)
fig, ax = plt.subplots(figsize=(11, 5))
sns.heatmap(theme_pivot, annot=True, fmt=".0f", cmap="RdBu_r", center=0, ax=ax,
            cbar_kws={"label": "percentage points vs. grand mean"})
ax.set_title("theme: each ethnicity's deviation from the grand-mean theme distribution")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "theme_deviation_from_grand_mean_by_ethnicity_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** Splitting "Non-German" into its 11 constituent categories (6,908 known-ethnicity racialised reviews total) shows the earlier German-vs-non-German gap was real but was averaging over a much sharper underlying pattern. **`Arab` and `Türkiye` are consistently the two most extreme categories, in the same direction as each other, and opposite to `Germany`:** both show far less `Crowd` (Arab 9.8% vs. a 26.5% grand mean across all 12 categories, -16.6pp; Türkiye 12.7%, -13.8pp), far less `Etiquette` (Arab 5.4% vs. 16.9%, -11.6pp; Türkiye 7.8%, -9.1pp), far more `Staff` (Arab 28.1% vs. 14.9%, +13.2pp; Türkiye 27.4%, +12.5pp), more `Access` (Arab 15.1% vs. 4.9%, +10.2pp), and much more `incident`-specific framing (Arab 49.0% vs. 33.1%, +15.9pp; Türkiye 47.3%, +14.2pp) -- visible as the two solidly blue-on-Crowd/Etiquette, red-on-Staff/Access columns in the heatmap above. `Germany` itself deviates in the *opposite* direction on the same labels (Crowd +6.1pp, incident -6.1pp) but by a much smaller margin -- Arab and Türkiye are roughly 2-3x further from the grand mean than Germany is.

The categories in between are not a uniform "everyone else": `Poland` and `Russian Federation` actually lean mildly *toward* the Germany pattern rather than sitting neutral -- Poland shows more `Explicit` approach (84.7% vs. 75.8%, +8.9pp), more `generalised` framing (74.5% vs. 66.9%), more `Crowd` (32.8% vs. 26.5%), and less `Negative` connotation (80.3% vs. 86.9%); Russian Federation shows less `Negative` (79.5% vs. 86.9%) and more `Explicit` (82.7% vs. 75.8%) too. `United States`, `France`, `Italy`, `Indonesia`, `Philippines`, `Brazil`, and `Other` sit closer to the grand mean throughout, without a strong signature of their own in this scan.

Put together: the German-vs-non-German split from the subsection above is best read as **"Arab + Türkiye vs. Germany (+ mildly Poland/Russian Federation)," not a uniform German-vs-everyone-else effect** -- the other seven non-German categories were diluting, not contradicting, the Arab/Türkiye pattern when lumped into one "Non-German" bucket.

**Caveat on comparing Cramér's V across the two subsections.** `theme`'s Cramér's V here (0.098) looks smaller than the German-status version above (0.180), and `context`'s here (0.139) looks larger than its German-status version (0.085) -- but these aren't directly comparable: Cramér's V is normalised by `min(rows, cols) - 1`, which grows from 1 (an 11-theme x 2-group table) to 10 (an 11-theme x 12-category table) once we stop collapsing to a binary. A smaller V here reflects testing a finer-grained, 12-way distinction against the same theme categories, not that theme actually matters less once you look at specific ethnicities -- the raw deviations above (Arab -16.6pp on Crowd alone) are, if anything, sharper than the binary comparison suggested.

### Consolidating into four theoretically-motivated groups: Germany, Arab, Türkiye, Europe

The full 12-category `ethnicity_final` scan above is useful for finding which specific ethnicities drive a pattern, but it's more categories than needed for reuse downstream (e.g. crossing against geography next, or against constituencies later). We consolidate into a smaller, deliberately-chosen scheme rather than an arbitrary top-N cut: **`Germany`**, **`Arab`**, and **`Türkiye`** keep their own groups (both because they're large enough on their own and because Arab and Türkiye reviewers have distinct migration histories in Germany that are worth being able to tell apart later, rather than merging them into one "non-European" catch-all), plus a new **`Europe`** group (`Russian Federation`, `Italy`, `Poland`, `France` -- the top-10 countries above that are European; Germany and Türkiye keep their own groups rather than folding into this one). Everything else (`United States`, `Philippines`, `Indonesia`, `Brazil`, and the long-tail `Other`) falls into a residual **`Other`** group, and `Unknown` (missing prediction) stays separate and excluded from comparisons, as throughout this section.


In [ ]:
EUROPE_COUNTRIES = {"Russian Federation", "Italy", "Poland", "France"}

def to_ethnicity_group4(ethnicity_final_value):
    if ethnicity_final_value in ("Germany", "Arab", "Türkiye", "Unknown"):
        return ethnicity_final_value
    if ethnicity_final_value in EUROPE_COUNTRIES:
        return "Europe"
    return "Other"

df["ethnicity_group4"] = df["ethnicity_final"].map(to_ethnicity_group4)
racialised_df["ethnicity_group4"] = df["ethnicity_group4"]

print("ethnicity_group4 counts (of all 7,768 racialised reviews):")
print(df["ethnicity_group4"].value_counts())
print((df["ethnicity_group4"].value_counts(normalize=True) * 100).round(1))

known4 = df[df["ethnicity_group4"] != "Unknown"].copy()
group4_results = {}
for dim in DIMENSIONS:
    exploded = known4[[dim, "ethnicity_group4"]].explode(dim)
    ct = pd.crosstab(exploded[dim].to_numpy(), exploded["ethnicity_group4"].to_numpy())
    chi2, p, dof_, _ = stats.chi2_contingency(ct)
    n = ct.values.sum()
    cv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    group4_results[dim] = {"ct": ct, "col_pct": ct.div(ct.sum(axis=0), axis=1) * 100, "chi2": chi2, "p": p, "cramers_v": cv}
    print(f"\n--- {dim}: chi2={chi2:.1f}, dof={dof_}, p={p:.2e}, Cramer's V={cv:.3f} ---")
    print(group4_results[dim]["col_pct"].round(1))


**Takeaway.** The four-group consolidation (Germany 2,146 / Arab 462 / Türkiye 371 / Europe 655 / Other 3,274 / Unknown 860) reproduces the same ranking as the full 12-category scan, just easier to read: `theme` is still the strongest dimension (Cramér's V=0.146), then `context` (0.132), then `approach` (0.065) and `connotation` (0.053) trailing well behind. And it makes the earlier "Arab + Türkiye vs. Germany" story directly visible in one table: on `theme`, **Arab and Türkiye track each other closely** (`Staff` 28.1%/27.4%, `Access` 15.1%/10.2%, `Crowd` 9.8%/12.7%, `Etiquette` 5.4%/7.8%) and sit at the opposite end from **Germany** (`Staff` 9.5%, `Access` 2.1%, `Crowd` 32.6%, `Etiquette` 21.5%). `Europe` and `Other` both land in between, close to each other and closer to Germany's profile than to Arab/Türkiye's -- confirming Europe isn't a third pole, it's a middling group. Same pattern on `context`: Arab (49.0% incident) and Türkiye (47.3%) are far more incident-framed than Germany (27.1%), with Europe (31.8%) and Other (33.5%) in between.

### Theme-pair composition: do Germany/Europe and Arab/Turkiye pair different themes together?

Beyond which single themes each group raises most (previous subsection), do multi-themed reviews (a review can carry up to 3 `theme` labels) pair themes together differently depending on who's writing? We check the *rate* of multi-theming by group first (is one group just more likely to pile on themes at all), then which specific theme-pairs dominate each group, then whether the same pair gets a different tone (connotation/context/approach) depending on the writer's `ethnicity_group4`.

In [ ]:
from itertools import combinations

GROUPS4 = ["Germany", "Arab", "Türkiye", "Europe"]
known4_grp = known4[known4["ethnicity_group4"].isin(GROUPS4)].copy()
known4_grp["n_themes"] = known4_grp["theme"].apply(
    lambda c: len(c) if isinstance(c, (list, tuple, set, np.ndarray)) else 1
)

print("multi-theme rate (>=2 themes) by group:")
multi_rate = known4_grp.groupby("ethnicity_group4")["n_themes"].apply(lambda s: (s >= 2).mean())
print(multi_rate.round(3))

ct_multi = pd.crosstab(known4_grp["ethnicity_group4"], known4_grp["n_themes"] >= 2)
chi2_m, p_m, dof_m, _ = stats.chi2_contingency(ct_multi)
v_m = np.sqrt(chi2_m / (ct_multi.values.sum() * (min(ct_multi.shape) - 1)))
print(f"chi2={chi2_m:.2f}, dof={dof_m}, p={p_m:.3f}, Cramer's V={v_m:.3f} "
      f"-- not a real group difference in *how often* reviews are multi-themed")

# --- theme-pair composition, restricted to multi-theme reviews ---
# pairs stored as "A + B" strings (not tuples) -- a Series of tuples confuses
# pandas' crosstab internals (tuple-valued Series reads as MultiIndex-like)
multi = known4_grp[known4_grp["n_themes"] >= 2].copy()

def theme_pairs(themes):
    return [f"{a} + {b}" for a, b in combinations(sorted(set(themes)), 2)]

multi["pairs"] = multi["theme"].apply(theme_pairs)
pair_rows = multi[["pairs", "ethnicity_group4"]].explode("pairs").dropna(subset=["pairs"])

pair_counts_overall = pair_rows["pairs"].value_counts()
top_pairs = pair_counts_overall.head(10).index.tolist()

top_mask = pair_rows["pairs"].isin(top_pairs)
# .to_numpy() (not raw Series): explode() leaves a duplicated index, which crosstab's
# internal alignment can't handle, same fix used in the theme x connotation cell earlier
ct_pairs = pd.crosstab(pair_rows.loc[top_mask, "pairs"].to_numpy(), pair_rows.loc[top_mask, "ethnicity_group4"].to_numpy())
chi2_p, p_p, dof_p, _ = stats.chi2_contingency(ct_pairs)
v_p = np.sqrt(chi2_p / (ct_pairs.values.sum() * (min(ct_pairs.shape) - 1)))
print(f"\ntop-10 theme-pairs x group: chi2={chi2_p:.1f}, dof={dof_p}, p={p_p:.2e}, Cramer's V={v_p:.3f}")

row_pct_pairs = (ct_pairs.div(ct_pairs.sum(axis=1), axis=0) * 100).round(1)
print("\nrow % (share of each pair's occurrences, by group):")
print(row_pct_pairs)

print("\ntop-5 pairs by group, as % of that group's own multi-theme reviews:")
for grp in GROUPS4:
    grp_pairs = pair_rows.loc[pair_rows["ethnicity_group4"] == grp, "pairs"].value_counts()
    grp_n = (multi["ethnicity_group4"] == grp).sum()
    print(f"\n{grp} (n multi-theme={grp_n}):")
    print((grp_pairs.head(5) / grp_n * 100).round(1))


In [ ]:
FOCUS_PAIRS = ["Belonging + Staff", "Access + Staff"]

for pair in FOCUS_PAIRS:
    mask = multi["pairs"].apply(lambda ps: pair in ps)
    sub = multi[mask]
    print(f"\n=== {pair} (n={len(sub)}) ===")
    for dim in ["connotation", "context", "approach"]:
        exploded = sub[[dim, "ethnicity_group4"]].explode(dim)
        ct = pd.crosstab(exploded[dim].to_numpy(), exploded["ethnicity_group4"].to_numpy())
        print(f"\n{dim} x group:")
        print(ct)
        if (ct < 10).any().any():
            print("(>=1 cell < 10 -- too thin to trust a chi-square here, not computing one)")
            continue
        chi2_t, p_t, dof_t, _ = stats.chi2_contingency(ct)
        v_t = np.sqrt(chi2_t / (ct.values.sum() * (min(ct.shape) - 1)))
        print(f"chi2={chi2_t:.2f}, dof={dof_t}, p={p_t:.3f}, Cramer's V={v_t:.3f}")


**Takeaway.** Multi-theming rate itself doesn't differ by group (57-65% carry >=2 themes, chi2=6.52, p=0.089) -- the split isn't in *how often* reviews stack themes. It's in *which* themes get stacked together (top-10 pairs x group: chi2=609.1, dof=27, p=3.7e-111, **Cramér's V=0.311**, substantially stronger than the theme-marginal association found in the previous subsection, V=0.146): Germany/Europe reviews cluster a crowd-behaviour/hygiene complex -- `Crowd+Etiquette`, `Crowd+Safety`, `Etiquette+Safety`, `Clothing+Crowd`/`Clothing+Hygiene` are 72-81% German by occurrence -- while Arab/Türkiye reviews cluster a staff-interaction complex instead: `Belonging+Staff` is Arab's (27.0% of Arab's own multi-theme reviews) and Türkiye's (24.2%) single most common pair, doesn't crack Germany's top-5 at all, and sits at only 8.6% for Europe; `Access+Staff` runs the same way (Arab's #2 pair at 18.3%, Türkiye's #2 at 13.8%). `Language+Staff` runs the other way -- 46.1% of its occurrences are German, 34.8% European, vs. ~9-10% each for Arab/Türkiye.\n\nChecking whether the *same* pair gets a different tone depending on who raises it (`Belonging+Staff` n=221, `Access+Staff` n=122, against connotation/context/approach -- 6 dimension x pair combinations total): connotation stays 88-100% Negative and approach 70-94% Explicit across every group in both pairs, and only one of the six combinations had cells large enough to actually test (`Belonging+Staff` x context: chi2=1.73, p=0.630, not significant -- a mild, non-significant drift toward incident-framing for Arab/Türkiye/Europe vs. Germany). The other five (both pairs' connotation and approach, and `Access+Staff`'s context) had cells of 0-9 -- too thin to trust a chi-square, so no significance claim is made there. The finding worth standing behind is compositional (*which* pairs get raised differs sharply by group, V=0.311) -- not tonal (*how* a shared pair is framed once raised looks the same across groups, as far as this sample size can tell).

### Does geography (East/West/Berlin) modulate any of this?

Before treating `ethnicity_group4` as fixed and moving to constituency-level work, one more cross-check: does *where in Germany* a reviewer is (East, West, or Berlin -- the `region` column built in Section 1) interact with any of the ethnicity-vs-dimension patterns above? We test this within each of the four known groups separately (rather than one combined test), since the hypothesis was specifically about whether e.g. German reviewers *in the East* differ from German reviewers *in the West*, not about East/West differing overall.


In [ ]:
GROUPS4 = ["Germany", "Arab", "Türkiye", "Europe", "Other"]

print("region counts within each ethnicity_group4:")
print(pd.crosstab(known4["ethnicity_group4"], known4["region"]))

region_results = {}
for grp in GROUPS4:
    sub = known4[known4["ethnicity_group4"] == grp]
    print(f"\n{'='*70}\n{grp} (n={len(sub)})\n{'='*70}")
    region_results[grp] = {}
    for dim in DIMENSIONS:
        exploded = sub[[dim, "region"]].explode(dim)
        ct = pd.crosstab(exploded[dim].to_numpy(), exploded["region"].to_numpy())
        chi2, p, dof_, _ = stats.chi2_contingency(ct)
        n = ct.values.sum()
        cv = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
        region_results[grp][dim] = {"chi2": chi2, "p": p, "cramers_v": cv}
        print(f"{dim}: chi2={chi2:.1f}, dof={dof_}, p={p:.3f}, Cramer's V={cv:.3f}")

print("\n" + "=" * 70)
print("summary -- p-value and Cramer's V for every (group, dimension) region test:")
summary = pd.DataFrame({
    (grp, dim): {"p": region_results[grp][dim]["p"], "V": region_results[grp][dim]["cramers_v"]}
    for grp in GROUPS4 for dim in DIMENSIONS
}).T
print(summary.round(3))
print(f"\ntests reaching p<0.05: {(summary['p'] < 0.05).sum()} / {len(summary)}")


**Takeaway.** No -- geography doesn't meaningfully modulate any of this, even at the finer four-group grain. Across all 20 tests (5 groups x 4 dimensions), only **3 reach p<0.05** (`Europe`/theme: p=0.028, V=0.122; `Other`/theme: p<0.001, V=0.072; `Other`/approach: p=0.026, V=0.040) -- and every one of them has a small Cramér's V, the same "statistically detectable at this N but substantively minor" pattern seen elsewhere in this notebook. Critically, **`Germany`, `Arab`, and `Türkiye` -- the three groups the original hypothesis was actually about -- show no significant region effect on any dimension** (all p>=0.155). The specific hypothesis (Germans in the East being more implicit/incident-prone) is not supported at either grain: `context` for German reviewers comes back p=0.404 (binary) and p=0.404 again here (unchanged, since it's the same German subset), and `approach` comes back p=0.817. **Region is not a variable worth carrying into the constituency-level work below** -- it doesn't explain who racialises differently or how.

### Region × dimension, unconditioned on ethnicity

The subsection above tested region's effect *within* each ethnicity group specifically. The simpler, aggregate version -- does `theme`/`connotation` shift by region at all, ignoring who's writing -- is worth having directly too, moved here from what was originally drafted as a Part 2 section since `region` (derived from `state`) needs no political/socioeconomic data. Every one of the 7,768 racialised reviews has a well-populated region with thousands of reviews each, so this is as reliable as the corpus-level dimension analysis in WHEN.

In [ ]:
def chi2_cramers_v(ct):
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    min_expected = expected.min()
    return chi2, p, dof, v, min_expected

# --- region x connotation (single-label, no explode needed) ---
ct_region_conn = pd.crosstab(df["region"], df["connotation"].apply(lambda x: x[0]))
chi2, p, dof, v, min_exp = chi2_cramers_v(ct_region_conn)
print("region x connotation:")
print(ct_region_conn)
print(f"chi2={chi2:.1f}, dof={dof}, p={p:.2e}, Cramer's V={v:.3f}, min expected count={min_exp:.1f}\n")

# --- region x theme (multi-label, explode) ---
region_theme = df[["review_id", "region", "theme"]].explode("theme")
ct_region_theme = pd.crosstab(region_theme["region"].to_numpy(), region_theme["theme"].to_numpy())
chi2, p, dof, v, min_exp = chi2_cramers_v(ct_region_theme)
print("region x theme:")
print(ct_region_theme)
print(f"chi2={chi2:.1f}, dof={dof}, p={p:.2e}, Cramer's V={v:.3f}, min expected count={min_exp:.1f}")


**Takeaway.** Same pattern the ethnicity-conditioned version above found: connotation doesn't shift with region at all (chi2=6.7, dof=4, p=0.152, Cramér's V=0.021 -- negligible), while theme does, modestly (chi2=84.7, dof=20, p=6.2e-10, Cramér's V=0.056 -- statistically real at this N, small in practice). Berlin skews slightly toward `Staff`/`Security` relative to West/East, and West carries proportionally more `Crowd`/`Etiquette` -- differences visible in the raw counts above but not large enough to call region a strong driver of *what* gets said, only *whether* it gets said (the rate-level finding elsewhere in this notebook). Some `theme` cells run below the usual expected-count-5 comfort threshold (min expected count=4.5) -- a caveat noted rather than acted on, consistent with how thin cells are flagged elsewhere in this notebook.

### Draft: paper write-up tables (WHO / cross-dimension significance)



Two tables: (1) ethnicity name-origin composition, social=yes vs racialised=yes, (2) a master chi-square/Cramer's V reference across every WHO cross-tab test run in this notebook (ethnicity x racialised, language x racialised, ethnicity x language, dimension x German-status/ethnicity_final/ethnicity_group4/language-within-Non-German, theme x connotation, theme-pair x group4, geography x group4). Source file: `who_section_findings_tables.tex` (session scratchpad, not reachable from your side -- this cell is the copy to actually use). Move into the paper repo when ready.

```latex
% Requires: \usepackage{booktabs} \usepackage{array}
% All numbers pulled directly from descriptive_analysis.ipynb WHO section
% (takeaway cells + underlying code-cell printed output, cells 03060c4b/
% ff15dc3f/7586cc41/1c5ac726/19e28b8e/bfc2f3b8/dac639cb/ed670d1d/83a62ea8/
% 8bf1a65c/8130b4b0/9e722a72 -- the theme-pair cell added earlier).

% ------------------------------------------------------------------
% Table 1 -- ethnicity name-origin composition: social=yes vs racialised=yes
% ------------------------------------------------------------------
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\caption{Reviewer name-origin ethnicity composition, social=yes vs.\ racialised=yes ($n$=504,072 social=yes; overall association chi2=1573.1, $p\ll0.001$, Cram\'er's V=0.056).}
\label{tab:who-ethnicity-composition}
\begin{tabular}{@{} l r r r @{}}
\toprule
Category & Social (\%) & Racialised (\%) & Fold change \\
\midrule
Germany & 39.62 & 27.63 & 0.70$\times$ \\
Arab & 1.92 & 5.95 & 3.1$\times$ \\
T\"urkiye & 1.81 & 4.78 & 2.6$\times$ \\
Other (real, non-top-10) & 23.27 & 25.54 & 1.1$\times$ \\
Unknown (no prediction) & 7.74 & 11.07 & 1.4$\times$ \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
Remaining $\sim$26\% spread across the other top-10 countries (US, Russia, France, Indonesia, Poland, Philippines, Italy, Brazil), each within 1--2pp of its social-grain share at every grain --- the racialised-vs-not shift is concentrated in Germany/Arab/T\"urkiye/Unknown, not spread evenly.
\end{table}

% ------------------------------------------------------------------
% Table 2 -- master significance / effect-size reference
% ------------------------------------------------------------------
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{3pt}
\caption{WHO section and cross-dimension association tests (chi-square, Cram\'er's V).}
\label{tab:who-significance-master}
\begin{tabular}{@{} l r r l r @{}}
\toprule
Test & $n$ & $\chi^2$ (dof) & $p$ & V \\
\midrule
\multicolumn{5}{@{}l}{\emph{Composition}} \\
Ethnicity (12-cat) $\times$ racialised & 504{,}072 & 1573.1 & $\ll 0.001$ & 0.056 \\
Language (11-cat) $\times$ racialised & 504{,}072 & 645.4 (10) & 3.3e-132 & 0.036 \\
Ethnicity (13) $\times$ language (15) & 7{,}768 & 5349.8 (168) & $\approx 0$ & 0.240 \\
\midrule
\multicolumn{5}{@{}l}{\emph{Dimension $\times$ German-status (binary, $n$=6,908)}} \\
theme & 6{,}908 & 393.1 (10) & 2.8e-78 & 0.180 \\
context & 6{,}908 & --- & $<0.05$ & 0.085 \\
approach & 6{,}908 & 18.2 (2) & 1.1e-04 & 0.050 \\
connotation & 6{,}908 & 7.1 (2) & 0.029 & 0.032 \\
\midrule
\multicolumn{5}{@{}l}{\emph{Dimension $\times$ ethnicity\_final (12-way)}} \\
theme & 6{,}908 & 1163.7 (110) & 1.9e-175 & 0.098 \\
context & 6{,}908 & 165.1 (11) & 1.2e-29 & 0.139 \\
approach & 6{,}908 & 70.8 (22) & 4.9e-07 & 0.070 \\
connotation & 6{,}908 & 57.7 (22) & 4.9e-05 & 0.064 \\
\midrule
\multicolumn{5}{@{}l}{\emph{Dimension $\times$ ethnicity\_group4 (5-way)}} \\
theme & 6{,}908 & 1036.6 (40) & 2.6e-191 & 0.146 \\
context & 6{,}908 & --- & --- & 0.132 \\
approach & 6{,}908 & 60.3 (8) & 4.0e-10 & 0.065 \\
connotation & 6{,}908 & 39.3 (8) & 4.4e-06 & 0.053 \\
\midrule
\multicolumn{5}{@{}l}{\emph{Dimension $\times$ language, within Non-German ($n$=4,762)}} \\
connotation & 4{,}762 & 324.5 (28) & 3.2e-52 & 0.183 \\
approach & 4{,}762 & 89.5 (28) & 2.4e-08 & 0.095 \\
context & 4{,}762 & 31.7 (14) & 0.004 & 0.073 \\
theme & 4{,}762 & 381.2 (140) & 3.2e-24 & 0.067 \\
\midrule
\multicolumn{5}{@{}l}{\emph{Theme content \& theme-pair composition}} \\
theme $\times$ connotation (full subset) & 7{,}768 & 413.9 (20) & 2.6e-75 & 0.122 \\
theme-pair (top-10) $\times$ group4 & 3{,}634 & 609.1 (27) & 3.7e-111 & 0.311 \\
multi-theme rate $\times$ group4 & 3{,}634 & 6.52 (3) & 0.089 & 0.042 \\
\midrule
\multicolumn{5}{@{}l}{\emph{Geography (East/West/Berlin) $\times$ dimension, per group4 (3/20 significant)}} \\
Europe $\times$ theme & 655 & 33.7 (20) & 0.028 & 0.122 \\
Other $\times$ theme & 3{,}274 & 60.7 (20) & $<0.001$ & 0.072 \\
Other $\times$ approach & 3{,}274 & 11.0 (4) & 0.026 & 0.040 \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
Cells marked --- were not reported as an exact statistic in the source cell (V and significance direction are still as stated). Every other geography$\times$dimension test (17/20, including \emph{all} of Germany/Arab/T\"urkiye on every dimension) is not significant, $p \geq 0.097$ throughout, Germany/Arab/T\"urkiye specifically $p \geq 0.155$. All effect sizes here are small-to-moderate by convention (Cram\'er's V $<0.35$); \texttt{theme-pair $\times$ group4} (V=0.311) is the single largest association found anywhere in this section.
\end{table}

```


**Taking stock of the WHO section.**

**Ethnicity facet.** A refined `ethnicity_final` categorization (top-10 non-Arab countries, an auditable `Arab` aggregate, explicit `Other` vs. `Unknown`, fit once on `social=='yes'` for stability) replaces a conflated first-pass cut, and is reused unchanged across every grain in this section. The racialised=yes vs. no comparison shows a small but real shift (Cramér's V=0.056): Germany down (39.8%->27.6%), `Arab` and `Türkiye` both roughly tripling. A three-way check (entire corpus / social / racialised) confirms this shift is concentrated entirely in the racialised grain, not an artifact of the `social=='yes'` filter -- entire-corpus and social percentages are nearly identical for every category. A data-quality fix along the way (65,908 corpus-wide nulls in `org_review_lang` were mechanically mislabeled English, not missing data) recovered 17 racialised reviews `pd.crosstab` had been silently dropping from every language-facet table below.

**Language facets -- two of them, and they don't behave the same way.** The ethnicity x language cross-tab (Cramér's V=0.240) is concentrated in exactly the categories where a genuine identity signal is plausible: `France` (49.7% French, ~16x its own corpus baseline), `Poland`/`Russian Federation` (16-33x baseline), `Arab`/`Türkiye` (~12-13x) -- while categories without an obvious "home language" in `org_review_lang` (Brazil, Indonesia, Philippines, US, Other, Unknown) show no elevation, a limit of what this column can register rather than necessarily an absence of signal. A parallel three-way language table (mirroring the ethnicity one) shows language does **not** stay stable between entire-corpus and social=yes the way ethnicity does -- German rises 88.6%->95.2%, `Other language` (mostly English) drops 6.4%->0.8%, since many non-social reviews are short generic non-German snippets that get filtered out once a review has to be about the venue's social experience. The social->racialised shift is significant (chi2=645, p=3.3e-132, Cramér's V=0.036) and reproduces the same France/Arab/Czech/Polish elevation.

**Content differences, four ways.** German vs. non-German racialised reviews differ mainly on `theme` (Cramér's V=0.180) and `context` (0.085): German-authored reviews skew `Crowd`/`Etiquette`, framed `generalised`; non-German-authored skew `Staff`/`Access`, framed as a specific `incident` -- both equally `Negative` in tone (~87%), so the ethnicity signal tracks *what*/*how concretely*, not *how negative*. Breaking "Non-German" into its actual constituent ethnicities (`Arab`/`Türkiye`/`Europe`/`Other`, consolidated from the full 12-category scheme) shows this is really an **`Arab` + `Türkiye` vs. `Germany`** polarity, not a uniform non-German effect -- `Poland`/`Russian Federation` lean mildly toward the Germany pattern instead, and `US`/`France`/`Italy`/`Indonesia`/`Philippines`/`Brazil`/`Other` show no strong signature either way. Within Non-German reviewers specifically, the *language* written in predicts `connotation` (tone) far better than name-origin does (Cramér's V=0.183 vs. 0.032) -- a clean gradient from German-language (89.8% Negative, harshest) down to Czech (56.8%, mildest among languages with adequate N). Crossing dimension pairs directly also surfaced a lead for RQ3: the `Coded` approach skews German across nearly every theme it co-occurs with, while `Access` stays non-German-skewed almost regardless of what else co-occurs with it.

**Geography checked and ruled out.** East/West/Berlin doesn't meaningfully modulate any of this -- only 3 of 20 (group x dimension) tests reach significance, all with small effect sizes, and none involve `Germany`, `Arab`, or `Türkiye`, the three groups the hypothesis was actually about.

**Still open / deferred**: the `"Other"` category (~140 distinct countries) remains too heterogeneous to interpret as one group; several small-language and thin-ethnicity cells throughout should be read as suggestive, not precise; and categories without a matching "home language" in `org_review_lang` can't be distinguished from genuinely having no language signal at all. The France/Poland/Russian-Federation/Arab/Türkiye language elevation and the `Arab`+`Türkiye`-vs-`Germany` content polarity are the two strongest leads worth carrying into the qualitative dimension-level work for RQ3.

## Content baseline: theme, approach, connotation, context

`approach`, `theme`, `connotation`, and `context` are **multi-label** (each review can carry more than one theme, for instance), stored as arrays and populated only for the 7,768 racialised=yes reviews. This establishes the unconditioned content baseline for WHO's dimension analysis above: every review in this subset is in scope regardless of geography, so frequencies and cross-dimension associations computed at this full N are as reliable as the corpus gets, and give the reference point that the German/non-German and ethnicity-group comparisons earlier in WHO are measured against. It also feeds directly into **RQ3** (the content of racialised discourse); geographic breakdowns of these dimensions are deferred to Part 2's state-level cut, where sample size becomes the binding constraint.

In [ ]:
# racialised_df was already built earlier in this notebook (Part 1 setup); reused here for the dimension analysis
DIMENSIONS = ["approach", "theme", "connotation", "context"]

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
dimension_counts = {}
for ax, dim in zip(axes.flat, DIMENSIONS):
    exploded = racialised_df[dim].explode()
    counts = exploded.value_counts()
    dimension_counts[dim] = counts
    pct_of_reviews = counts / len(racialised_df)
    pct_of_reviews.plot(kind="barh", ax=ax, color="#55A868")
    ax.set_title(f"{dim} (% of the 7,768 racialised reviews carrying this label)")
    ax.set_xlabel("share of reviews")
    ax.invert_yaxis()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "dimension_label_prevalence_overview.png", dpi=150, bbox_inches="tight")
plt.show()

for dim in DIMENSIONS:
    n_labels_per_review = racialised_df[dim].apply(len)
    print(f"{dim}: avg {n_labels_per_review.mean():.2f} labels/review, "
          f"{(n_labels_per_review > 1).mean():.1%} of reviews carry >1 label")


In [ ]:
# theme x connotation association: explode both, one row per (review, theme, connotation) combination
pair_df = racialised_df[["review_id", "theme", "connotation"]].explode("theme").explode("connotation")
# pass .values (not Series) to crosstab: the double explode leaves a duplicated index, which
# crosstab's internal alignment can't handle if given Series that "look" index-aligned but aren't unique
ct_theme_conn = pd.crosstab(pair_df["theme"].to_numpy(), pair_df["connotation"].to_numpy())
ct_theme_conn.index.name, ct_theme_conn.columns.name = "theme", "connotation"
print(ct_theme_conn)

chi2, p, dof, _ = stats.chi2_contingency(ct_theme_conn)
n = ct_theme_conn.values.sum()
cramers_v = np.sqrt(chi2 / (n * (min(ct_theme_conn.shape) - 1)))
print(f"\ntheme x connotation: chi2 = {chi2:.1f}, dof = {dof}, p = {p:.2e}, Cramer's V = {cramers_v:.3f}")

fig, ax = plt.subplots(figsize=(7, 6))
row_pct = ct_theme_conn.div(ct_theme_conn.sum(axis=1), axis=0)
sns.heatmap(row_pct, annot=ct_theme_conn.values, fmt="d", cmap="Greens", ax=ax, cbar_kws={"label": "row %"})
ax.set_title("Theme x connotation (annotated with raw counts, colour = row-normalised %)")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "theme_connotation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** `theme` is the dimension that actually behaves as multi-label in practice (avg 1.77 labels/review, 61.6% of reviews carry more than one), while `approach` and `connotation` are functionally close to single-label (3.6% and 1.4% multi-label respectively) — worth knowing before choosing how to encode each dimension for any downstream modeling. `connotation` is overwhelmingly **Negative** across every theme, but not uniformly so: `Security` and `Safety` are almost purely negative (97.7% and 98.3%), while `Crowd`, `Belonging`, and `Staff` have a noticeably larger (still minority) neutral/positive share. The theme×connotation association is real and moderate (Cramér's V = 0.122, larger than the ethnicity association above) — i.e. which theme a review invokes does meaningfully shift how likely it is to carry a non-negative connotation, which is a legitimate quantitative finding at full sample size regardless of geography. `Crowd` is the single largest theme by a wide margin (3,724 of the theme-label instances), ahead of `Etiquette`, `Staff`, and `Belonging`.

# Bringing WHO and WHERE together: what's actually driving the constituency-level rate?

WHERE established *that* the racialised rate varies sharply by constituency (1.54% overall, up to 8.6% in `Offenburg`) and *that* raw border-proximity doesn't cleanly explain it. WHO established *who* is racialising (`Germany`/`Arab`/`Türkiye`/`Europe`/`Other`) and *what* they racialise about. This section asks the question those two only gesture at separately: does knowing *who* is present in a constituency help explain *why* its rate is high or low -- and is any of this just an artifact of how many reviews/pools a constituency happens to have? This directly matters for whether `racialised_rate` has real signal to work with as a target variable, or would just be modelling review-count noise.


## Is the constituency-level rate just a volume artifact?

The concentration subsection in WHERE already ruled out a *venue-level* artifact (a high rate isn't usually one bad pool skewing an otherwise-quiet constituency -- rate correlates with the *share* of pools affected, not with one dominant pool). But a *constituency-level* volume artifact is a distinct question: do bigger constituencies (more pools, more reviews) simply show different rates because they're bigger, regardless of anything about who's there? We check this directly across all 247 `reliable_n` constituencies (already restricted to >=100 social / >=10 racialised reviews, so this isn't a small-N noise question).


In [ ]:
rel = const_agg[const_agg["reliable_n"]].copy()
print(f"reliable_n constituencies: {len(rel)} / 299")

rho_sr, p_sr = stats.spearmanr(rel["social_reviews"], rel["racialised_rate"])
rho_pc, p_pc = stats.spearmanr(rel["pool_count"], rel["racialised_rate"])
print(f"racialised_rate vs. social_reviews: rho={rho_sr:.3f}, p={p_sr:.2e}")
print(f"racialised_rate vs. pool_count:     rho={rho_pc:.3f}, p={p_pc:.2e}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, col, rho, p in [(axes[0], "social_reviews", rho_sr, p_sr), (axes[1], "pool_count", rho_pc, p_pc)]:
    colors = np.where(rel["reliable_n"], "#4C72B0", "#C44E52")
    ax.scatter(rel[col], rel["racialised_rate"], c=colors, s=16, alpha=0.7)
    ax.set_xscale("log")
    ax.set_xlabel(f"{col} (log scale)")
    ax.set_ylabel("racialised_rate")
    ax.set_title(f"{col} vs. racialised_rate\n(rho={rho:.3f}, p={p:.1e})")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "racialised_rate_vs_social_reviews_and_pool_count.png", dpi=150, bbox_inches="tight")
plt.show()


**Takeaway.** This is real, and it's not small. Across all 247 `reliable_n` constituencies (already restricted to >=100 social / >=10 racialised reviews, so this isn't a thin-sample artifact), `racialised_rate` correlates negatively with both **`social_reviews`** (rho=-0.337, p=5.5e-08) and, more strongly, **`pool_count`** (rho=-0.556, p=1.8e-21) -- constituencies with more pools and more reviews systematically show *lower* racialised rates, visible directly in the funnel-shaped scatter above.

This connects to, and is reassured by, the concentration finding in WHERE: that check showed a high rate isn't usually one bad *venue* skewing an otherwise-quiet constituency (rate tracks the *share* of pools affected, not one dominant pool's contribution). Put together: **this is a genuine constituency-scale pattern, not a venue-scale artifact and not sampling noise** -- something about *larger, more pool-dense* constituencies (whether that's genuinely different demographics, more diffuse/anonymous venues, or something else entirely) goes with a lower rate. Whatever the mechanism, `pool_count` (and to a lesser extent `social_reviews`) is too strong a relationship to ignore as a covariate -- see the urban/rural investigation below, which explains why.

## Who is present in the highest- vs. lowest-rate constituencies?

We compare the 15 highest-rate and 15 lowest-rate `reliable_n` constituencies directly: their volume (confirming the pattern above concretely), the `ethnicity_group4` composition of their racialised reviews, and their dimension profile. Then, rather than relying on just these 30 hand-picked constituencies, we check the same relationship properly across all 247 `reliable_n` constituencies: does a constituency's `Arab`/`Türkiye`/`Germany`/`Europe`/`Other` *share* of racialised reviews correlate with its `racialised_rate` directly?


In [ ]:
high15 = rel.sort_values("racialised_rate", ascending=False).head(15)["constituency_code"].tolist()
low15 = rel.sort_values("racialised_rate", ascending=True).head(15)["constituency_code"].tolist()

print("Top 15 (highest rate) -- volume:")
print(rel[rel["constituency_code"].isin(high15)][["social_reviews", "pool_count"]].agg(["mean", "min", "max"]).round(1))
print("\nBottom 15 (lowest rate) -- volume:")
print(rel[rel["constituency_code"].isin(low15)][["social_reviews", "pool_count"]].agg(["mean", "min", "max"]).round(1))

df["tier"] = np.where(df["constituency_code"].isin(high15), "High-rate (top 15)",
              np.where(df["constituency_code"].isin(low15), "Low-rate (bottom 15)", "Other"))
sub = df[df["tier"] != "Other"]
print(f"\nracialised reviews backing each tier: {sub['tier'].value_counts().to_dict()}")

print("\nethnicity_group4 composition by tier (col %):")
print((pd.crosstab(sub["ethnicity_group4"], sub["tier"], normalize="columns") * 100).round(1))

print("\ntheme by tier (col %) -- the dimension with the sharpest tier difference:")
exploded_theme = sub[["theme", "tier"]].explode("theme")
ct_theme = pd.crosstab(exploded_theme["theme"].to_numpy(), exploded_theme["tier"].to_numpy())
print((ct_theme.div(ct_theme.sum(axis=0), axis=1) * 100).round(1))


In [ ]:
# more rigorous version: correlate each ethnicity_group4's *share* of a constituency's racialised
# reviews directly against racialised_rate, across all 247 reliable_n constituencies -- not just
# the hand-picked top/bottom 15
eth_share = pd.crosstab(
    df.loc[df["ethnicity_group4"] != "Unknown", "constituency_code"],
    df.loc[df["ethnicity_group4"] != "Unknown", "ethnicity_group4"],
    normalize="index",
)
eth_share = eth_share.reindex(rel["constituency_code"]).fillna(0)
merged = rel.set_index("constituency_code").join(eth_share)

print("correlation: a constituency's ethnicity_group4 share (of its racialised reviews) vs. its racialised_rate:")
for grp in ["Germany", "Arab", "Türkiye", "Europe", "Other"]:
    rho, p = stats.spearmanr(merged[grp], merged["racialised_rate"])
    rho_pc, p_pc = stats.spearmanr(merged[grp], merged["pool_count"])
    print(f"{grp:10s}: vs. racialised_rate  rho={rho:+.3f} p={p:.3f}   |   vs. pool_count  rho={rho_pc:+.3f} p={p_pc:.3f}")


**Takeaway.** The volume pattern holds concretely for these specific constituencies: the low-rate tier isn't smaller or thinner, **it's bigger** -- mean 3,195 social reviews / 32.4 pools vs. the high-rate tier's 1,427 / 10.9 (roughly 2-3x more of both). This directly answers "is it just volume": partly yes (see above), but not *only* volume -- composition differs too, and does so systematically, not just for these 30 hand-picked constituencies.

The rigorous version (all 247 `reliable_n` constituencies, not just the top/bottom 15) shows a constituency's `racialised_rate` correlates with its **`ethnicity_group4` composition directly**: **`Arab` share (rho=+0.211, p=0.001)** and **`Türkiye` share (rho=+0.180, p=0.004)** both predict a *higher* rate; **`Germany` share (rho=-0.161, p=0.011)** predicts a *lower* one; `Europe` trends the same direction as Arab/Türkiye but doesn't clear significance (rho=+0.105, p=0.100); **`Other` (the long-tail/non-top-ethnicity group) shows no relationship at all (rho=-0.036, p=0.573)**. This specificity matters -- it isn't "any non-German presence," it's specifically Arab and Türkiye presence that tracks a higher rate.

**One genuinely counter-intuitive result, worth flagging rather than smoothing over.** `theme` shows the sharpest tier difference, but in a direction that doesn't simply mirror the individual-level ethnicity/theme finding earlier in this section: the **low-rate tier is far more `Staff`-themed (18.5% vs. 8.6%)**, while the high-rate tier skews toward `Crowd` (31.3% vs. 25.0%) and `Clothing` (11.4% vs. 7.4%) -- even though the high-rate tier also has *more* Arab/Türkiye share, and Arab/Türkiye reviews were individually shown (earlier in WHO) to skew *toward* `Staff`. This is a real example of the ecological-fallacy caution this notebook has flagged before: a constituency-level aggregate pattern doesn't have to match the individual-level pattern that (partly) composes it. It's a genuine, non-obvious lead worth a closer qualitative look, not something this descriptive scan can fully explain.

**On "have some places figured out how to handle it."** This data can't distinguish that hypothesis from a duller one (different venue mix, different demographic composition, or the volume-dilution effect above) -- all three predict the same observable pattern. Flagging it as an open question rather than a finding.

### Following up on the Staff-theme puzzle directly

The tier comparison above flagged something that looked contradictory: the low-rate tier is *more* `Staff`-themed even though it has *less* Arab/Türkiye presence (the two groups that individually skew `Staff`-heavy). Two explanations are possible: either the ethnicity mix shift is doing this indirectly (a compositional/ecological effect), or every ethnicity group independently writes more about `Staff` in low-rate constituencies (a genuine place-level effect that composition can't explain). We test this directly with a standard decomposition: **hold each tier's actual ethnicity mix fixed, and ask what the low tier's `Staff`-share would be if it kept the high tier's within-group `Staff` rates (and vice versa)** -- if the gap mostly disappears under this swap, it's compositional; if it survives, it's a real within-group/place effect.


In [ ]:
exploded_theme = sub[["theme", "tier", "ethnicity_group4"]].explode("theme")

# within-group Staff-mention share, by tier
denom = exploded_theme.groupby(["tier", "ethnicity_group4"]).size()
staff_n = exploded_theme[exploded_theme["theme"] == "Staff"].groupby(["tier", "ethnicity_group4"]).size()
staff_share_by_group = (staff_n / denom).fillna(0)
print("Staff theme-mention share, by ethnicity_group4 x tier:")
print(staff_share_by_group.unstack("tier").round(3))

overall = exploded_theme.groupby("tier")["theme"].apply(lambda s: (s == "Staff").mean())
print(f"\noverall Staff share: high tier={overall['High-rate (top 15)']:.3f}, low tier={overall['Low-rate (bottom 15)']:.3f}")

# decomposition: high tier's ethnicity mix (by theme-mention count) x low tier's within-group rates
mix_high = denom["High-rate (top 15)"] / denom["High-rate (top 15)"].sum()
counterfactual = sum(
    mix_high.get(g, 0) * staff_share_by_group.get(("Low-rate (bottom 15)", g), 0)
    for g in mix_high.index
)
print(f"\ncounterfactual: high tier's ethnicity mix + low tier's within-group Staff rates = {counterfactual:.3f}")
print(f"(compare: actual high tier = {overall['High-rate (top 15)']:.3f}, actual low tier = {overall['Low-rate (bottom 15)']:.3f})")
print("If this counterfactual sits close to the LOW tier's actual share, composition explains "
      "little of the gap -- it's a within-group effect, not an ethnicity-mix effect.")


In [ ]:
from scipy.stats import rankdata
import statsmodels.api as sm

# does this hold as a continuous relationship across all 247 reliable_n constituencies,
# not just the hand-picked top/bottom 15?
exploded_all = df[["constituency_code", "theme"]].explode("theme")
staff_by_const = exploded_all.groupby("constituency_code")["theme"].apply(lambda s: (s == "Staff").mean())
rel = rel.set_index("constituency_code") if rel.index.name != "constituency_code" else rel
rel["staff_share"] = staff_by_const.reindex(rel.index)

rho_rate, p_rate = stats.spearmanr(rel["staff_share"], rel["racialised_rate"])
rho_pool, p_pool = stats.spearmanr(rel["staff_share"], rel["pool_count"])
print(f"staff_share vs. racialised_rate (n=247): rho={rho_rate:.3f}, p={p_rate:.2e}")
print(f"staff_share vs. pool_count (n=247):      rho={rho_pool:.3f}, p={p_pool:.2e}")

rx, ry, rc = rankdata(rel["staff_share"]), rankdata(rel["racialised_rate"]), rankdata(rel["pool_count"])
resid_x = sm.OLS(rx, sm.add_constant(rc)).fit().resid
resid_y = sm.OLS(ry, sm.add_constant(rc)).fit().resid
rho_partial, p_partial = stats.pearsonr(resid_x, resid_y)
print(f"partial (controlling for pool_count):    rho={rho_partial:.3f}, p={p_partial:.3f}")

# is Staff just structurally more "incident"-coded than other themes, regardless of who/where?
exploded_ctx = df[["theme", "context"]].explode("theme").explode("context")
ct_ctx = pd.crosstab(exploded_ctx["theme"].to_numpy(), exploded_ctx["context"].to_numpy())
ctx_pct = (ct_ctx.div(ct_ctx.sum(axis=1), axis=0) * 100).sort_values("incident", ascending=False)
print("\nshare of each theme's mentions framed as 'incident' (all 7,768 racialised reviews):")
print(ctx_pct.round(1))

print(f"\nstate composition -- high-rate 15: {rel.loc[high15, 'state'].value_counts().to_dict()}")
print(f"state composition -- low-rate 15: {rel.loc[low15, 'state'].value_counts().to_dict()}")


**Resolved: it's a within-group effect, not a compositional one -- composition, if anything, works against it.** Every single ethnicity group independently writes about `Staff` roughly 2x more in the low-rate tier than the high-rate tier: `Germany` 5.2%->14.4%, `Arab` 22.0%->44.4%, `Türkiye` 18.5%->25.0%, `Europe` 9.5%->24.0%, `Other` 8.4%->19.9%, `Unknown` 6.8%->12.1%. The decomposition confirms this directly: giving the low tier the *high* tier's ethnicity mix but keeping the low tier's own within-group `Staff` rates still predicts **19.5%** -- almost exactly the actual observed low-tier share (18.5%), and nowhere near the high tier's 8.6%. The ethnicity-mix shift explains essentially none of the gap.

This holds up as a genuine, continuous relationship too, not just a top/bottom-15 artifact: across all 247 `reliable_n` constituencies, a constituency's `Staff`-share of its racialised reviews correlates with a *lower* `racialised_rate` directly (rho=-0.200, p=1.6e-03), and **survives controlling for `pool_count`** (partial rho=-0.155, p=0.015) even though `pool_count` itself has a small positive relationship with `Staff`-share (rho=+0.131, p=0.040) -- so this isn't simply riding on the volume effect either.

**Part of the mechanism is structural, not about who's writing at all**: `Staff` is intrinsically the second-most "incident"-coded theme in the entire corpus (53.1% of `Staff` mentions are framed as a specific `incident`, vs. only 20.9% for `Crowd`) -- a `Staff` complaint is almost definitionally about a specific interaction, while a `Crowd` complaint is almost definitionally ambient/general. That single fact mechanically links the low-rate tier's `Staff`-tilt to its earlier-noted incident-tilt (32.7% vs. 28.3%) -- these aren't two independent findings, they're the same one showing up in two dimensions.

**What this doesn't resolve**: *why* low-rate (and larger, more pool-dense) constituencies skew toward `Staff`-type racialised content regardless of who's writing it. The state breakdown is suggestive context, not an answer -- the high-rate 15 are dominated by Ruhr-area NRW and Berlin (urban), the low-rate 15 by Bayern/Mecklenburg-Vorpommern/Schleswig-Holstein (rural/coastal) -- consistent with a genuine urban/rural or venue-type difference, but this notebook can't distinguish that from the "rare racialised commentary selects for more clearly incident-worthy content" hypothesis floated earlier. Both remain open, qualitative leads for RQ3, not something this descriptive scan settles -- but the one thing now settled is that **it is not an artifact of which ethnicities are present.**

## Where is each ethnicity group most dominant?

The subsections above show *that* a constituency's `ethnicity_group4` composition correlates with its rate -- this asks the more concrete WHO x WHERE question directly: *which specific constituencies* have the highest share of `Arab`, `Türkiye`, `Europe`, or `Germany` in their racialised reviews? We restrict to constituencies with at least 15 known-ethnicity racialised reviews (a floor for a stable share, well below `reliable_n`'s own bar since we're ranking within-constituency composition, not re-testing the rate itself).


In [ ]:
MIN_RACIALISED_FOR_SHARE = 15
known_all = df[df["ethnicity_group4"] != "Unknown"]
eth_share_dom = pd.crosstab(known_all["constituency_code"], known_all["ethnicity_group4"], normalize="index")
n_known_dom = known_all.groupby("constituency_code").size()
eth_share_dom = eth_share_dom[n_known_dom.reindex(eth_share_dom.index) >= MIN_RACIALISED_FOR_SHARE]

meta_cols = ["constituency", "state", "racialised_rate", "racialised_reviews", "reliable_n"]
eth_share_dom = eth_share_dom.join(const_agg.set_index("constituency_code")[meta_cols])
print(f"constituencies with >={MIN_RACIALISED_FOR_SHARE} known-ethnicity racialised reviews: {len(eth_share_dom)} / 299")

for grp in ["Arab", "Türkiye", "Europe", "Germany"]:
    print(f"\ntop 8 constituencies by {grp} share of racialised reviews:")
    top = eth_share_dom.sort_values(grp, ascending=False).head(8)
    print(top[[grp, "constituency", "state", "racialised_reviews", "racialised_rate", "reliable_n"]].round(3).to_string(index=False))

print("\nstate composition of the top-10-by-share constituencies, per group (a rough geographic-clustering check):")
for grp in ["Arab", "Türkiye", "Europe", "Germany"]:
    top10_states = eth_share_dom.sort_values(grp, ascending=False).head(10)["state"].value_counts()
    print(f"{grp:10s}: {top10_states.to_dict()}")


**Takeaway.** Two of the four groups show real geographic clustering; two don't. **`Europe` dominance clusters in Baden-Württemberg** (4 of the top 10 constituencies, including `Lörrach – Müllheim` itself -- the known high-rate hotspot near the France/Switzerland border, at 21.9% Europe share and a large, reliable N=228) -- consistent with the France-border proximity signal already found in the WHO language facet. **`Germany` dominance clusters in eastern states** (`Sachsen`/`Thüringen`/`Sachsen-Anhalt` account for 5 of the top 10, e.g. `Zwickau` at 71.4% Germany share), consistent with the East's lower foreign-population share found throughout this notebook. **`Arab` and `Türkiye` dominance, by contrast, show no comparable single-state cluster** -- `Arab`'s top 10 spans 5 states (NRW appearing most often, 5 of 10) and `Türkiye`'s spans 7 states (NRW again most frequent, 3 of 10) -- concentrated in specific constituencies rather than specific regions.

Read together with the earlier density finding: `Europe`'s clustering in southwestern Baden-Württemberg and `Germany`'s clustering in the (less dense, more rural) East both plausibly ride on the same geography/density story documented earlier in this section. `Arab`/`Türkiye`'s more scattered, constituency-specific pattern (rather than state-level clustering) is consistent with immigration settlement patterns following specific cities/industrial areas (era Ruhr-area industrial recruitment) rather than a broad regional gradient -- a genuine difference in *kind* of geographic pattern, not just degree.

## Does this connect to the border-distance finding from WHERE?

WHERE found no significant relationship between `distance_to_neighbor_country_km` and
`racialised_rate`, because different border segments pull in opposite directions (the
France/Switzerland stretch high, the Denmark stretch average-to-low). Now that we have
`ethnicity_group4` and the land-border-only distance measure, we can check the more specific
question directly: is *who* is racialising different near the border vs. in the interior?

In [ ]:
merged["is_land_border_constituency"] = const_agg.set_index("constituency_code").loc[merged.index, "is_land_border_constituency"]
merged["distance_to_neighbor_country_km"] = const_agg.set_index("constituency_code").loc[merged.index, "distance_to_neighbor_country_km"]

print("ethnicity_group4 share vs. distance_to_neighbor_country_km (n=247 reliable_n):")
for grp in ["Germany", "Arab", "Türkiye", "Europe", "Other"]:
    rho, p = stats.spearmanr(merged[grp], merged["distance_to_neighbor_country_km"])
    print(f"{grp:10s}: rho={rho:+.3f} p={p:.3f}")

print(f"\nmean ethnicity_group4 share: true land-border (n={merged['is_land_border_constituency'].sum()}) "
      f"vs. interior (n={(~merged['is_land_border_constituency']).sum()}):")
print((merged.groupby("is_land_border_constituency")[["Germany", "Arab", "Türkiye", "Europe", "Other"]].mean() * 100).round(1))

print(f"\ntrue land-border constituencies in the top-15/bottom-15 rate tiers (corrected measure): "
      f"top15={merged.loc[high15, 'is_land_border_constituency'].sum()}, "
      f"bottom15={merged.loc[low15, 'is_land_border_constituency'].sum()}")


**Takeaway.** No -- once the coastline-inclusion bug is fixed, **border-proximity essentially doesn't matter for who's racialising either.** Ethnicity share vs. `distance_to_neighbor_country_km` is at best borderline: `Germany` share rises slightly farther from a border (rho=+0.126, p=0.047), `Arab` share falls slightly (rho=-0.115, p=0.072, just short of significance) -- both tiny effects. Comparing true land-border constituencies (n=39) directly against interior ones (n=208), the composition is **nearly identical**: Germany 30.1% vs. 32.0%, Arab 7.6% vs. 6.9%, Türkiye 4.0% vs. 5.4%, Europe 8.8% vs. 8.4%, Other 49.5% vs. 47.3% -- no meaningful gap anywhere. And among the rate tiers, true land-border constituencies are a small minority either way (2/15 high-rate, 4/15 low-rate) -- not the lopsided pattern the *uncorrected* coastline-inclusive measure seemed to suggest.

Combined with the corrected null correlation between distance and rate directly (previous subsection above), the conclusion is consistent throughout: **whatever makes `Offenburg`/`Lörrach – Müllheim`/`Saarbrücken` distinctive is specific to those particular places, not a general "being near a border" effect** -- and it isn't explained by who's racialising there either.

### Draft: paper write-up tables (Bringing WHO+WHERE together)

Two tables: (1) constituency-level correlations with racialised_rate (pool_count, social_reviews, ethnicity_group4 shares, border distance), (2) descriptive summary (volume tiers, theme-mix tier reversal, geographic clustering by group, land-border composition). Source file: `bringing_together_findings_tables.tex` (session scratchpad, not reachable from your side -- this cell is the copy to actually use). Move into the paper repo when ready.

```latex
% Requires: \usepackage{booktabs} \usepackage{array}
% All numbers pulled directly from descriptive_analysis.ipynb's "Bringing WHO
% and WHERE together" section takeaway cells (90d2ee84/07c72082/60eec58c/3dc05a02).

% ------------------------------------------------------------------
% Table 1 -- constituency-level correlations
% ------------------------------------------------------------------
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\caption{Constituency-level correlations with \texttt{racialised\_rate} (Spearman $\rho$, 247 \texttt{reliable\_n} constituencies unless noted).}
\label{tab:bringing-together-correlations}
\begin{tabular}{@{} l r r c @{}}
\toprule
Predictor & $\rho$ & $p$ & Sig. \\
\midrule
pool\_count & $-0.556$ & 1.8e-21 & *** \\
social\_reviews & $-0.337$ & 5.5e-08 & *** \\
Arab share (ethnicity\_group4) & \phantom{-}0.211 & 0.001 & ** \\
T\"urkiye share & \phantom{-}0.180 & 0.004 & ** \\
Germany share & $-0.161$ & 0.011 & * \\
Europe share & \phantom{-}0.105 & 0.100 & ns \\
Other share & $-0.036$ & 0.573 & ns \\
\midrule
\multicolumn{4}{@{}l}{\emph{Ethnicity share vs.\ distance to nearest land border (299 constituencies)}} \\
Germany share & \phantom{-}0.126 & 0.047 & * \\
Arab share & $-0.115$ & 0.072 & ns \\
\bottomrule
\end{tabular}

\vspace{2pt}
\raggedright\footnotesize
$^{*}p<0.05$, $^{**}p<0.01$, $^{***}p<0.001$, ns = not significant. \texttt{pool\_count}/\texttt{social\_reviews} correlate negatively with the rate (larger, more pool-dense constituencies show a lower rate); it is specifically Arab/T\"urkiye presence, not non-German presence generally, that predicts a higher rate. Both border-distance correlations are small and one is borderline non-significant --- consistent with WHERE's own null border-distance finding.
\end{table}

% ------------------------------------------------------------------
% Table 2 -- descriptive summary (volume tiers, geography, border composition)
% ------------------------------------------------------------------
\begin{table}[htbp]
\centering
\footnotesize
\setlength{\tabcolsep}{4pt}
\caption{Bringing WHO and WHERE together: descriptive summary.}
\label{tab:bringing-together-summary}
\begin{tabular}{@{} p{0.4\linewidth} p{0.54\linewidth} @{}}
\toprule
Finding & Value \\
\midrule
Volume, low-- vs.\ high-rate tier (15 each) & mean 3{,}195 social reviews / 32.4 pools (low) vs.\ 1{,}427 / 10.9 (high) --- low-rate tier is 2--3$\times$ bigger on both \\
Theme mix, low-- vs.\ high-rate tier & Staff 18.5\% (low) vs.\ 8.6\% (high); Crowd 25.0\% vs.\ 31.3\%; Clothing 7.4\% vs.\ 11.4\% --- reverses the individual-level Arab/T\"urkiye$\to$Staff pattern found in WHO (ecological-fallacy caveat) \\
Europe dominance, geography & clusters in Baden-W\"urttemberg (4/10 top constituencies), incl.\ L\"orrach--M\"ullheim itself (21.9\% Europe share, $n$=228) \\
Germany dominance, geography & clusters in the East (Sachsen/Th\"uringen/Sachsen-Anhalt, 5/10 top constituencies), e.g.\ Zwickau at 71.4\% \\
Arab/T\"urkiye dominance, geography & no single-state cluster --- Arab spans 5 states (NRW most frequent, 5/10), T\"urkiye spans 7 (NRW 3/10); concentrated in specific cities, not a regional gradient \\
Land-border ($n$=39) vs.\ interior ($n$=208) composition & Germany 30.1\% vs.\ 32.0\%; Arab 7.6\% vs.\ 6.9\%; T\"urkiye 4.0\% vs.\ 5.4\%; Europe 8.8\% vs.\ 8.4\%; Other 49.5\% vs.\ 47.3\% --- no meaningful gap anywhere \\
Land-border representation in rate tiers & only 2/15 high-rate and 4/15 low-rate constituencies are true land-border seats --- a small minority either way \\
\bottomrule
\end{tabular}
\end{table}

```


## Investigating what's still open: is this urban vs. rural?

Two candidate explanations were left open above for why low-rate (larger, more pool-dense) constituencies skew `Staff`-themed: a genuine urban/rural or venue-type difference, or a "rare racialised commentary selects for more notable content" selection effect. We can actually test the first one directly -- `political` (loaded in Section 0, already used for the electoral columns) also carries `population_density_per_km2` and `area_km2` from the federal baseline, which haven't been pulled into this notebook before now. This also lets us check something the synthesis above didn't: whether the `ethnicity_group4` composition finding is itself partly an urban/rural proxy, since immigrant populations are well known to concentrate in German cities.


In [ ]:
from scipy.stats import rankdata
import statsmodels.api as sm

def partial_spearman(x, y, control):
    """Rank-residualize x and y against control, then Spearman-correlate the residuals."""
    rx, ry, rc = rankdata(x), rankdata(y), rankdata(control)
    Z = sm.add_constant(rc)
    x_resid = sm.OLS(rx, Z).fit().resid
    y_resid = sm.OLS(ry, Z).fit().resid
    return stats.pearsonr(x_resid, y_resid)

political_idx = political.set_index("constituency_code")
rel["population_density_per_km2"] = rel.index.map(political_idx["population_density_per_km2"])
rel["area_km2"] = rel.index.map(political_idx["area_km2"])
rel["foreign_population"] = rel.index.map(political_idx["foreign_population"])
rel["total_population"] = rel.index.map(political_idx["total_population"])
rel["foreign_population_pct"] = rel["foreign_population"] / rel["total_population"] * 100

rho_dp, p_dp = stats.spearmanr(rel["population_density_per_km2"], rel["pool_count"])
rho_dr, p_dr = stats.spearmanr(rel["population_density_per_km2"], rel["racialised_rate"])
rho_ds, p_ds = stats.spearmanr(rel["population_density_per_km2"], rel["staff_share"])
print(f"population_density_per_km2 vs. pool_count:     rho={rho_dp:.3f}, p={p_dp:.2e}")
print(f"population_density_per_km2 vs. racialised_rate: rho={rho_dr:.3f}, p={p_dr:.2e}")
print(f"population_density_per_km2 vs. staff_share:      rho={rho_ds:.3f}, p={p_ds:.3f}")

print(f"\nhigh-rate 15 -- density: mean={rel.loc[high15, 'population_density_per_km2'].mean():.0f}, "
      f"range {rel.loc[high15, 'population_density_per_km2'].min():.0f}-{rel.loc[high15, 'population_density_per_km2'].max():.0f} per km2; "
      f"area: mean={rel.loc[high15, 'area_km2'].mean():.0f} km2")
print(f"low-rate 15 -- density: mean={rel.loc[low15, 'population_density_per_km2'].mean():.0f}, "
      f"range {rel.loc[low15, 'population_density_per_km2'].min():.0f}-{rel.loc[low15, 'population_density_per_km2'].max():.0f} per km2; "
      f"area: mean={rel.loc[low15, 'area_km2'].mean():.0f} km2")

# is density just standing in for pool_count, or does each carry independent information?
rho_d_partial, p_d_partial = partial_spearman(rel["population_density_per_km2"], rel["racialised_rate"], rel["pool_count"])
rho_pc_partial, p_pc_partial = partial_spearman(rel["pool_count"], rel["racialised_rate"], rel["population_density_per_km2"])
print(f"\ndensity vs. rate, controlling for pool_count:  rho={rho_d_partial:.3f}, p={p_d_partial:.3f}")
print(f"pool_count vs. rate, controlling for density:  rho={rho_pc_partial:.3f}, p={p_pc_partial:.3f}")
print("(both survive controlling for the other -- density is not just relabelling pool_count)")


In [ ]:
# venue-type composition of the pools themselves (not reviews), by tier -- a structural look at
# what kind of facility dominates each tier's swimming-pool stock
places_tier = corpus[["unique_identifier", "constituency_code", "primary_place_type"]].drop_duplicates(subset="unique_identifier")
places_tier["tier"] = np.where(places_tier["constituency_code"].isin(high15), "High-rate (top 15)",
                       np.where(places_tier["constituency_code"].isin(low15), "Low-rate (bottom 15)", "Other"))
venue_ct = pd.crosstab(places_tier.loc[places_tier["tier"] != "Other", "primary_place_type"],
                        places_tier.loc[places_tier["tier"] != "Other", "tier"], normalize="columns") * 100
print("primary_place_type composition of pools, by tier (col %, top 8 by high-tier share):")
print(venue_ct.sort_values("High-rate (top 15)", ascending=False).head(8).round(1))

# reviewer_rating (the 1-5 star rating attached to the review) by theme -- a severity signal
exploded_rating = df[["theme", "reviewer_rating"]].explode("theme")
print("\nmean reviewer_rating by theme, all 7,768 racialised reviews (lower = more negative overall experience):")
print(exploded_rating.groupby("theme")["reviewer_rating"].mean().sort_values().round(2))


In [ ]:
eth_share_all = pd.crosstab(df["constituency_code"], df["ethnicity_group4"], normalize="index")
eth_share_aligned = eth_share_all.reindex(rel.index)
for grp in ["Germany", "Arab", "Türkiye", "Europe", "Other"]:
    rel[grp] = eth_share_aligned[grp].values

rho_fp, p_fp = stats.spearmanr(rel["population_density_per_km2"], rel["foreign_population_pct"])
print(f"population_density_per_km2 vs. foreign_population_pct: rho={rho_fp:.3f}, p={p_fp:.2e}")

print("\ndensity vs. each ethnicity_group4 share of a constituency's racialised reviews:")
for grp in ["Germany", "Arab", "Türkiye", "Europe", "Other"]:
    rho, p = stats.spearmanr(rel["population_density_per_km2"], rel[grp])
    print(f"{grp:10s}: rho={rho:+.3f} p={p:.3f}")

print("\ndoes ethnicity_group4 share still predict racialised_rate once density is controlled?")
for grp in ["Arab", "Türkiye", "Germany"]:
    rho_p, p_p = partial_spearman(rel[grp], rel["racialised_rate"], rel["population_density_per_km2"])
    print(f"{grp:10s} share vs. rate | controlling for density: rho={rho_p:+.3f} p={p_p:.3f}")


**Yes -- this is decisively an urban/rural story, much bigger than the `pool_count` finding alone suggested.** `population_density_per_km2` correlates with `racialised_rate` at **rho=0.597** (p=3.1e-25) -- one of the strongest relationships found anywhere in this notebook -- and with `pool_count` at **rho=-0.794**. The tier means make this concrete: the high-rate 15 average **3,425 people/km2** across a **260 km2** footprint; the low-rate 15 average **186 people/km2** across **2,355 km2** -- a ~20x density gap and ~9x area gap. This also explains *why* `pool_count` behaves the way it does, mechanically: Germany's Bundestag constituencies are drawn to roughly equal population, so a sparse rural constituency must cover far more land (and therefore accumulate more small local pools) to reach the same population quota that a dense city constituency reaches in a fraction of the area with fewer, larger venues. **The `pool_count` finding earlier in this section was, in substantial part, an urban/rural proxy, not an independent "volume" effect** -- though both survive controlling for each other (density partial rho=0.307, p<0.001; pool_count partial rho=-0.169, p=0.008), so neither fully subsumes the other.

**This also reframes the ethnicity-composition finding from earlier.** Density correlates with `foreign_population_pct` at rho=0.773 (the well-documented fact that immigrant populations concentrate in German cities), and with `Arab`/`Türkiye` share of racialised reviews (rho=+0.196/+0.195) and `Germany` share (rho=-0.202) in the expected directions. Once density is controlled: **`Germany` share's relationship with `racialised_rate` disappears completely** (rho=-0.014, p=0.828, vs. the raw -0.161) -- it was entirely a density proxy. **`Türkiye` share's relationship weakens to non-significance** (rho=+0.082, p=0.198, vs. raw +0.180). **Only `Arab` share survives as an independent effect**, weakened but still real (rho=+0.132, p=0.039, vs. raw +0.211). So of the three ethnicity findings reported earlier in this section, one was fully a density artifact, one was mostly one, and one -- `Arab` presence -- adds something beyond urban/rural alone.

**`Staff`-share itself is not simply explained by density** (rho=-0.113, p=0.076, not significant) -- so the specific Staff-vs-Crowd theme shift is a distinct phenomenon that happens to co-occur with the urban/rural split in these tiers, not a direct product of it. Two supporting, more circumstantial observations: the pool stock itself differs by tier (high-rate constituencies' pools skew toward staffed municipal facilities -- `Public swimming pool` 14.0% vs. 4.9%, `Sports complex` 6.1% vs. 2.5% -- while low-rate constituencies have proportionally more `Swimming lake`/natural sites), and `Staff` and `Access` are, by a wide margin, the two most negatively-rated themes in the entire racialised corpus (mean reviewer rating 1.64 and 1.37 respectively, vs. 2.63 for `Crowd` and 2.68 for `Other`) -- consistent with `Staff`-labelled racialised content marking a more severely negative individual experience, though this doesn't yet explain why that concentrates in lower-rate places specifically.

**In short**: `population_density_per_km2` explains much of what `pool_count` was proxying for, and the earlier `ethnicity_group4`-as-independent-predictor story needs a real caveat: only `Arab` share clearly survives density control, `Germany` and `Türkiye` shares mostly or fully don't. The `Staff`-theme mechanism remains the one genuinely open thread -- density doesn't explain it, so it's a real candidate for the qualitative RQ3 follow-up, not a solved case.

# Part 1 findings: prevalence of racialised discourse across WHEN, WHERE, and WHO



**WHEN.** The racialised rate rises significantly from 2016-2025 (p=1.9e-48), and 2.7x faster from 2021-2025 specifically (p=1.9e-71; 0.92%->2.11%, z-test p=1.7e-49) -- a nationwide rise, not a data artifact: 288/299 constituencies (96.3%) have racialised reviews both before and after 2021. Seasonal effects are significant but small (outdoor 1.92% vs. indoor 1.25%; summer 2.03% vs. other 1.33%; both p<1e-54), and not stable over time: absent in 2016-2017, present in 2018-2019, gone again in 2020-2021 (plausibly COVID-related), then widening every year from 2022 onward. `Clothing` is the one theme with both the steepest post-2021 rise and the strongest season effect of any label tested. Full-sample (N=7,768) dimension prevalence and cross-dimension associations (theme x connotation, Cramér's V=0.122) hold at the corpus level, independent of geography.

**WHERE.** Constituency-grain prevalence estimates cover 247/299 constituencies (83%, `>=10` racialised-review reliability bar), state-grain covers all 16. A pool-level concentration check shows high-rate constituencies are ones where *more* of their pools carry a racialised review (rho=0.65 with rate), not ones dominated by a single outlier venue (rho=0.09, ns) -- ruling out a venue-level artifact for the pattern *across* constituencies. Within individual constituencies, though, the dominant pool itself is often a genuine outlier: 99/247 (40%) have a rate significantly elevated above the rest of their own constituency, and 90/247 (36%) of those carry a minority of the local review volume -- a real venue-specific effect in a substantial minority of cases, not just a big-venue volume artifact. Land-neighbour border distance shows no significant relationship with the rate at any grain (all 299: rho=-0.055, p=0.341; reliable_n only: rho=-0.000, p=0.996) -- different border segments (France/Switzerland high, Denmark average-to-low) pull in opposite directions and cancel out in aggregate.

**WHO.** Refined ethnicity categorization (Germany/Arab/Türkiye/Europe/Other/Unknown, consolidated from a 12-category scheme) with a three-way population comparison (entire corpus / social / racialised) for both ethnicity and language -- language shifts noticeably between entire-corpus and social=yes grains, unlike ethnicity, and both shifts are statistically significant (language: chi2=645, p=3.3e-132; ethnicity: chi2=1573, p<<.001), with small effect sizes typical of this large-N corpus. German-authored racialised reviews skew toward `Crowd`/`Etiquette` themes framed `generalised`; non-German-authored ones skew toward `Staff`/`Access` framed as a specific `incident` -- really an Arab + Türkiye vs. Germany polarity (Cramér's V up to 0.180 for theme), not a uniform non-German effect: `Poland`/`Russian Federation` lean mildly toward the Germany pattern, half the other categories show no strong signature either way. Within non-German reviewers, the *language* written in predicts tone (`connotation`) far better than name-origin does -- a clean gradient from German-language (harshest) to Czech (mildest). Geography checked twice and ruled out: East/West/Berlin doesn't modulate any ethnicity group's dimension profile in a way that survives effect-size scrutiny (3/20 tests significant, all Cramér's V<=0.12), and `Arab`/`Türkiye`-dominant constituencies show no state-level clustering (unlike `Europe`, clustered in Baden-Württemberg, and `Germany`, clustered in the East).

**The WHO x WHERE capstone finding.** Constituency-level `racialised_rate` is driven substantially by **population density/urban-rural status** (rho=+0.597 with rate -- the strongest relationship in this notebook), which also mechanically explains the `pool_count` finding (rho=-0.556): Germany's equal-population constituencies force sparse rural areas to be geographically huge, accumulating more small pools. Once density is controlled, the earlier ethnicity-composition story mostly evaporates -- `Germany` share's link to rate disappears completely, `Türkiye` share weakens to non-significance -- and only `Arab` share survives as an independent predictor (rho=+0.132, p=0.039). The `Staff`-theme concentration in low-rate constituencies is real, holds within every ethnicity group (not compositional), and is not explained by density either -- the one genuinely open mechanism, flagged for qualitative RQ3 follow-up.

# Open threads: what needs socioeconomic and political data to resolve



Everything below is a loose end from Part 1 that Part 2 (`02_socioeconomic_political_relations.ipynb`) is needed to tie off -- gathered here in one place rather than scattered as forward-references throughout.

**Ecological-fallacy / venue-level caveats.** Two checks in WHERE narrow this concern without eliminating it. Pool-concentration shows high-rate constituencies are ones where *more* of their pools carry a racialised review, not areas dominated by one outlier venue (rho=0.65 vs. rho=0.09, ns) -- so a socioeconomic/political correlation at constituency level is not, in general, an artifact of one-venue outliers. But within individual constituencies, 90/247 (36%) have a dominant pool that is itself a genuine, small-share, statistically distinct outlier -- a real limitation on reading any single constituency's rate as a pure area-level signal. These 90 are flagged (`dominant_pool_is_small_high_rate` on `const_agg`) to list explicitly wherever constituency-level rate is used, and as a candidate sensitivity check: exclude or down-weight them and see whether the socioeconomic/political correlations hold.

**Covariates for a predictive/regression model of `racialised_rate`.** `population_density_per_km2` is the strongest single relationship found in this notebook (rho=+0.597) and should lead the covariate list, ahead of `pool_count` (rho=-0.556, largely a density proxy given Germany's equal-population constituency boundaries -- both survive controlling for each other, so neither subsumes the other). `Arab` share of racialised reviews is a secondary candidate: the only `ethnicity_group4` share that survives density control (rho=+0.132, p=0.039), while `Germany` share's link disappears entirely once density is controlled and `Türkiye` share weakens to non-significance. East/West/Berlin geography and land-border proximity were checked directly and are safely excludable from a first-pass specification -- both are null.

**Genuinely open, not yet explained.** The `Staff`-theme concentration in low-rate (rural, low-density) constituencies is real, holds within every ethnicity group, and is explained by neither density nor ethnicity composition -- flagged for qualitative RQ3 follow-up, not resolved here.

**Data-quality caveats that stand regardless of Part 2.**
- The 52 constituencies below the reliability bar (`reliable_n`) should be shown, not hidden, but flagged wherever they appear in a ranking or map.
- The temporal trend rests on 99.94% timestamp coverage after the relative-time recovery step (up from 69.5%); recovery isn't perfectly neutral across states, so treat the trend as directionally reliable, not exact to the percentage point.
- The ethnicity/language `"Other"` category (~140 distinct countries) is too heterogeneous to interpret as one group; several small-language/thin-N cells throughout WHO are suggestive, not precise.
- `Arab` share and `pool_count` are themselves correlated (rho=-0.199) -- the partial-correlation checks here show both relationships exist but don't fully disentangle them; needs the regression-based control Part 2 already applies to `second_vote_afd_pct`/`foreign_population_pct`, extended to `ethnicity_group4` shares and density together.

**Deferred to Part 2's state-level cut.** Dimension-level (`theme`/`approach`/`connotation`/`context`) geographic breakdowns -- 7,768 racialised reviews over 299 constituencies is too thin per-cell for most dimension categories; state level has enough N to be worth trying.

**Spatial pattern to test formally.** The choropleth shows a visually convincing cluster (Ruhr NRW + southwestern Baden-Württemberg); Part 2's Moran's I test should confirm or disconfirm this is genuine spatial autocorrelation, not an eyeballed pattern.

**A note on paper structure.** Several results here are fundamentally *ruling-out-a-confound* findings (pool-level concentration, the land-border null, the East/West null) rather than new substantive claims about racialised discourse itself -- candidates for an appendix or robustness-checks section rather than main-text placement; the final call is Ethel's.

This is where Part 1 hands off: `population_density_per_km2`, `pool_count`, and `Arab` share as the covariate starting point, the 90 dominant-pool-outlier constituencies as a sensitivity check, and the `Staff`-theme mechanism as an open qualitative thread -- everything else needed to test these formally requires the socioeconomic and political data Part 2 brings in.

# Part 2 — Relating racialised discourse to socioeconomic/political data

**Scope for this pass** (as agreed before building this notebook): descriptive and correlational only — Spearman correlations with multiple-comparison correction, non-parametric group comparisons, and proper spatial autocorrelation. No predictive regression model here; that's the natural next notebook once IAA/robustness work has established how much to trust the `racialised` label itself.


## Leads from Part 1 carried into this pass

Part 1's closing "Open threads" section flagged several leads that need socioeconomic/political data to resolve. This pass picks up one directly:

- **Arab ethnicity-share** (§2.4g below): the only `ethnicity_group4` share that survives controlling for `population_density_per_km2` (rho=+0.132 vs. raw +0.211) — `Germany`'s link disappears entirely under that control and `Türkiye`'s weakens to non-significance. That check only ever controlled for density; it has not yet been run against this notebook's own deprivation/political predictors.

**Deliberately not pursued in this pass** (so scope stays bounded):
- The `Staff`-theme rural-concentration puzzle — Part 1 resolved it as a genuine within-group effect, not a compositional one, but *why* places differ remains an open, qualitative lead, not something a socioeconomic correlate is likely to explain.
- `approach` has near-chance inter-annotator agreement (κ≈0-0.09) in Part 1 — carried forward here only as a caution, not re-tested, wherever `approach` would otherwise appear (it doesn't feature in this notebook's own crosstabs, so this is a non-issue in practice, noted for completeness).


## 0. Setup — continuing from Part 1

Reuses `const_agg` and `racialised_df` exactly as built and verified in Part 1 above (place→constituency spatial join, constituency-level aggregate table) -- no reload, no re-derivation. Only the socioeconomic/political CSV is new to this Part; the constituency shapefile loaded in Part 1 is reused too (needed again here only for its polygon geometry, to draw maps and compute spatial autocorrelation -- not to redo the point-in-polygon join itself).

## 2.1 Building the constituency-level analysis table

`const_agg` (built in Part 1) already carries `constituency_code`/`constituency`/`state` in the same form as `federal_baseline_eng_selected_political.csv` (both ultimately keyed off the same `federal_baseline_eng_selected_political.csv` backbone, so there's no naming mismatch left to fix here — that mismatch was between the raw shapefile fields (`WKR_NR`/`LAND_NR`) and this file, and we already resolved it in Section 1). We merge in every other socioeconomic/political column.

In [ ]:
political_extra = soci_political.drop(columns=["state", "constituency"])
analysis_df = const_agg.merge(political_extra, on="constituency_code", how="left")
assert analysis_df["total_population"].notna().all(), "every constituency should have matched socioeconomic data"
print(f"analysis_df: {analysis_df.shape[0]} constituencies x {analysis_df.shape[1]} cols")
analysis_df[["constituency", "state", "racialised_rate", "total_population", "second_vote_afd_pct"]].head()


In [ ]:
# Task 2 pipeline check: clean_federal_baseline_eng.py now selects `population_density_per_km2`
# and `area_km2` through to federal_baseline_eng_selected.csv (both were already computed via
# SHORT_COLUMN_NAMES but not previously included in SELECTED_COLUMNS). const_agg was also rebuilt
# since this notebook last ran, replacing the coastline-inclusive `is_border_constituency` with
# the corrected `is_land_border_constituency` (land-neighbour distance <150m only, coastline
# excluded — the old flag wrongly put coastal-but-inland seats like `Bremen II – Bremerhaven` on
# the same footing as a genuine cross-border constituency like `Offenburg`) plus the underlying
# `distance_to_neighbor_country_km`. Confirm all four landed in analysis_df through the existing
# generic merge above, with no unexpected NaNs.
new_pipeline_cols = ["population_density_per_km2", "area_km2",
                      "is_land_border_constituency", "distance_to_neighbor_country_km"]
print("present in analysis_df:", {c: c in analysis_df.columns for c in new_pipeline_cols})
print("NaN counts:")
print(analysis_df[new_pipeline_cols].isna().sum())

berlin_density = analysis_df.loc[analysis_df["state"] == "Berlin", "population_density_per_km2"]
rural_row = analysis_df.loc[analysis_df["population_density_per_km2"].idxmin()]
print(f"\nBerlin density range: {berlin_density.min():.1f}-{berlin_density.max():.1f} inhabitants/km²")
print(f"Least dense constituency: {rural_row['constituency']} ({rural_row['state']}), "
      f"{rural_row['population_density_per_km2']:.1f} inhabitants/km²")
print(f"\nis_land_border_constituency: {analysis_df['is_land_border_constituency'].sum()} / {len(analysis_df)} True "
      f"(land-neighbour only, from Part 1's corrected geometric check)")


## 2.2 A framing caveat, stated once and applied throughout Part 2

Everything below correlates a **constituency-aggregate** rate (racialised reviews / social reviews, computed across all reviewers who happened to review a pool in that constituency) with **constituency-aggregate** socioeconomic/political indicators (population shares, vote shares, etc). This is an **ecological-level analysis**: a finding like "constituencies with a higher AfD vote share tend to have a higher racialised-review rate" is a claim about *places*, not about *individual reviewers* — it does not mean AfD voters wrote the racialised reviews, or that any individual reviewer's characteristics can be inferred from their constituency's aggregate statistics (the ecological fallacy). This framing matters for how any of Part 2's results should be written up for an ICWSM audience: as area-level associations that motivate further (ideally individual- or review-level) investigation, not as claims about who is behind the discourse.

A sharper version of this same concern is whether a "constituency-level" pattern is really a *one-venue* pattern wearing a constituency's name — Part 1 §1.2b checked this directly: racialised rate correlates strongly with how many of a constituency's *pools* are affected (ρ=0.65), not with how much its single most-implicated pool dominates (ρ=0.09, not significant). So the area-level framing below is reasonably earned, not just a formality — high-rate constituencies tend to be broadly, not narrowly, affected.

## 2.3 Deriving comparable rate/per-capita variables

The socioeconomic file mixes raw totals (`total_population`, `foreign_population`, `disposable_income_total_eur`, ...), already-published rates (`unemployment_total`, `second_vote_*_pct`, `turnout_pct`), and derived counts (`employees_total`, `sgb_ii_recipients_total`, ...). Correlating `racialised_rate` (already a rate) against raw population-scaled totals would mostly just reflect constituency size, not composition — so we convert every relevant total into a per-capita or percentage form ourselves, following the same `x / total_population` logic already documented in the socioeconomic README's derivation notes (not inventing a new formula, just extending the same one to the columns we need for correlation that weren't already expressed as a rate).

In [ ]:
pop = analysis_df["total_population"]

analysis_df["foreign_population_pct"] = analysis_df["foreign_population"] / pop * 100
for age_col in ["age_under_18_population", "age_18_24_population", "age_25_34_population",
                "age_35_59_population", "age_60_74_population", "age_75_plus_population"]:
    analysis_df[age_col.replace("_population", "_pct")] = analysis_df[age_col] / pop * 100

analysis_df["disposable_income_eur_per_capita"] = analysis_df["disposable_income_total_eur"] / pop
analysis_df["gdp_eur_per_capita"] = analysis_df["gdp_total_eur"] / pop
analysis_df["employees_pct_of_pop"] = analysis_df["employees_total"] / pop * 100
analysis_df["sgb_ii_recipients_pct"] = analysis_df["sgb_ii_recipients_total"] / pop * 100

PREDICTOR_COLS = [
    "foreign_population_pct", "age_under_18_pct", "age_18_24_pct", "age_25_34_pct",
    "age_60_74_pct", "age_75_plus_pct", "disposable_income_eur_per_capita", "gdp_eur_per_capita",
    "employees_pct_of_pop", "sgb_ii_recipients_pct", "unemployment_total", "turnout_pct",
    "second_vote_spd_pct", "second_vote_cdu_csu_pct", "second_vote_greens_pct",
    "second_vote_afd_pct", "second_vote_left_pct", "second_vote_bsw_pct",
]
print(f"{len(PREDICTOR_COLS)} predictors prepared for correlation analysis")
analysis_df[PREDICTOR_COLS].describe().T[["mean", "std", "min", "max"]]


## 2.3b Socioeconomic denominator audit and a Berlin/Hamburg/Bremen caveat

Before treating all 18 `PREDICTOR_COLS` as directly comparable in §2.4, two things are worth checking explicitly: whether they really share a common denominator basis, and whether the merge that built `analysis_df` could be silently duplicating rows.

**Denominator basis.** Of the 18 predictors, **10** are population-per-capita variables derived directly above in §2.3 (`x / total_population`, or `x / total_population * 100`): `foreign_population_pct`, the five age-share columns, `disposable_income_eur_per_capita`, `gdp_eur_per_capita`, `employees_pct_of_pop`, and `sgb_ii_recipients_pct`. **7** are election variables denominated by `valid_votes` or `eligible_voters` (`turnout_pct` and the six `second_vote_*_pct` columns) -- not population-based by construction, so not really a methodological trap since nobody would expect otherwise. **`unemployment_total` is the one that matters**: per `README_data_processing.md`'s explicit calibration note ("Unemployment rates are kept as rates... the proper denominator is the labor force, not total population"), it is a published labor-force-denominated rate, not something derived from `total_population` the way the 10 population-share predictors are. It sits directly alongside `sgb_ii_recipients_pct`/`foreign_population_pct` in §2.4's ranking (ρ=0.43, the third-strongest predictor) -- a reader could reasonably assume all three are on the same population basis. They are not, and that should be flagged whenever `unemployment_total` is cited alongside the population-share predictors.

**Merge integrity.** `constituency_code` -- the actual join key -- is unique in both `political` and `const_agg` before the merge, and `analysis_df` has exactly as many rows after it as before: a clean 1:1 join, no row duplication.

In [ ]:
# denominator audit: split PREDICTOR_COLS into population-per-capita vs. election-denominated
# vs. the one labor-force-denominated exception
population_denominated = [
    "foreign_population_pct", "age_under_18_pct", "age_18_24_pct", "age_25_34_pct",
    "age_60_74_pct", "age_75_plus_pct", "disposable_income_eur_per_capita",
    "gdp_eur_per_capita", "employees_pct_of_pop", "sgb_ii_recipients_pct",
]
election_denominated = [
    "turnout_pct", "second_vote_spd_pct", "second_vote_cdu_csu_pct", "second_vote_greens_pct",
    "second_vote_afd_pct", "second_vote_left_pct", "second_vote_bsw_pct",
]
labor_force_denominated = ["unemployment_total"]
assert set(population_denominated + election_denominated + labor_force_denominated) == set(PREDICTOR_COLS)
print(f"{len(population_denominated)} population-per-capita predictors")
print(f"{len(election_denominated)} election-denominated predictors (valid_votes / eligible_voters)")
print(f"{len(labor_force_denominated)} labor-force-denominated predictor: {labor_force_denominated}")

# merge integrity: is constituency_code -- the actual join key -- unique in both source tables,
# and did the merge inflate the row count?
print(f"\npolitical: {political['constituency_code'].nunique()} unique constituency_code / {len(political)} rows")
print(f"const_agg: {const_agg['constituency_code'].nunique()} unique constituency_code / {len(const_agg)} rows")
print(f"analysis_df: {len(analysis_df)} rows after merge (no inflation)")

n_dup_pop_values = analysis_df["total_population"].duplicated().sum()
print(f"\ntotal_population values that happen to match another constituency's: {n_dup_pop_values} "
      f"(coincidence, not a duplication bug -- see takeaway below)")

In [ ]:
# which columns are actually just a shared state-wide rate in Berlin/Hamburg/Bremen, and do any
# of the predictors we actually use in §2.4 inherit that?
repeated_report = pd.read_csv(SOCIO_DIR / "state_repeated_columns_report.csv")
repeated_report["repeated_states_list"] = repeated_report["repeated_states"].fillna("").apply(
    lambda s: [x.strip() for x in s.split(";") if x.strip()]
)
bhb_repeated = repeated_report[repeated_report["repeated_states_list"].apply(
    lambda states: any(s in states for s in ["Berlin", "Hamburg", "Bremen"])
)]
print(f"{len(bhb_repeated)} columns flagged as repeated within Berlin, Hamburg, and/or Bremen:")
print(bhb_repeated[["column", "repeated_states"]].to_string(index=False))

# of those, which feed directly into this notebook's §2.3 predictor derivation?
predictor_ingredient_cols = [
    "age_under_18_pct", "age_18_24_pct", "age_25_34_pct", "age_35_59_pct", "age_60_74_pct", "age_75_plus_pct",
    "disposable_income_eur_per_inhabitant", "gdp_eur_per_inhabitant",
    "sgb_ii_recipients_per_1k", "employees_per_1k", "unemployment_total",
]
predictor_relevant_repeated = bhb_repeated[bhb_repeated["column"].isin(predictor_ingredient_cols)]
print(f"\nof those, {len(predictor_relevant_repeated)} feed directly into a §2.3 predictor:")
print(predictor_relevant_repeated[["column", "repeated_states"]].to_string(index=False))

# spot-check: are these ACTUALLY identical within a state once re-derived as %, or just close?
berlin_check = analysis_df.loc[analysis_df["state"] == "Berlin",
    ["constituency", "sgb_ii_recipients_pct", "disposable_income_eur_per_capita", "gdp_eur_per_capita"]]
print(f"\nBerlin's {len(berlin_check)} constituencies -- unique values of sgb_ii_recipients_pct / "
      f"disposable_income_eur_per_capita / gdp_eur_per_capita: "
      f"{berlin_check['sgb_ii_recipients_pct'].nunique()}, {berlin_check['disposable_income_eur_per_capita'].nunique()}, "
      f"{berlin_check['gdp_eur_per_capita'].nunique()}")

bhb_mask = analysis_df["state"].isin(["Berlin", "Hamburg", "Bremen"])
print(f"\nconstituencies in Berlin/Hamburg/Bremen: {bhb_mask.sum()} / {len(analysis_df)}")
print(analysis_df.loc[bhb_mask, "state"].value_counts())

**Takeaway.** Two real findings and one non-issue. (1) **`unemployment_total` is a genuine denominator mismatch** -- a labor-force rate sitting next to 10 population-share predictors it's implicitly compared against in §2.4's ranking; that comparison should be read with this in mind. (2) **The merge is clean** -- `constituency_code` is unique in every source table and `analysis_df`'s row count doesn't inflate after the join; the 27 `total_population` value collisions are a rounding coincidence between similarly-sized constituencies (Bundestag districts are deliberately population-balanced), not a data-quality problem. (3) **The Berlin/Hamburg caveat is real, and sharper than a 'similar values' concern for two of the three spot-checked columns**: 11 predictor-ingredient columns are flagged as state-wide constants in Berlin and/or Hamburg (the six age-share columns, `disposable_income_eur_per_inhabitant`, `gdp_eur_per_inhabitant`, `sgb_ii_recipients_per_1k`, `employees_per_1k`, and `unemployment_total`). Because `total_population` cancels out algebraically in the per-capita derivation, `disposable_income_eur_per_capita` and `gdp_eur_per_capita` come back out **bit-for-bit identical** across Berlin's 12 constituencies (1 unique value each, confirmed above) -- these two are pure linear reconstructions of a single city-wide EUR-per-inhabitant figure, with no independent constituency-level information at all. `sgb_ii_recipients_pct` is *not* bit-identical (12 distinct values, confirmed above) but is practically constant, varying only in the 4th decimal place (~12.8199-12.8202 across all 12 Berlin constituencies) -- the intermediate step (`sgb_ii_recipients_total`, rounded to a whole number per `README_data_processing.md`) breaks exact algebraic cancellation, but the underlying value is still overwhelmingly the shared Berlin-wide rate re-scaled by local population, not an independently observed constituency-level measurement. Bremen's 2 constituencies, by contrast, don't share any of the columns that actually feed §2.4's predictors (only `craft_enterprises_per_1k`, `employees_agriculture_pct`, and `municipalities` are Bremen-repeated, none of which are used here) -- so this is really an 18-constituency (Berlin+Hamburg) issue for the predictors in play, not a full Berlin/Hamburg/Bremen one, even though the exclusion check below drops all 20 city-state constituencies together for a clean, consistent comparison. This means a chunk of `analysis_df`'s deprivation/demographic predictors aren't independently observed at the constituency level for those 18 places -- they're one shared city rate, re-scaled by local population. §2.4d below re-runs §2.4/§2.4b excluding all 20 Berlin/Hamburg/Bremen constituencies to check whether this actually changes anything (it needs `run_correlations`/`partial_spearman`, defined below in §2.4/§2.4b, so the robustness re-computation itself is deferred to right after §2.4c rather than run here).

## 0.2 Is the derivation actually making a difference?

§2.3 converts every population-denominated `PREDICTOR_COLS` variable from a raw total into a per-capita/percentage form. But most of those raw totals are themselves *already* a derivation: `clean_federal_baseline_eng.py` took an originally-published rate or per-1,000 figure (e.g. `age_under_18_pct`, `foreign_population_pct`, `disposable_income_eur_per_inhabitant`, `employees_per_1k`, `sgb_ii_recipients_per_1k`), multiplied it back out into a rounded whole-number total, and only kept that total in `federal_baseline_eng_selected_political.csv` (the file this notebook actually loads) -- so §2.3's per-capita/percentage predictors are a **round trip**: published rate → derived total (rounded) → re-derived rate. This checks whether that round trip actually preserves the originally-published rate, or introduces meaningful drift, and then folds in §2.3b's Berlin/Hamburg finding as the sharpest case where the derivation doesn't add real constituency-level information at all.

In [ ]:
# round-trip fidelity: published rate -> derived total (rounded, per clean_federal_baseline_eng.py)
# -> re-derived rate (as this notebook's own §2.3 does). Uses federal_baseline_eng_clean.csv
# (loaded above in §0.1 as clean_const), which -- unlike the selected file §2.1-§2.3 actually
# load -- still carries the originally-published rate columns alongside the derived totals, so both
# ends of the round trip are available to compare directly.
pop_full = clean_const["total_population"].astype(float)

ROUND_TRIP_CHECKS = [
    ("age_under_18_pct", "age_under_18_population", lambda cnt: cnt / pop_full * 100),
    ("age_18_24_pct", "age_18_24_population", lambda cnt: cnt / pop_full * 100),
    ("age_25_34_pct", "age_25_34_population", lambda cnt: cnt / pop_full * 100),
    ("age_60_74_pct", "age_60_74_population", lambda cnt: cnt / pop_full * 100),
    ("age_75_plus_pct", "age_75_plus_population", lambda cnt: cnt / pop_full * 100),
    ("foreign_population_pct", "foreign_population", lambda cnt: cnt / pop_full * 100),
    ("disposable_income_eur_per_inhabitant", "disposable_income_total_eur", lambda cnt: cnt / pop_full),
    ("gdp_eur_per_inhabitant", "gdp_total_eur", lambda cnt: cnt / pop_full),
    ("employees_per_1k", "employees_total", lambda cnt: cnt / pop_full * 1000),
    ("sgb_ii_recipients_per_1k", "sgb_ii_recipients_total", lambda cnt: cnt / pop_full * 1000),
]

roundtrip_rows = []
for orig_col, count_col, reconstruct in ROUND_TRIP_CHECKS:
    orig = clean_const[orig_col].astype(float)
    reconstructed = reconstruct(clean_const[count_col].astype(float))
    diff = (orig - reconstructed).abs()
    worst = diff.idxmax()
    roundtrip_rows.append({
        "published_rate": orig_col, "via_derived_total": count_col,
        "max_abs_drift": diff.max(), "n_over_0.05": int((diff > 0.05).sum()),
        "worst_constituency": clean_const.loc[worst, "constituency"],
    })
roundtrip_df = pd.DataFrame(roundtrip_rows).sort_values("max_abs_drift", ascending=False)
print("Round-trip drift (published rate vs. rate reconstructed from the rounded derived total):")
roundtrip_df


In [ ]:
# fold in §2.3b's finding: for the predictor-ingredient columns already flagged as state-wide
# constant in Berlin and/or Hamburg, does the round trip above still "pass" even though there's no
# real constituency-level variation being preserved in the first place?
print(f"§2.3b flagged {len(predictor_relevant_repeated)} predictor-ingredient columns as "
      f"state-wide constant in Berlin/Hamburg: {predictor_relevant_repeated['column'].tolist()}")
print(f"\nBerlin's 12 constituencies -- unique reconstructed values for the same three columns "
      f"checked in §2.3b: sgb_ii_recipients_pct={berlin_check['sgb_ii_recipients_pct'].nunique()}, "
      f"disposable_income_eur_per_capita={berlin_check['disposable_income_eur_per_capita'].nunique()}, "
      f"gdp_eur_per_capita={berlin_check['gdp_eur_per_capita'].nunique()} (of 12 possible)")
print("\nThe round-trip check above and this state-constancy check catch two different failure "
      "modes: round-trip fidelity asks whether converting a rate to a total and back distorts the "
      "*value*; state-constancy asks whether there was ever independent constituency-level "
      "information to distort in the first place. A perfectly state-wide-constant rate round-trips "
      "almost exactly (there's nothing for rounding to disturb), which is exactly what both checks "
      "show for Berlin/Hamburg -- passing the value-fidelity check above does not contradict "
      "§2.3b's finding.")


**Takeaway -- Part 0 bottom line.** No new problems, and one confirmed-benign exception. Round-trip fidelity is essentially exact for 9 of 10 checked predictors (max drift on the order of 0.01 percentage points / EUR / per-1,000 units -- rounding noise from `clean_federal_baseline_eng.py`'s whole-number rounding, not a distortion worth worrying about). The one exception, `foreign_population_pct`, drifts by up to ~0.075 percentage points (roughly 30 of 299 constituencies exceed a 0.05pp threshold) -- but this is not a corruption or a rounding artifact of *this* round trip: `foreign_population` is deliberately *not* derived from `foreign_population_pct` (the pipeline instead subtracts two independently-rounded actual counts, per `README_data_processing.md`, specifically because the published percentage is only accurate to 1 decimal place), so this is the data's own reporting precision showing through, not a bug. Folding in §2.3b: the round-trip check and the state-constancy finding are not in tension -- a shared state-wide rate re-scaled by local population round-trips almost perfectly (nothing for rounding to disturb) while still carrying zero independent constituency-level information for the 18 Berlin/Hamburg constituencies already flagged there. **Bottom line: this derivation adds legitimate constituency-level information everywhere except those 18 (of 299) constituencies, where it repackages a shared city rate rather than measuring anything new -- exactly what §2.3b already found, now confirmed from the value-fidelity side rather than just the state-constancy side.** §2.4d later re-runs the correlations excluding all 20 Berlin/Hamburg/Bremen constituencies to check whether this actually changes the substantive results.

# Correlations, relationships, and spatial analysis

Part 0 above established that the raw data survived cleaning intact and that this notebook's own rate/per-capita derivations add real information except for the already-flagged Berlin/Hamburg constituencies. Everything from here on (§2.4 onward) builds on that: Spearman correlations against the socioeconomic/political predictors, confound checks, group comparisons, spatial autocorrelation, and political cross-tabs.

## 2.4 Correlations: racialised rate vs. socioeconomic/political predictors

**Method choices:**
- **Spearman**, not Pearson: several predictors (e.g. `gdp_eur_per_capita`, `sgb_ii_recipients_pct`) are right-skewed, and `racialised_rate` itself is a small proportion bounded at 0 — Spearman only assumes a monotonic relationship, which is more defensible here than assuming linearity.
- **Benjamini-Hochberg FDR correction** across the 18 simultaneous predictor tests, rather than treating each p-value in isolation — with this many tests, some would clear p<.05 by chance alone.
- **Bootstrap CI on rho** (constituency-level resampling, 2,000 draws) reported alongside the point estimate, since a correlation computed over 299 units deserves an uncertainty range, not just a point estimate + p-value.
- **Robustness check**: every correlation is recomputed restricted to the 247 constituencies that clear Part 1's reliability bar (`racialised_reviews >= 10`, tightened from the earlier `>= 5` bar), so we can see whether any result is being driven by the 52 thin constituencies.

In [ ]:
from statsmodels.stats.multitest import multipletests

rng = np.random.default_rng(seed=42)

def spearman_with_bootstrap_ci(x, y, n_boot=2000):
    rho, p = stats.spearmanr(x, y)
    n = len(x)
    boot_rhos = np.empty(n_boot)
    x_arr, y_arr = np.asarray(x), np.asarray(y)
    for i in range(n_boot):
        idx = rng.integers(0, n, n)
        boot_rhos[i] = stats.spearmanr(x_arr[idx], y_arr[idx])[0]
    ci_lo, ci_hi = np.nanpercentile(boot_rhos, [2.5, 97.5])
    return rho, p, ci_lo, ci_hi

def run_correlations(df, target_col="racialised_rate"):
    rows = []
    for col in PREDICTOR_COLS:
        rho, p, ci_lo, ci_hi = spearman_with_bootstrap_ci(df[col], df[target_col])
        rows.append({"predictor": col, "rho": rho, "p": p, "ci_lo": ci_lo, "ci_hi": ci_hi})
    out = pd.DataFrame(rows)
    out["p_fdr"] = multipletests(out["p"], method="fdr_bh")[1]
    out["significant_fdr"] = out["p_fdr"] < 0.05
    return out.sort_values("rho", key=np.abs, ascending=False).reset_index(drop=True)

corr_all = run_correlations(analysis_df)
corr_reliable = run_correlations(analysis_df[analysis_df["reliable_n"]])

print("All 299 constituencies:")
print(corr_all.to_string(index=False))


In [ ]:
# robustness: compare rho with vs without the constituencies below §1.2's racialised/social-review
# reliability bar (not a fixed count — depends on Part 1's reliable_n column)
n_reliable_constituencies = analysis_df["reliable_n"].sum()
robustness = corr_all[["predictor", "rho"]].merge(
    corr_reliable[["predictor", "rho"]], on="predictor", suffixes=("_all", "_reliable_only")
)
robustness["rho_shift"] = robustness["rho_reliable_only"] - robustness["rho_all"]
print(f"Robustness check (rho using all 299 vs. the {n_reliable_constituencies} 'reliable' constituencies):")
print(robustness.sort_values("rho_shift", key=np.abs, ascending=False).to_string(index=False))

# forest plot, sorted by |rho|, using the full-sample (all 299) results
plot_df = corr_all.sort_values("rho")
fig, ax = plt.subplots(figsize=(8, 7))
colors = np.where(plot_df["significant_fdr"], "#C44E52", "#8C8C8C")
ax.errorbar(
    plot_df["rho"], range(len(plot_df)),
    xerr=[plot_df["rho"] - plot_df["ci_lo"], plot_df["ci_hi"] - plot_df["rho"]],
    fmt="o", color="black", ecolor=list(colors), markersize=4,
)
ax.axvline(0, color="black", linewidth=0.8)
ax.set_yticks(range(len(plot_df)))
ax.set_yticklabels(plot_df["predictor"])
ax.set_xlabel("Spearman rho vs. racialised_rate (bootstrap 95% CI)")
ax.set_title("Correlations with racialised rate\n(red = significant after Benjamini-Hochberg FDR correction)")
plt.tight_layout()
plt.show()


In [ ]:
# the AfD-vote-share result runs counter to a naive prior (higher AfD share -> higher racialised rate);
# before over-interpreting, check whether predictors are themselves intercorrelated in a way that could
# explain this (univariate correlations can't separate confounded predictors from each other)
key_predictor_corr = analysis_df[["second_vote_afd_pct", "foreign_population_pct", "sgb_ii_recipients_pct", "unemployment_total"]].corr(method="spearman")
print("Spearman correlations among the predictors themselves:")
print(key_predictor_corr.round(3))


**Takeaway.** This is the most substantial result in the notebook: **17 of 18 predictors are significant after FDR correction** (only `age_under_18_pct` isn't), and the robustness check confirms none of it is being driven by the 52 constituencies below §1.2's racialised/social-review reliability bar (Part 1's bar was tightened from `racialised_reviews >= 5` to `>= 10` in the most recent refresh, so this is a larger held-out set than in an earlier pass of this notebook) -- rho shifts are larger than before (up to 0.091 for `second_vote_left_pct`) but every predictor keeps the same sign and none changes FDR-significance status. The strongest, most defensible associations are **deprivation and composition variables**: `sgb_ii_recipients_pct` (ρ=0.47), `foreign_population_pct` (ρ=0.46), and `unemployment_total` (ρ=0.43) — constituencies with more social-benefit dependency, more residents with a foreign-population share, and higher unemployment show a higher racialised-review rate. Age structure follows a coherent pattern too: younger-skewing constituencies (`age_25_34_pct` +0.33, `age_18_24_pct` +0.24) show higher rates, older ones (`age_60_74_pct` −0.34, `age_75_plus_pct` −0.22) lower.

The political pattern is the one that most needs careful handling: `second_vote_left_pct` (+0.34), `second_vote_spd_pct` (+0.28), and `second_vote_greens_pct` (+0.24) are positive, while `second_vote_cdu_csu_pct` (−0.27) and, notably, **`second_vote_afd_pct` (−0.20) are negative** — the opposite direction of the naive prior that higher AfD support would go with more racialised discourse. Checking the predictors against each other shows why: `second_vote_afd_pct` is strongly negatively correlated with `foreign_population_pct` itself (ρ=−0.54) — AfD support in this data is concentrated in lower-foreign-population constituencies, and `sgb_ii_recipients_pct`/`unemployment_total` are themselves almost collinear (ρ=0.96, unsurprising — they're two views of the same underlying deprivation). **A univariate correlation matrix cannot separate these confounded predictors from each other or say which one is doing the explanatory work** — this is exactly the ecological-fallacy and confounding risk flagged in §2.2, and exactly why Part 2 stops at correlation rather than attempting a multivariate model in this pass: any single-predictor story (e.g. "AfD support is associated with less racialised discourse") would be misleading without controlling for the others, which needs the multivariate model in Part 3 below, not a univariate correlation matrix.

## 2.4b Is `racialised_rate` itself confounded by population size or venue count?

Every predictor above was normalized to a per-capita/percentage form (§2.3) specifically to avoid a constituency-size artifact on the *predictor* side. But we haven't yet checked the *outcome* side: `racialised_rate` is racialised/social reviews, which controls for review volume by construction, but it could still be mechanically related to how many pools a constituency has (`pool_count`) or how many people live there (`total_population`) — e.g. if bigger, denser constituencies happen to generate systematically different rates for reasons that have nothing to do with deprivation or demographics. If that's true, the §2.4 correlations could be partly an artifact of constituency size rather than composition.

We test this directly (Spearman, same method as §2.4), and then — since the honest answer turns out to require it — recompute the strongest §2.4 predictors as **partial correlations controlling for whichever size variable turns out to matter**, using rank-residualization (regress the ranks of each variable on the ranks of the control variable via OLS, then Spearman-correlate the residuals — the standard nonparametric equivalent of a partial correlation, since `pingouin` isn't installed in this environment).

In [ ]:
SIZE_VARS = ["pool_count", "total_text_reviews", "total_population"]

rows = []
for col in SIZE_VARS:
    rho, p, ci_lo, ci_hi = spearman_with_bootstrap_ci(analysis_df[col], analysis_df["racialised_rate"])
    rows.append({"size_variable": col, "rho": rho, "p": p, "ci_lo": ci_lo, "ci_hi": ci_hi})
size_check = pd.DataFrame(rows)
size_check["p_fdr"] = multipletests(size_check["p"], method="fdr_bh")[1]
print("racialised_rate vs. constituency-size variables:")
print(size_check.round(4).to_string(index=False))


In [ ]:
import statsmodels.api as sm
from scipy.stats import rankdata

def partial_spearman(x, y, control):
    x, y, control = np.asarray(x, dtype=float), np.asarray(y, dtype=float), np.asarray(control, dtype=float)
    rx, ry, rz = rankdata(x), rankdata(y), rankdata(control)
    Z = sm.add_constant(rz)
    rx_resid = sm.OLS(rx, Z).fit().resid
    ry_resid = sm.OLS(ry, Z).fit().resid
    return stats.pearsonr(rx_resid, ry_resid)

# how confounded are the §2.4 predictors themselves with pool_count (venue density)?
predictor_vs_poolcount = pd.Series(
    {col: stats.spearmanr(analysis_df[col], analysis_df["pool_count"])[0] for col in PREDICTOR_COLS},
    name="rho_vs_pool_count",
).sort_values(key=np.abs, ascending=False)
print("How correlated is each §2.4 predictor with pool_count (venue density)?")
print(predictor_vs_poolcount.round(3).to_string())

# partial correlations, controlling for pool_count, for the predictors that were significant in §2.4
partial_rows = []
for col in PREDICTOR_COLS:
    r_partial, p_partial = partial_spearman(analysis_df[col], analysis_df["racialised_rate"], analysis_df["pool_count"])
    original_rho = corr_all.loc[corr_all["predictor"] == col, "rho"].iloc[0]
    partial_rows.append({"predictor": col, "rho_original": original_rho, "rho_partial_ctrl_poolcount": r_partial,
                          "p_partial": p_partial, "shift": r_partial - original_rho})
partial_df = pd.DataFrame(partial_rows).sort_values("rho_original", key=np.abs, ascending=False)
partial_df["p_partial_fdr"] = multipletests(partial_df["p_partial"], method="fdr_bh")[1]
partial_df["still_significant_fdr"] = partial_df["p_partial_fdr"] < 0.05
print("\nOriginal vs. pool_count-controlled partial Spearman correlations:")
print(partial_df.round(3).to_string(index=False))


**Takeaway — this is an important correction to §2.4, not a footnote.** `racialised_rate` is substantially confounded by `pool_count` (ρ=−0.49, p<0.0001) — constituencies with more pools have systematically *lower* rates — and more weakly by `total_population` (ρ=0.27) and `total_text_reviews` (ρ=−0.13). Worse, **almost every §2.4 predictor is itself strongly correlated with `pool_count`** (|ρ| = 0.32-0.57 for most of them) — deprivation, age structure, and vote share are all entangled with venue density in this data, likely because pool_count is a proxy for urban/rural character (dense cities have fewer, larger, heavily-reviewed pools; spread-out rural areas have more, smaller ones).

Controlling for `pool_count` via partial Spearman correlation, **only 6 of 18 predictors remain significant after FDR correction**: `sgb_ii_recipients_pct` (ρ 0.47→0.26), `foreign_population_pct` (0.46→0.28), `unemployment_total` (0.43→0.23), `turnout_pct` (−0.27→−0.26, barely moved), `second_vote_spd_pct` (0.28→0.14), and `second_vote_bsw_pct` (0.19→0.15). Everything else loses significance once venue density is accounted for — including `second_vote_left_pct`, the age-structure variables, `second_vote_greens_pct`, `gdp_eur_per_capita`, and `employees_pct_of_pop`.

**`second_vote_afd_pct` is the clearest casualty**: its partial correlation collapses from ρ=−0.20 to ρ=+0.03 (p=0.58, nowhere near significant) once `pool_count` is controlled — even more decisively confounded than the §2.4 finding against `foreign_population_pct` alone suggested. This *strengthens* rather than changes that section's conclusion: the AfD association was never independent, and controlling for venue density kills it more thoroughly than controlling for foreign population share did.

**Net effect**: the deprivation cluster (`sgb_ii_recipients_pct`, `foreign_population_pct`, `unemployment_total`) and, more surprisingly, `turnout_pct` are the only predictors that look genuinely robust to this specific confound — and even they are meaningfully attenuated, not just "confirmed." This is exactly the kind of result that argues for a proper multivariate model (with `pool_count` as a covariate, not just a nuisance to partial out one at a time) rather than reading any of §2.4's univariate correlations at face value — reinforcing, with much more evidence now, why Part 3's multivariate model below controls for `pool_count` directly rather than this section's one-at-a-time partial correlations.

## 2.4c Robustness check: are the correlations driven by venue-concentration outliers?

§1.2b-§1.2c (notebook 01) found that most constituencies' racialised rate is broadly spread across their pools, and their "dominant" pool is usually just a big, popular venue — but flagged **19 constituencies where the dominant pool is a small venue with a genuinely extreme rate** (`Lübeck` and `Mansfeld` most sharply, both at 23.8%). Those 19 are exactly the candidates for a real hole in the analysis: if the §2.4/§2.4b correlations depend on them, the "deprivation cluster" story could partly be an artifact of a handful of small, outlier-driven venues rather than a genuine area-level pattern.

We rerun the §2.4 correlations excluding those 19 constituencies (analogous to the existing low-`N` robustness check, just against a different, substantively motivated exclusion set) and compare rho shifts directly.

In [ ]:
n_flagged = analysis_df["dominant_pool_is_small_high_rate"].sum()
print(f"constituencies flagged as venue-concentration outliers: {n_flagged} / {len(analysis_df)}")

excl_df = analysis_df[~analysis_df["dominant_pool_is_small_high_rate"]]

# univariate correlations (§2.4), with vs. without the 19 flagged constituencies
corr_excl_concentration = run_correlations(excl_df)
univariate_check = corr_all[["predictor", "rho"]].merge(
    corr_excl_concentration[["predictor", "rho"]], on="predictor", suffixes=("_all", "_excl_concentrated")
)
univariate_check["shift"] = univariate_check["rho_excl_concentrated"] - univariate_check["rho_all"]
print("\nUnivariate rho: all 299 vs. excluding the 19 venue-concentration outliers:")
print(univariate_check.sort_values("shift", key=np.abs, ascending=False).to_string(index=False))

# partial correlations (§2.4b, controlling for pool_count), same comparison, for the 6 predictors
# that survived that check
survivors = partial_df.loc[partial_df["still_significant_fdr"], "predictor"].tolist()
partial_rows = []
for col in survivors:
    r_partial, p_partial = partial_spearman(excl_df[col], excl_df["racialised_rate"], excl_df["pool_count"])
    original = partial_df.loc[partial_df["predictor"] == col, "rho_partial_ctrl_poolcount"].iloc[0]
    partial_rows.append({"predictor": col, "partial_rho_all": original, "partial_rho_excl_concentrated": r_partial,
                          "shift": r_partial - original, "p_excl_concentrated": p_partial})
print("\nSame check on the 6 pool_count-controlled §2.4b survivors:")
print(pd.DataFrame(partial_rows).round(3).to_string(index=False))


**Takeaway.** This hole doesn't hold up — the correlations are not being driven by the venue-concentration outliers. Excluding the 19 flagged constituencies shifts every univariate rho by at most 0.019 (`age_18_24_pct`), and the 6 predictors that survived §2.4b's pool-count control shift by at most 0.022 (`sgb_ii_recipients_pct`) while staying significant at essentially the same p-values. Nothing changes sign, nothing crosses a significance threshold either direction.

Put together with §1.2b-§1.2c: constituency-level racialised rate is (a) mostly driven by broad-based patterns across many pools rather than single venues, and (b) even in the minority of constituencies where one small venue really is an outlier, removing those specific constituencies doesn't move the Part 2 findings. That's a real answer to "could this all just be a few outlier venues" — no, not by this check. It doesn't make the ecological-fallacy caveat in §2.2 go away (these are still area-level associations, not claims about individuals), but it does rule out the specific worry that they're an artifact of a handful of unusual pools.

## 2.4d Robustness check: Berlin/Hamburg/Bremen city-state constituencies

§2.3b flagged that several §2.3 predictors are state-wide constants (not independently observed constituency-level measurements) for 18 of Berlin's and Hamburg's constituencies, while Bremen's 2 don't share any of the columns that actually feed §2.4's predictors. We nonetheless drop all 20 Berlin/Hamburg/Bremen constituencies together here, matching the city-state grouping used throughout this notebook and in the audit report, and re-run §2.4's raw correlations and §2.4b's `pool_count`-controlled partial correlations excluding them -- using the same rho-shift comparison as §2.4c's venue-concentration check, against the same ~0.02 'no meaningful change' benchmark that check established.

In [ ]:
bhb_mask = analysis_df["state"].isin(["Berlin", "Hamburg", "Bremen"])
print(f"excluding {bhb_mask.sum()} Berlin/Hamburg/Bremen constituencies "
      f"({analysis_df.loc[bhb_mask, 'state'].value_counts().to_dict()})")

excl_bhb_df = analysis_df[~bhb_mask]

corr_excl_bhb = run_correlations(excl_bhb_df)
bhb_check = corr_all[["predictor", "rho", "significant_fdr"]].merge(
    corr_excl_bhb[["predictor", "rho", "significant_fdr"]], on="predictor", suffixes=("_all", "_excl_bhb")
)
bhb_check["shift"] = bhb_check["rho_excl_bhb"] - bhb_check["rho_all"]
bhb_check["sig_flip"] = bhb_check["significant_fdr_all"] != bhb_check["significant_fdr_excl_bhb"]
print(f"\nUnivariate rho: all 299 vs. excluding the {bhb_mask.sum()} Berlin/Hamburg/Bremen constituencies:")
print(bhb_check.sort_values("shift", key=np.abs, ascending=False).round(4).to_string(index=False))
print(f"\nmax |shift| = {bhb_check['shift'].abs().max():.4f}, any sign flips: "
      f"{((corr_all.set_index('predictor')['rho'] * corr_excl_bhb.set_index('predictor')['rho']) < 0).any()}, "
      f"significance flips: {bhb_check.loc[bhb_check['sig_flip'], 'predictor'].tolist()}")

In [ ]:
# same check on the 6 pool_count-controlled §2.4b survivors
partial_bhb_rows = []
for col in survivors:
    r_partial, p_partial = partial_spearman(excl_bhb_df[col], excl_bhb_df["racialised_rate"], excl_bhb_df["pool_count"])
    original = partial_df.loc[partial_df["predictor"] == col, "rho_partial_ctrl_poolcount"].iloc[0]
    partial_bhb_rows.append({"predictor": col, "partial_rho_all": original, "partial_rho_excl_bhb": r_partial,
                             "shift": r_partial - original, "p_excl_bhb": p_partial})
partial_bhb_df = pd.DataFrame(partial_bhb_rows)
print("pool_count-controlled partial rho, all 299 vs. excluding Berlin/Hamburg/Bremen:")
print(partial_bhb_df.round(4).to_string(index=False))
print(f"\nmax |shift| = {partial_bhb_df['shift'].abs().max():.4f}")

**Takeaway.** Unlike the venue-concentration check in §2.4c, this one does move the needle somewhat, though not enough to change the notebook's substantive conclusions. Excluding the 20 Berlin/Hamburg/Bremen constituencies shifts univariate rho by more than the ~0.02 'no meaningful change' benchmark for **most** predictors (up to ~0.05 for `employees_pct_of_pop`; `second_vote_afd_pct` and `unemployment_total` both shift by ~0.04-0.05 as well) -- the 20 city-state constituencies pull their correlated predictors in the same direction those predictors already pointed, since Berlin/Hamburg/Bremen run high on both `racialised_rate` (mean ~2.5% vs. ~1.5% elsewhere) and several of the deprivation/composition predictors, so dropping them shrinks (not reverses) most associations. **Nothing flips sign.** **One predictor does cross the FDR-significance line**: `disposable_income_eur_per_capita` stops clearing FDR-significance once Berlin/Hamburg/Bremen are excluded -- notable given it's exactly one of the columns flagged in §2.3b as a Berlin/Hamburg state-wide constant. The `pool_count`-controlled partial correlations for the 6 §2.4b survivors are considerably more stable (shifts well below the 0.02 benchmark, no significance changes) -- once venue density is already controlled for, excluding the city-states barely moves anything further. **Net read**: the deprivation-cluster story (`sgb_ii_recipients_pct`, `foreign_population_pct`, `unemployment_total`) survives excluding Berlin/Hamburg/Bremen and stays FDR-significant both raw and pool_count-partialled, but the raw §2.4 rankings are somewhat inflated by these 20 constituencies (exactly what §2.3b's 'shared city-wide rate' caveat predicts), and one weaker predictor (`disposable_income_eur_per_capita`) doesn't survive without them.

In [ ]:
# do 1 and 2 together: excluding Berlin/Hamburg/Bremen (above) is one way to handle the 10
# state-constant predictors flagged in §2.3b; the other is to just not use them. Report both
# side by side rather than picking one -- does the deprivation-cluster story still hold using
# *only* the predictors with zero state-constancy contamination?
CONTAMINATED_PREDICTORS = [c for c in population_denominated if c != "foreign_population_pct"] + labor_force_denominated
clean_predictors = [c for c in PREDICTOR_COLS if c not in CONTAMINATED_PREDICTORS]
print(f"{len(CONTAMINATED_PREDICTORS)} contaminated predictors (state-wide constant in Berlin/Hamburg): "
      f"{CONTAMINATED_PREDICTORS}")
print(f"{len(clean_predictors)} clean predictors (never flagged): {clean_predictors}\n")

print("All 299, clean predictors only:")
print(corr_all[corr_all["predictor"].isin(clean_predictors)].to_string(index=False))
print("\nExcluding Berlin/Hamburg/Bremen, clean predictors only:")
print(corr_excl_bhb[corr_excl_bhb["predictor"].isin(clean_predictors)].to_string(index=False))


**Takeaway.** The political-lean pattern from §2.4 doesn't rest on any of the 10 flagged columns at all — every political predictor is clean by construction (election-denominated, not population-share-denominated), so `second_vote_left_pct` (+0.34), `second_vote_spd_pct` (+0.28), `second_vote_greens_pct` (+0.24) positive and `second_vote_cdu_csu_pct` (−0.27), `second_vote_afd_pct` (−0.20) negative all stand exactly as reported in §2.4, with or without Berlin/Hamburg/Bremen. What the 10 contaminated columns take with them is most of the *deprivation* story's specific texture: `sgb_ii_recipients_pct`, `unemployment_total`, `disposable_income_eur_per_capita`, `gdp_eur_per_capita`, `employees_pct_of_pop`, and five age-share columns can't be independently verified at the 18-constituency (Berlin+Hamburg) grain, so their exact ρ values inherit that uncertainty. What's left standing on its own is `foreign_population_pct` — clean, and still the single strongest predictor in the whole notebook (ρ=0.464 all constituencies, 0.431 excluding Berlin/Hamburg/Bremen, both FDR-significant) — so the deprivation/composition cluster's core claim survives using only uncontaminated data, even though its two next-strongest legs (`sgb_ii_recipients_pct`, `unemployment_total`) can't be cross-checked the same way. **Net effect of doing both 1 and 2**: nothing in §2.4's substantive conclusions depends on a contaminated column being trustworthy at face value — the political pattern was never at risk, and the deprivation pattern's weakest link is exactly the columns already flagged, not a new problem.

## 2.4e Is `pool_count` really a population-density / urban-rural proxy?

Already answered in Part 1's WHERE section, not re-derived here (that section re-ran this exact check with the same `population_density_per_km2`/`pool_count`/`racialised_rate` columns this notebook also has, so re-running it here would just reproduce the same numbers from the same source data): `pool_count` vs. `population_density_per_km2` runs ρ=-0.794 (more pools ↔ lower density -- Germany's equal-population constituency boundaries mean a sparse rural constituency must cover more land, and land, to accumulate more small pools, to reach the same population quota). Density independently out-predicts `pool_count` for `racialised_rate` (raw ρ=0.597 vs. ρ=-0.556) and survives controlling for `pool_count` (partial ρ=0.307, p<0.001), while `pool_count` controlling for density collapses to ρ=-0.169 (p=0.008) -- population density, not `pool_count`, is the more fundamental driver of the urban/rural pattern in `racialised_rate`. §2.4b's `pool_count`-based confound checks below stay as the primary control variable throughout this notebook regardless (matching Part 3's own choice, below, made on stronger multivariate grounds — AIC/BIC/cross-validation — than a single bivariate comparison like this one). (Figures refreshed to match `descriptive_analysis.ipynb`'s current WHO×WHERE capstone section — the density finding there was added/rerun after this cell was first written, and the two had drifted slightly out of sync.)

## 2.4f Re-running the AfD finding controlling for foreign population share and border status

§2.4 found `second_vote_afd_pct` negatively correlated with `racialised_rate` (ρ=-0.20), the opposite of a naive prior. §2.4 also showed this is confounded with `foreign_population_pct` (ρ=-0.54 between the two predictors themselves), and §2.4b showed the correlation collapses entirely once `pool_count` is controlled (ρ=+0.03, not significant). Neither of those checks has directly partialled out `foreign_population_pct` itself, or `is_land_border_constituency` (Part 1's corrected land-neighbour-only border flag, coastline excluded) -- border constituencies are disproportionately rural/eastern, where AfD support tends to be higher, so it's a plausible additional confound. `partial_spearman` (defined in §2.4b) only ever takes a single control variable -- it's called with `pool_count` in both its defining cell and in §2.4c's robustness check, and nowhere else -- so we add a small generalization, `partial_spearman_multi`, using the identical rank-residualization logic extended to more than one control column via a *multiple* (not simple) OLS regression on the ranks, rather than reaching for `pingouin` (not installed in this environment).

In [ ]:
def partial_spearman_multi(x, y, controls):
    """Generalizes partial_spearman to more than one control variable: rank-residualize x and y
    against a *multiple* OLS regression on the ranks of every control (instead of just one), then
    Spearman-correlate what's left. `controls` is a list of array-likes (may mix continuous and
    binary/dummy columns -- rankdata on a binary column just splits it into its two groups, so this
    is equivalent to using the raw dummy as a regressor)."""
    x, y = np.asarray(x, dtype=float), np.asarray(y, dtype=float)
    rx, ry = rankdata(x), rankdata(y)
    control_ranks = np.column_stack([rankdata(np.asarray(c, dtype=float)) for c in controls])
    Z = sm.add_constant(control_ranks)
    rx_resid = sm.OLS(rx, Z).fit().resid
    ry_resid = sm.OLS(ry, Z).fit().resid
    return stats.pearsonr(rx_resid, ry_resid)

afd_raw_rho, afd_raw_p = stats.spearmanr(analysis_df["second_vote_afd_pct"], analysis_df["racialised_rate"])
afd_poolcount_rho, afd_poolcount_p = partial_spearman(
    analysis_df["second_vote_afd_pct"], analysis_df["racialised_rate"], analysis_df["pool_count"]
)
afd_foreignpop_rho, afd_foreignpop_p = partial_spearman(
    analysis_df["second_vote_afd_pct"], analysis_df["racialised_rate"], analysis_df["foreign_population_pct"]
)
afd_joint_rho, afd_joint_p = partial_spearman_multi(
    analysis_df["second_vote_afd_pct"], analysis_df["racialised_rate"],
    [analysis_df["foreign_population_pct"], analysis_df["is_land_border_constituency"].astype(float)],
)

afd_control_table = pd.DataFrame([
    {"specification": "raw (§2.4)", "rho": afd_raw_rho, "p": afd_raw_p},
    {"specification": "ctrl. pool_count (§2.4b)", "rho": afd_poolcount_rho, "p": afd_poolcount_p},
    {"specification": "ctrl. foreign_population_pct", "rho": afd_foreignpop_rho, "p": afd_foreignpop_p},
    {"specification": "ctrl. foreign_population_pct + is_land_border_constituency", "rho": afd_joint_rho, "p": afd_joint_p},
])
print(afd_control_table.round(4).to_string(index=False))


**Takeaway.** The AfD-`racialised_rate` relationship is **fully explained away, not partially** -- and adding `is_land_border_constituency` on top of `foreign_population_pct` changes essentially nothing. The raw ρ=-0.204 (p=3.8e-4) drops to a small, non-significant, sign-flipped ρ≈+0.03 (p≈0.58) once `pool_count` is controlled, and to ρ≈+0.06 (p≈0.27) once `foreign_population_pct` is controlled instead -- both controls independently kill it, and neither leaves anything resembling a genuine negative AfD effect. Adding `is_land_border_constituency` jointly with `foreign_population_pct` barely moves the estimate at all (ρ≈+0.06, p≈0.31) -- border status adds essentially no further explanatory power once foreign population share is already accounted for. This holds with the corrected, coastline-excluded border measure just as it held with the old one -- swapping definitions doesn't change the conclusion, because the joint control was already dominated by `foreign_population_pct`. **Bottom line for the reviewer's question**: there is no independent AfD-`racialised_rate` relationship left after controlling for either confound alone, let alone both together -- the original univariate correlation was never picking up an AfD-specific effect, it was standing in for `foreign_population_pct` (and, separately, for `pool_count`/venue density) the whole time.

## 2.4g Arab ethnicity-share vs. socioeconomic/political predictors

Part 1's WHO section found a constituency's `Arab` share of its own racialised reviews correlates with `racialised_rate` (ρ=+0.211, p=0.001, n=247 `reliable_n` constituencies) and is the only `ethnicity_group4` share that survives controlling for `population_density_per_km2` (ρ=+0.132, p=0.039) -- `Germany`'s link (ρ=-0.161 raw) disappears entirely under that control and `Türkiye`'s (ρ=+0.180 raw) weakens to non-significance. That check only ever controlled for density; it was never run against this notebook's own deprivation/political predictors. This section does that directly, reusing `racialised_df["ethnicity_group4"]` (added to Part 1's interchange file specifically for this check) and `run_correlations` (generalized above to take any `target_col`, not just `racialised_rate`).

In [ ]:
# reuse racialised_df's ethnicity_group4 to build each constituency's Arab share of its own
# racialised reviews, exactly as Part 1's WHO section did -- restricted to the same
# reliable_n=247 constituencies Part 1 used, since a share computed over a handful of racialised
# reviews isn't stable.
eth_share = pd.crosstab(
    racialised_df.loc[racialised_df["ethnicity_group4"] != "Unknown", "constituency_code"],
    racialised_df.loc[racialised_df["ethnicity_group4"] != "Unknown", "ethnicity_group4"],
    normalize="index",
)
reliable_df = analysis_df[analysis_df["reliable_n"]].copy()
eth_share = eth_share.reindex(reliable_df["constituency_code"]).fillna(0)
reliable_df = reliable_df.merge(
    eth_share[["Arab"]].rename(columns={"Arab": "arab_share"}), on="constituency_code"
)

# sanity check against Part 1's own number before trusting anything downstream
rho_check, p_check = stats.spearmanr(reliable_df["arab_share"], reliable_df["racialised_rate"])
print(f"sanity check vs. Part 1 (expect rho=+0.211, p=0.001): rho={rho_check:+.3f}, p={p_check:.3f}")

arab_corr = run_correlations(reliable_df, target_col="arab_share")
print("\nArab share vs. socioeconomic/political predictors (reliable_n=247):")
print(arab_corr.to_string(index=False))

# Part 1's WHO section already controlled arab_share vs. racialised_rate for
# population_density_per_km2 (rho=0.132, p=0.039) -- not re-derived here. What Part 1 couldn't
# test (it didn't have this notebook's predictors yet) is whether pool_count -- the confound
# this notebook leans on throughout, per §2.4e -- does the same thing:
rho_p, p_p = partial_spearman(reliable_df["arab_share"], reliable_df["racialised_rate"], reliable_df["pool_count"])
print(f"\narab_share vs. racialised_rate | controlling for pool_count: rho={rho_p:+.3f} p={p_p:.3f}")


**Takeaway.** The sanity check confirms this notebook's `ethnicity_group4` (replicated from Part 1's own categorization, added to the interchange file for this check) reproduces Part 1's number exactly (ρ=+0.211, p=0.001) -- the new correlations above build on trustworthy ground. **Arab share is not simply riding on general foreign-population share** -- `foreign_population_pct` itself only weakly and non-significantly correlates with `arab_share` (ρ=+0.125, p=0.049, doesn't survive FDR) -- **it's specifically tracking the deprivation/labor-market cluster**: `sgb_ii_recipients_pct` (ρ=+0.314, p=4.5e-7), `unemployment_total` (ρ=+0.296, p=2.2e-6), and `disposable_income_eur_per_capita` negatively (ρ=-0.236, p=1.8e-4) are all FDR-significant, alongside a political pattern that mirrors §2.4's own (`turnout_pct` -0.240, `second_vote_spd_pct` +0.227, `second_vote_left_pct` +0.208, `second_vote_cdu_csu_pct` -0.180, all FDR-significant; `second_vote_afd_pct` is **not** significant here, ρ=-0.101, p=0.115 -- consistent with §2.4f's finding that AfD's link to the *rate* was never Arab-specific). Note three of these five significant predictors (`sgb_ii_recipients_pct`, `unemployment_total`, `disposable_income_eur_per_capita`) are exactly the Berlin/Hamburg-contaminated columns flagged in §2.3b -- the same caveat from §2.4d applies here.

Controlling for `pool_count` attenuates the raw ρ=0.211 to roughly half (ρ=+0.123, p=0.053, borderline by the conventional cutoff) -- essentially the same attenuation Part 1 found controlling for `population_density_per_km2` instead (ρ=0.132, p=0.039), consistent with §2.4e's finding that `pool_count` and density are two views of the same urban/rural mechanism. **Arab share weakens under this control, roughly by half, but doesn't vanish the way `Germany` and `Türkiye` shares did in Part 1** -- unlike those two, it now also has a plausible non-density explanation on the table (the deprivation/labor-market cluster above), which Part 1 never tested since it only had density available at that point.

## 2.5 Group comparisons

Correlation treats each predictor as continuous and linear-in-rank; a quartile-split comparison is a complementary, more interpretable cut for a supervisor meeting ("bottom-quartile vs. top-quartile constituencies differ this much"), and doesn't assume the relationship is monotonic across the whole range. We pick the two predictors with a clear theoretical motivation for this specific comparison rather than testing all 18 again: **`foreign_population_pct`** (the most directly relevant compositional variable to racialised discourse about pool-goers) and **`second_vote_afd_pct`** (the political variable a reader will most expect to see tested directly, especially given the counter-intuitive negative correlation just found). We use **Kruskal-Wallis** (not one-way ANOVA) because `racialised_rate` is a small, right-skewed proportion, not a normally distributed measure, and quartile groups here only have ~75 constituencies each — exactly the small-N/non-normal combination Kruskal-Wallis is built for. Effect size is reported as **epsilon-squared** (rank-based, the Kruskal-Wallis analogue of eta-squared).

In [ ]:
def kruskal_with_epsilon_squared(df, group_col, value_col):
    groups = [g[value_col].values for _, g in df.groupby(group_col, observed=True)]
    h_stat, p = stats.kruskal(*groups)
    n = len(df)
    k = len(groups)
    epsilon_sq = (h_stat - k + 1) / (n - k)  # rank-based effect size analogue of eta-squared
    return h_stat, p, epsilon_sq

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

for ax, predictor, label in zip(
    axes, ["foreign_population_pct", "second_vote_afd_pct"], ["foreign population %", "AfD second-vote %"]
):
    analysis_df[f"{predictor}_quartile"] = pd.qcut(analysis_df[predictor], 4, labels=["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"])
    h_stat, p, eps_sq = kruskal_with_epsilon_squared(analysis_df, f"{predictor}_quartile", "racialised_rate")
    print(f"{label}: Kruskal-Wallis H = {h_stat:.2f}, p = {p:.2e}, epsilon-squared = {eps_sq:.3f}")
    print(analysis_df.groupby(f"{predictor}_quartile", observed=True)["racialised_rate"].agg(["median", "mean", "count"]))
    print()

    sns.boxplot(data=analysis_df, x=f"{predictor}_quartile", y="racialised_rate", ax=ax, color="#8172B2")
    ax.set_title(f"Racialised rate by {label} quartile\nH={h_stat:.1f}, p={p:.1e}, ε²={eps_sq:.3f}")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=20)

plt.tight_layout()
plt.show()


**Takeaway.** The two predictors tell different-strength stories. **Foreign population % shows a clean, monotonic, moderate-sized effect**: median racialised rate rises steadily from 0.97% (lowest quartile) to 1.93% (highest quartile) — roughly double — with epsilon-squared = 0.196, the largest effect size in this notebook. **AfD vote share is a much weaker and less clean effect** (epsilon-squared = 0.031, statistically significant only at p<.01 not the near-zero p-values seen elsewhere): the pattern isn't a smooth gradient across quartiles but mostly one contrast — the lowest-AfD quartile sits noticeably higher (median 1.61%) than quartiles 2-4, which are all similar to each other (~1.10-1.15%). This is consistent with §2.4's finding that the AfD correlation is likely picking up the foreign-population confound rather than an AfD-specific effect: foreign population % has both the cleaner pattern and the much larger effect size, so it's the stronger and more defensible of the two candidate predictors going into any future modeling work.

## 2.6 Spatial autocorrelation

Moved to `descriptive_analysis.ipynb`'s WHERE section (right after the choropleths), since testing whether `racialised_rate`'s clustering is real needs only the rate itself and constituency geometry -- both already available there, none of this notebook's socioeconomic/political data. Headline result, for reference: **Global Moran's I = 0.28** (z=7.91, p=0.001 over 999 permutations) -- real, significant positive spatial autocorrelation, with two named LISA hotspot clusters (Ruhr-area NRW, Berlin) that turn out to sit in exactly the higher-deprivation constituencies §2.4-2.4g above flag as correlated with the rate.

## 2.7 Dimension × political-lean cross-tabs: AfD-vote-share tercile

This is the one place where we deliberately don't just flag-and-skip because of thin cells -- the sub-dimensions need to count as real quantitative content for the paper, not only a caveat. The region × connotation/theme cut moved to `descriptive_analysis.ipynb`'s WHO section (right after its ethnicity-conditioned region test) since `region` is derived from `state` alone and needs no political data. What's left here needs this notebook's data specifically: **constituency AfD-vote-share tercile × connotation/theme** -- chosen because `second_vote_afd_pct` is the political variable most likely to be asked about directly, despite §2.4-2.4f already showing its univariate correlation with the *rate* is confounded by foreign population share/venue density. This does **not** carry that confound over automatically -- this cross-tab asks a different question (given a review is already racialised, does its *content* differ by political context?), so it's still worth running rather than assuming the answer. We report cell counts alongside every test and use Fisher's exact test (not just chi-square) wherever expected cell counts drop below 5, rather than excluding those results.

In [ ]:
def chi2_cramers_v(ct):
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    n = ct.values.sum()
    v = np.sqrt(chi2 / (n * (min(ct.shape) - 1)))
    min_expected = expected.min()
    return chi2, p, dof, v, min_expected


In [ ]:
# --- constituency AfD-vote-share tercile x connotation / theme (secondary, exploratory cut) ---
analysis_df["afd_tercile"] = pd.qcut(
    analysis_df["second_vote_afd_pct"], 3, labels=["Low AfD", "Mid AfD", "High AfD"]
)
afd_tercile_lookup = analysis_df.set_index("constituency_code")["afd_tercile"]
racialised_df = racialised_df.assign(afd_tercile=racialised_df["constituency_code"].map(afd_tercile_lookup))

print("racialised reviews per AfD tercile (constituency-level tercile, broadcast to reviews):")
print(racialised_df["afd_tercile"].value_counts())
print()

ct_afd_conn = pd.crosstab(racialised_df["afd_tercile"], racialised_df["connotation"].apply(lambda x: x[0]))
chi2, p, dof, v, min_exp = chi2_cramers_v(ct_afd_conn)
print("AfD tercile x connotation:")
print(ct_afd_conn)
print(f"chi2={chi2:.1f}, dof={dof}, p={p:.2e}, Cramer's V={v:.3f}, min expected count={min_exp:.1f}\n")

afd_theme = racialised_df[["review_id", "afd_tercile", "theme"]].explode("theme")
ct_afd_theme = pd.crosstab(afd_theme["afd_tercile"].to_numpy(), afd_theme["theme"].to_numpy())
chi2, p, dof, v, min_exp = chi2_cramers_v(ct_afd_theme)
print("AfD tercile x theme:")
print(ct_afd_theme)
print(f"chi2={chi2:.1f}, dof={dof}, p={p:.2e}, Cramer's V={v:.3f}, min expected count={min_exp:.1f}")
n_thin_cells = (ct_afd_theme.values < 5).sum()
print(f"cells with raw count < 5: {n_thin_cells} / {ct_afd_theme.size}")


**Takeaway.** Across both cuts, **connotation doesn't shift with political/regional context, but theme does** — a genuinely informative and fairly clean pattern:

- **Region × connotation**: not significant (χ²=6.7, p=0.15, V=0.021). **Region × theme**: significant but small (χ²=84.7, p=6.2e-10, V=0.056) — one cell (`Other` theme in Berlin) has an expected count of 4.5, just under the conventional 5 threshold, so that specific cell should be read cautiously, but the overall test isn't driven by it alone.
- **AfD tercile × connotation**: not significant (χ²=4.2, p=0.38, V=0.016) — all cell counts are comfortably large (min expected 101.3), so this is a clean null, not an underpowered one. **AfD tercile × theme**: significant, small effect (χ²=70.4, p=1.6e-7, V=0.051), and here every cell clears the count-5 threshold (min expected 37, no thin cells at all) — the cleanest-powered test of the four. The clearest single contrast: `Security` as a theme is notably more common in Low-AfD-tercile reviews (102) than Mid (73) or High-AfD-tercile (47) — roughly 2x more frequent at the low end than the high end.

**Overall reading**: once a review is already racialised, *how negative/neutral/positive* it reads doesn't depend on the region or the local political lean — but *what it's about* (crowd, staff, security, etc.) does, modestly, in both cuts. This is a legitimately new, adequately-powered quantitative finding about the sub-dimensions (not just a caveated non-result), and it's a useful qualifier on Part 2's earlier area-level correlations: those were about *how often* racialised discourse occurs; this section shows its *content* is comparatively stable by connotation but does vary somewhat by theme across political/regional context — worth a closer qualitative read on the `Security`/`Safety` themes specifically, which also stood out in §1.6's full-sample theme×connotation analysis.

## 2.8 Closing synthesis: candidate refined RQs for the supervisor meeting

**What's well-supported right now, across both notebooks:**
- Overall racialised-review prevalence (1.54% of social reviews, tight CI) and its constituency/state distribution, backed by 83% constituency coverage (247/299, at Part 1's current `racialised_reviews >= 10` reliability bar) at a defensible reliability bar based on racialised/social-review counts, not pool count (`descriptive_analysis.ipynb`'s WHERE section).
- A real, complete-data (99.94% timestamp coverage after recovery) temporal rise from 0.92% (2021) to 2.11% (2025).
- Genuine, statistically confirmed spatial clustering (Global Moran's I = 0.28, p=0.001) in two named hotspot regions: Ruhr-area NRW and Berlin (computed in `descriptive_analysis.ipynb`'s WHERE section, right after the choropleths -- moved there from an earlier draft of this notebook since it needs none of Part 2's data).
- A deprivation cluster (`sgb_ii_recipients_pct`, `foreign_population_pct`, `unemployment_total`) that survives controlling for the strongest confound found (`pool_count`/venue density, §2.4b) — attenuated to about half its univariate magnitude (ρ≈0.23-0.28 partial vs. 0.43-0.47 univariate) but still FDR-significant. `turnout_pct` also survives essentially unchanged.
- A specific, now doubly-confirmed caution around the political predictors: `second_vote_afd_pct`'s univariate correlation (ρ=−0.20) collapses to ρ=+0.03 (p=0.58, not significant) once `pool_count` is controlled — an even more decisive confound than the §2.4 check against `foreign_population_pct` alone found. **This was never an independent AfD effect.**
- A broader methodological finding from §2.4b that reshapes how to read *all* of §2.4: almost every predictor (age structure, `second_vote_left_pct`/`greens_pct`/`cdu_csu_pct`, GDP, employment share) is itself strongly correlated with `pool_count` (|ρ| 0.32-0.57), and loses FDR significance once that's controlled for — meaning most of §2.4's univariate correlations, taken individually, overstate how much each variable independently matters. Only the deprivation cluster and turnout hold up to this specific check.
- A clean split in how the racialised reviews' *sub-dimensions* relate to context: `connotation` is stable across region/political-lean, `theme` shifts modestly (small but adequately powered effects, notably around `Security`).
- **A new, specific compositional lead** (§2.4g, following up on Part 1's WHO section): a constituency's `Arab` share of its own racialised reviews tracks the deprivation/labor-market cluster directly (`sgb_ii_recipients_pct` ρ=0.31, `unemployment_total` ρ=0.30, `disposable_income_eur_per_capita` ρ=-0.24, all FDR-significant) and a political pattern mirroring §2.4's own -- but **not** `foreign_population_pct` (ρ=0.13, not FDR-significant) or `second_vote_afd_pct` (ρ=-0.10, ns). Attenuates to roughly half under `pool_count`/density control (ρ≈0.12, borderline p≈0.05-0.06) rather than vanishing, unlike `Germany`/`Türkiye` share in Part 1 -- a genuine candidate covariate for the deferred multivariate model, not just a density proxy.
- **A data-provenance fix, not a finding**: `const_agg` was rebuilt since this notebook was first written, replacing the coastline-inclusive `is_border_constituency` with the corrected, land-neighbour-only `is_land_border_constituency`. Re-run under the corrected flag, §2.4f's conclusion is unchanged (border status still adds nothing once `foreign_population_pct` is controlled) -- flagged here so the swap doesn't get mistaken for a substantive result.
- **§2.4d's robustness check was extended to also report a "clean predictors only" view** (§2.3b's 10 Berlin/Hamburg-contaminated columns dropped entirely, not just Berlin/Hamburg/Bremen constituencies dropped): the political-lean pattern is fully clean by construction and unaffected either way; the deprivation story's one clean leg, `foreign_population_pct`, remains the strongest predictor in the notebook (ρ=0.46) on its own, but `sgb_ii_recipients_pct`/`unemployment_total`/income can't be independently cross-checked at the Berlin+Hamburg grain.

**Three sharper RQs this supports, in order of how supported they already are:**

1. **"Where and when does racialised discourse concentrate, and is that concentration spatially structured or incidental?"** — fully supported now with Moran's I/LISA + the temporal trend; could be written up close to as-is.
2. **"What structural (socioeconomic, demographic) features of a place are associated with racialised-discourse prevalence, net of confounding between those features — including venue density itself?"** — §2.4b already shows this can't be answered with one-at-a-time partial correlations (too many predictors are mutually confounded with `pool_count` and likely each other); it needs the deferred multivariate model (negative-binomial GLM with a review-volume offset, `pool_count`/venue-density included as a covariate rather than partialled out separately). That's the next notebook, once IAA/robustness work has established how much to trust the `racialised` label itself feeding into it.
3. **"Does the *content* of racialised discourse (not just its frequency) vary systematically by place?"** — §2.7 gives an initial, adequately-powered answer (theme shifts modestly, connotation doesn't), but this was only tested against two political/regional cuts; a fuller treatment (e.g. against the deprivation variables that were strongest in RQ2, or against `approach`/`context` which weren't tested here) is a natural extension once the sub-dimension labels have been through the planned inter-annotator-agreement check — worth prioritising IAA on `theme` specifically, since it's the dimension carrying the geographic signal.

**What this notebook deliberately does not do**, so the next steps are unambiguous: no predictive/explanatory model, no causal language anywhere (every Part 2 result is explicitly framed as area-level/ecological), and no dimension-level constituency-grain modeling (only region- and tercile-level, which have adequate power; true constituency-grain dimension analysis would need either a much larger racialised-review sample or a deliberate oversampling strategy).

# Part 3 — Predictive modelling: racialised rate as a function of socioeconomic/political structure

**Scope for this pass** (deliberately simple, per the plan set out in `02_socioeconomic_political_relations.ipynb` §2.8): a single multivariate model with `pool_count` included as a covariate, rather than the one-at-a-time partial correlations §2.4b had to rely on. This directly follows §2.8's RQ2 spec: *"a negative-binomial GLM with a review-volume offset, `pool_count`/venue-density included as a covariate rather than partialled out separately."* This is a preliminary pass on the working assumption that the IAA and robustness work already completed is sufficient to trust the `racialised` label going into a model — both the label and the model itself are expected to be iterated on further.

## 3.0 Setup — continuing from Parts 1 and 2

Reuses `analysis_df` and `PREDICTOR_COLS` exactly as built in Part 2 above (§2.1 merge, §2.3 derived percentage columns) -- no reload, no re-derivation, no separate `FULL_PREDICTOR_COLS` name for the same list.

In [ ]:
from statsmodels.discrete.discrete_model import NegativeBinomial
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import KFold

analysis_df["log_social_reviews"] = np.log(analysis_df["social_reviews"])

print(f"analysis_df: {analysis_df.shape[0]} constituencies x {analysis_df.shape[1]} cols")
print(f"{len(PREDICTOR_COLS)} §2.4 predictor candidates + pool_count")
analysis_df[["constituency", "state", "racialised_reviews", "social_reviews", "pool_count", "reliable_n"]].head()


## 3.1 Model specification

Two choices, each following directly from what Parts 1-2 already established:

- **Outcome is the count, not the rate, with an offset.** `racialised_reviews` (not `racialised_rate`) is the response, with `log(social_reviews)` as an offset — the standard way to model a rate that comes from a variable-sized denominator (analogous to person-time in epidemiology). It lets the model use the count's actual sampling variance instead of treating `racialised_rate` as a continuous measurement with constant precision, which it isn't: a constituency with 300 social reviews and one with 3,000 have very different sampling noise around the same rate.
- **Negative binomial, not Poisson.** §2.6 already found genuine spatial clustering (Moran's I = 0.28, p = 0.001) — clustering implies extra between-constituency variance beyond what a Poisson model's mean-equals-variance assumption allows. We fit the Poisson model first specifically to demonstrate this overdispersion numerically before moving to NB2, rather than asserting it.

What still needs deciding is *which* covariates go in alongside `pool_count` — §3.2 below.

In [ ]:
X_full = sm.add_constant(analysis_df[["pool_count"] + PREDICTOR_COLS])
y = analysis_df["racialised_reviews"]
offset = analysis_df["log_social_reviews"]

poisson_full = sm.GLM(y, X_full, family=sm.families.Poisson(), offset=offset).fit()
dispersion_full = poisson_full.pearson_chi2 / poisson_full.df_resid
print(f"Poisson GLM (pool_count + all {len(PREDICTOR_COLS)} predictors): "
      f"Pearson chi2 / df = {dispersion_full:.2f} (>>1 => overdispersed, NB2 needed either way)")

Pearson chi2/df = 7.99, well above the 1.0 a well-specified Poisson model would show — confirms overdispersion, consistent with §2.6's spatial clustering finding, and is the empirical justification for using NB2 rather than Poisson throughout the rest of this notebook (the reduced-set Poisson model in §3.3 shows the same pattern, chi2/df = 9.46).

## 3.2 Predictor set: why not just put all 18 in?

§2.4b/§2.8 called for "`pool_count` as a covariate, not partialled out separately" precisely *because* the §2.4 predictors are themselves heavily intercorrelated with `pool_count` and with each other (deprivation, age structure, and vote share all move together). The natural first attempt is therefore the maximal one: `pool_count` + all 18 predictors in a single NB2. We check that model's viability with VIF (variance inflation factor) *before* trusting any coefficient from it — a VIF above ~10 means a predictor's estimated effect is essentially undetermined by the data, drowned out by its correlation with the other covariates in the model, not evidence of a null effect.

In [ ]:
vif_full = pd.DataFrame({
    "predictor": X_full.columns,
    "vif": [variance_inflation_factor(X_full.values, i) for i in range(X_full.shape[1])],
}).drop(index=0)  # drop the constant term's VIF, which is uninformative here
vif_full.sort_values("vif", ascending=False)

Confirms the concern directly: VIF exceeds 10 for **11 of the 19 covariates**, topping out at **79.7 for `second_vote_afd_pct`**, 58.6 for `second_vote_cdu_csu_pct`, and 38.4 for `second_vote_greens_pct` — the vote-share variables are so intercorrelated with each other (they're shares of the same total, on top of §2.4b's already-established correlation with `pool_count`) that no single one of them has an independently estimable effect in a model that includes all of them at once. `pool_count` itself has a comfortable VIF of 2.3 — it's the combination of many mutually collinear vote-share/deprivation variables that's unusable, not `pool_count`'s inclusion.

Given that, we don't fit the 19-covariate NB2 as the primary model at all — attempting it directly confirms the concern: the MLE fails to converge (Hessian inversion fails, so no standard errors are even computable) and 5-fold cross-validation on it produces predictions that are numerically unstable across folds. That's not a coding bug, it's the expected consequence of asking an MLE to separate the effects of 19 covariates, several pairwise/multiply collinear, from n=299 constituencies.

Instead we use the **6-predictor set §2.4b already validated as robust to the `pool_count` confound** — `sgb_ii_recipients_pct`, `foreign_population_pct`, `unemployment_total`, `turnout_pct`, `second_vote_spd_pct`, `second_vote_bsw_pct` (the only ones that stayed FDR-significant once `pool_count` was partialled out one at a time). Moving from "one-at-a-time partial correlation" to "one joint multivariate model" only for *this* set is the actual intuitive next step from §2.4b/§2.8 — it keeps the covariate-to-observation ratio sane (7 covariates, n=299) while still jointly modelling the interactions those 6 predictors have with each other and with `pool_count`, which one-at-a-time partial correlation could not do.

In [ ]:
REDUCED_PREDICTOR_COLS = [
    "sgb_ii_recipients_pct", "foreign_population_pct", "unemployment_total",
    "turnout_pct", "second_vote_spd_pct", "second_vote_bsw_pct",
]
MODEL_COLS = ["pool_count"] + REDUCED_PREDICTOR_COLS

X = sm.add_constant(analysis_df[MODEL_COLS])

vif_reduced = pd.DataFrame({
    "predictor": X.columns,
    "vif": [variance_inflation_factor(X.values, i) for i in range(X.shape[1])],
}).drop(index=0)
vif_reduced.sort_values("vif", ascending=False)

Much better, but not perfect: `sgb_ii_recipients_pct` (VIF 26.8) and `unemployment_total` (VIF 25.8) are still highly collinear with each other — unsurprising, since both are deprivation measures — while `pool_count` and the remaining four predictors are all comfortably below 4. This is a known limitation of this pass (flagged again in §3.7): the model below can jointly estimate "the deprivation cluster's" effect reasonably well, but can't cleanly separate `sgb_ii_recipients_pct`'s individual contribution from `unemployment_total`'s.

## 3.3 Fitting the negative binomial model on the reduced set

In [ ]:
poisson_model = sm.GLM(y, X, family=sm.families.Poisson(), offset=offset).fit()
dispersion = poisson_model.pearson_chi2 / poisson_model.df_resid
print(f"Poisson GLM (reduced set): Pearson chi2 / df = {dispersion:.2f}")

nb_full = NegativeBinomial(y, X, offset=offset).fit(disp=False)
print(nb_full.summary())

In [ ]:
irr_table = pd.DataFrame({
    "coef": nb_full.params,
    "irr": np.exp(nb_full.params),
    "irr_ci_lo": np.exp(nb_full.conf_int()[0]),
    "irr_ci_hi": np.exp(nb_full.conf_int()[1]),
    "p": nb_full.pvalues,
}).drop(index=["alpha", "const"], errors="ignore")
irr_table.sort_values("p")

**Takeaway.** The model converges cleanly this time (`converged: True`), and is overall highly significant (LLR p = 6.8e-30 vs. the null offset-only model), with a modest pseudo-R² of 0.066 — expected for a preliminary ecological model of a rare, noisy outcome. Two covariates are clearly significant once everything is modelled jointly:

- **`foreign_population_pct`** (IRR = 1.034, 95% CI [1.019, 1.049], p < 0.0001): each 1 percentage-point increase in foreign-population share is associated with a 3.4% higher expected racialised-review rate, holding `pool_count` and the other five predictors constant. This is the one predictor that has now survived three separate checks — §2.4's univariate correlation (ρ=0.46), §2.4b's `pool_count`-partialled correlation (ρ=0.28), and this joint multivariate model — making it the most defensible finding across all three notebooks so far.
- **`pool_count`** (IRR = 0.989, 95% CI [0.980, 0.997], p = 0.010): each additional pool in a constituency is associated with a ~1.1% lower expected rate, confirming §2.4b's confound finding survives being modelled as a covariate rather than partialled out.

Everything else is weaker than §2.4b's partial correlations suggested: `sgb_ii_recipients_pct` (p=0.070) and `turnout_pct` (p=0.096) are only marginal, and **`unemployment_total`'s sign flips to negative** (coef=-0.068, p=0.317, not significant) — almost certainly an artifact of its VIF=25.8 collinearity with `sgb_ii_recipients_pct` (§3.2) rather than a real reversal of the §2.4/§2.4b positive association; the two deprivation variables are fighting over the same shared variance once modelled together. `second_vote_spd_pct` and `second_vote_bsw_pct` are both null here (p=0.849, p=0.461) — §2.4b's partial correlations for these two (ρ=0.14, ρ=0.15) don't survive being placed in a joint model alongside the stronger deprivation/composition predictors.

In [ ]:
X_reduced_null = sm.add_constant(analysis_df[["pool_count"]])
nb_pool_only = NegativeBinomial(y, X_reduced_null, offset=offset).fit(disp=False)

lr_stat = 2 * (nb_full.llf - nb_pool_only.llf)
df_diff = len(nb_full.params) - len(nb_pool_only.params)
p_lr = stats.chi2.sf(lr_stat, df_diff)

print(f"pool_count-only NB2: log-likelihood = {nb_pool_only.llf:.1f}")
print(f"pool_count + {len(REDUCED_PREDICTOR_COLS)} predictors NB2: log-likelihood = {nb_full.llf:.1f}")
print(f"likelihood-ratio test: LR = {lr_stat:.1f}, df = {df_diff}, p = {p_lr:.2e}")

**Takeaway.** The 6 predictors jointly add real explanatory power beyond `pool_count` alone: LR = 77.2 on 6 df, p = 1.3e-14. This directly answers §2.8's RQ2 — "is there a real multivariate model here, or does pool_count/venue density explain the pattern by itself" — and confirms that no, deprivation and composition variables (mainly `foreign_population_pct`) explain something venue density alone does not.

## 3.4 Does the model actually predict better, out-of-sample?

§3.3 tests whether the predictors are jointly *significant* in-sample; that's a different question from whether they improve *predictive* performance on constituencies the model hasn't seen. We run 5-fold cross-validation comparing the `pool_count`-only model against the reduced model, scoring each fold's held-out predictions on the rate scale (predicted rate vs. actual `racialised_rate`) with mean absolute error — a simple, interpretable predictive check appropriate for a preliminary pass on n=299.

In [ ]:
def cv_mae(cols, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    X_full_cv = sm.add_constant(analysis_df[cols])
    y_full = analysis_df["racialised_reviews"]
    offset_full = analysis_df["log_social_reviews"]
    rate_actual = analysis_df["racialised_rate"].values
    errors = []
    for train_idx, test_idx in kf.split(np.arange(len(analysis_df))):
        model = NegativeBinomial(
            y_full.iloc[train_idx], X_full_cv.iloc[train_idx], offset=offset_full.iloc[train_idx]
        ).fit(disp=False)
        pred_rate = np.exp(X_full_cv.iloc[test_idx].values @ model.params.drop("alpha").values)
        errors.append(np.abs(pred_rate - rate_actual[test_idx]))
    return np.concatenate(errors)

mae_pool_only = cv_mae(["pool_count"])
mae_reduced = cv_mae(MODEL_COLS)

print("5-fold CV mean absolute error (percentage points of racialised_rate):")
print(f"  pool_count-only model: {mae_pool_only.mean() * 100:.3f} pp")
print(f"  reduced model:         {mae_reduced.mean() * 100:.3f} pp")

**Takeaway.** The reduced model beats the `pool_count`-only model out-of-sample too, not just in-sample: 0.610pp mean absolute error vs. 0.691pp — an ~12% relative reduction. The improvement is real but modest, consistent with the in-sample picture above: one strong predictor (`foreign_population_pct`) plus a weak confound-adjustment covariate (`pool_count`), rather than a large multi-predictor gain. That's expected for a first predictive pass on a rare, noisy ecological outcome with n=299, and it sets a concrete baseline (0.61pp MAE) that future iterations of this model can be compared against.

## 3.4b Does `population_density_per_km2` outperform `pool_count` as the density covariate?

`02_socioeconomic_political_relations.ipynb` §2.7-2.8 added `population_density_per_km2` to the data pipeline and found it out-predicts `pool_count` in raw correlation terms (density vs `racialised_rate` ρ=0.54 vs `pool_count` vs `racialised_rate` ρ=-0.49), while the two are themselves strongly collinear (Spearman ρ=-0.80: more pools ↔ lower density, confirming `pool_count` is essentially a noisy proxy for density). That reframes a question this notebook's §3.1-3.3 model didn't ask: is `pool_count` in `MODEL_COLS` actually the best choice of density covariate, or would swapping in `population_density_per_km2` (or including both) fit better?

We compare three specifications, all using the same 6-predictor `REDUCED_PREDICTOR_COLS` set from §3.2:
1. **`pool_count`-based** (the existing §3.3 model, `nb_full`).
2. **Density-only**: `population_density_per_km2` in place of `pool_count`.
3. **Both together**: `pool_count` and `population_density_per_km2` jointly (checking whether their ρ=-0.80 collinearity makes this poorly identified).

First, though: `population_density_per_km2` ranges from ~37/km² in rural constituencies to ~12,190/km² in dense Berlin seats, which is likely far more right-skewed than `pool_count` and may need a log-transform before it can be used as a sane linear covariate in the GLM's linear predictor.

In [ ]:
print("Skewness (right-skew if >> 0):")
print(f"  pool_count:                          {analysis_df['pool_count'].skew():.3f}")
print(f"  population_density_per_km2 (raw):    {analysis_df['population_density_per_km2'].skew():.3f}")
print(f"  log(population_density_per_km2):     {np.log(analysis_df['population_density_per_km2']).skew():.3f}")

analysis_df["log_population_density"] = np.log(analysis_df["population_density_per_km2"])

print(f"\nSpearman rho, pool_count vs population_density_per_km2 (raw): "
      f"{stats.spearmanr(analysis_df['pool_count'], analysis_df['population_density_per_km2'])[0]:.3f}")

`population_density_per_km2`'s skewness (~3.3) is roughly 3.7x `pool_count`'s (~0.9) and drops to a much more GLM-friendly ~0.6 once log-transformed — confirming the concern flagged in the task spec. `pool_count` itself is mildly right-skewed but nowhere near as extreme, and it's already working fine in the existing §3.3 model (IRR, CIs, and diagnostics all sensible), so per the "don't change what isn't broken" principle it is left untransformed. `population_density_per_km2` is used in **log form** (`log_population_density`) throughout this section — confirmed necessary below, not just a precaution: the raw-scale density model fails to converge.

In [ ]:
REDUCED_PLUS = REDUCED_PREDICTOR_COLS  # sgb_ii_recipients_pct, foreign_population_pct, unemployment_total,
                                        # turnout_pct, second_vote_spd_pct, second_vote_bsw_pct

def fit_reduced(cols):
    Xv = sm.add_constant(analysis_df[cols])
    return Xv, NegativeBinomial(y, Xv, offset=offset).fit(disp=False)

# Variant 1: pool_count-based -- this is exactly nb_full from §3.3, refit here under a
# variant-specific name for a clean side-by-side comparison.
X_v_pool, nb_v_pool = fit_reduced(["pool_count"] + REDUCED_PLUS)

# Variant 1b: does the raw (untransformed) density scale even converge? (diagnostic only)
X_v_density_raw, nb_v_density_raw = fit_reduced(["population_density_per_km2"] + REDUCED_PLUS)
print(f"Raw-scale density model converged: {nb_v_density_raw.mle_retvals.get('converged')}")

# Variant 2: density-only (log scale)
X_v_density, nb_v_density = fit_reduced(["log_population_density"] + REDUCED_PLUS)

# Variant 3: both together (log density + pool_count)
X_v_both, nb_v_both = fit_reduced(["pool_count", "log_population_density"] + REDUCED_PLUS)

vif_both = pd.DataFrame({
    "predictor": X_v_both.columns,
    "vif": [variance_inflation_factor(X_v_both.values, i) for i in range(X_v_both.shape[1])],
}).drop(index=0)
vif_both.sort_values("vif", ascending=False)

**Convergence and VIF.** The raw-scale density model fails to converge (`converged: False`) -- confirming the skew concern isn't just cosmetic, it breaks the MLE outright, exactly as the log-transform diagnostic above predicted. The log-scale density model converges cleanly.

For the "both together" spec, `log_population_density`'s VIF is **7.24** against `pool_count`'s 2.44 -- elevated, and by far the third-highest VIF in that model after the already-known `sgb_ii_recipients_pct`/`unemployment_total` pair (~26-29 each, unchanged from §3.2), but it stays under the conventional VIF>10 danger threshold. So the two aren't so collinear that a joint model is uninterpretable the way the 19-covariate model in §3.2 was -- but as the LR test below shows, "not fatally collinear" isn't the same as worth including both.

In [ ]:
def cv_mae_variant(cols, n_splits=5, seed=42):
    kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)
    Xcv = sm.add_constant(analysis_df[cols])
    rate_actual = analysis_df["racialised_rate"].values
    errors = []
    for train_idx, test_idx in kf.split(np.arange(len(analysis_df))):
        model = NegativeBinomial(y.iloc[train_idx], Xcv.iloc[train_idx], offset=offset.iloc[train_idx]).fit(disp=False)
        pred_rate = np.exp(Xcv.iloc[test_idx].values @ model.params.drop("alpha").values)
        errors.append(np.abs(pred_rate - rate_actual[test_idx]))
    return np.concatenate(errors)

variant_cols = {
    "pool_count-based":  ["pool_count"] + REDUCED_PLUS,
    "density-only (log)": ["log_population_density"] + REDUCED_PLUS,
    "both together":      ["pool_count", "log_population_density"] + REDUCED_PLUS,
}
variant_models = {"pool_count-based": nb_v_pool, "density-only (log)": nb_v_density, "both together": nb_v_both}

# LR tests: "both" nests both single-covariate variants (drop one term each), so both are
# directly testable; pool_count-only vs density-only are non-nested (can't LR test directly).
lr_add_density = 2 * (nb_v_both.llf - nb_v_pool.llf)
p_add_density = stats.chi2.sf(lr_add_density, 1)
lr_add_pool = 2 * (nb_v_both.llf - nb_v_density.llf)
p_add_pool = stats.chi2.sf(lr_add_pool, 1)

rows = []
for name, cols in variant_cols.items():
    m = variant_models[name]
    mae = cv_mae_variant(cols).mean() * 100
    fp_row = {
        "spec": name, "AIC": m.aic, "BIC": m.bic, "llf": m.llf,
        "cv_mae_pp": mae,
        "foreign_population_pct_IRR": np.exp(m.params["foreign_population_pct"]),
        "foreign_population_pct_p": m.pvalues["foreign_population_pct"],
    }
    rows.append(fp_row)
comparison = pd.DataFrame(rows).set_index("spec")
print(comparison.round(4))

print(f"\nLR test, adding log_population_density to pool_count-based model: "
      f"LR={lr_add_density:.3f}, df=1, p={p_add_density:.4f}")
print(f"LR test, adding pool_count to density-only model: "
      f"LR={lr_add_pool:.3f}, df=1, p={p_add_pool:.4f}")

**Takeaway -- three-way comparison and recommendation.**

| Spec | AIC | BIC | 5-fold CV MAE (pp) | `foreign_population_pct` IRR (p) |
|---|---|---|---|---|
| `pool_count`-based (current, §3.3) | 2200.6 | 2233.9 | 0.610 | 1.034 (p<0.0001) |
| Density-only (log) | 2202.8 | 2236.1 | 0.613 | 1.026 (p=0.014) |
| Both together | 2201.6 | 2238.6 | 0.615 | 1.027 (p=0.010) |

Three findings:

1. **`foreign_population_pct` is robust across every specification tried.** IRR ranges narrowly from 1.026 to 1.034 (a 2.6-3.4pp effect per 1pp of foreign-population share), and it is significant at p<0.05 in all three, p<0.0001 in the primary spec. Swapping the density covariate does not touch this finding -- exactly what notebook 02's independent confirmation would predict.
2. **The `pool_count`-based model still edges out both alternatives on every metric** (lowest AIC/BIC, lowest CV MAE), though the margins are small (AIC differs by ~1-2 points, CV MAE by ~0.005pp) -- these are not decisive wins, more like a photo finish. Consistent with that, the LR tests confirm neither addition earns its extra parameter: adding `log_population_density` to the `pool_count`-based model is not significant (LR=0.98, p=0.32), and adding `pool_count` to the density-only model is only marginal (LR=3.20, p=0.074).
3. **We do not adopt the "both together" specification.** Even though its VIF (7.24 for `log_population_density`) stays just under the conventional danger threshold and the model is technically estimable, the LR tests above show it isn't buying real explanatory power over either single-covariate variant -- it's just spending a degree of freedom on two measures of essentially the same underlying construct (pool density vs population density, ρ=-0.80).

**Recommendation: keep `pool_count` as the primary density covariate** (`MODEL_COLS` and `nb_full` in §3.2-3.3 are unchanged). It has a marginal but consistent edge in fit and out-of-sample prediction, and -- unlike `population_density_per_km2` -- it doesn't need a log-transform to be usable, which keeps the existing IRR-based interpretation simple. This is a closer call than notebook 02's raw correlation numbers alone would suggest (density "wins" ρ=0.54 vs -0.49 there), and it's worth being honest about why: once `foreign_population_pct` and the other five predictors are already in the model, whichever density measure is added contributes comparatively little either way, and `pool_count`'s advantage is small enough that a future iteration with a cleaner or updated density measure could plausibly flip this. For now, `population_density_per_km2` (log-transformed) is flagged as a close, theoretically well-motivated second choice worth revisiting rather than a decisively rejected alternative -- and the log-transform is a **permanent requirement if it's ever adopted**, not just a one-off diagnostic check for this comparison (the raw scale genuinely fails to converge).

## 3.5 Robustness check: restricting to the `reliable_n` constituencies

Same check §2.4 ran for the univariate correlations -- refit the model restricted to constituencies clearing §1.2's reliability bar. **Note:** `descriptive_analysis.ipynb` tightened this bar from `social_reviews>=100 AND racialised_reviews>=5` to `social_reviews>=100 AND racialised_reviews>=10` -- so the reliable subset is now smaller than it was when this notebook was first built (247/299 pass, not the 288/299 the bar used to yield; that number is computed directly from `analysis_df["reliable_n"]` just below rather than hardcoded, so it always reflects whatever the upstream notebook currently produces). We compare coefficients to the all-299 model to see whether any result is being driven by the (now 52, not 11) thinner constituencies excluded by the stricter bar.

In [ ]:
n_reliable = int(analysis_df["reliable_n"].sum())
print(f"reliable_n constituencies: {n_reliable} / {len(analysis_df)}")

reliable_df = analysis_df[analysis_df["reliable_n"]]
X_rel = sm.add_constant(reliable_df[MODEL_COLS])
y_rel = reliable_df["racialised_reviews"]
offset_rel = reliable_df["log_social_reviews"]

nb_reliable = NegativeBinomial(y_rel, X_rel, offset=offset_rel).fit(disp=False)

coef_compare = pd.DataFrame({
    "coef_all_299": nb_full.params,
    f"coef_reliable_{n_reliable}": nb_reliable.params,
}).drop(index="alpha", errors="ignore")
coef_compare["shift"] = coef_compare[f"coef_reliable_{n_reliable}"] - coef_compare["coef_all_299"]
coef_compare.sort_values("shift", key=np.abs, ascending=False)

**Takeaway.** All 7 coefficients are stable between the 299-constituency and the (now 247, corrected from a stale 288 -- see §3.5 header) `reliable_n` fits -- the largest shifts are `unemployment_total` (-0.039) and `sgb_ii_recipients_pct` (+0.026), both larger in absolute terms than the old 288-subset check reported (expected, since the stricter `>=10` bar now excludes 52 constituencies rather than 11, a much bigger and more different subset), but still small relative to their standard errors in §3.3 and every coefficient keeps its sign. None of §3.3's findings are being driven by the thin constituencies the reliability bar screens out, matching the pattern §2.4's correlation robustness check already found -- the stricter threshold changes *which* constituencies are excluded but not the substantive conclusion.

## 3.6 Do model residuals still show spatial clustering?

§2.6 found genuine spatial clustering in the raw `racialised_rate` (Moran's I = 0.28). If the covariates in this model fully explained that clustering, the model's residuals should look spatially random; if Moran's I on the residuals is still significant, that's a sign the NB2 specification is missing something structurally spatial (e.g. a spatial lag or a state-level random effect) — a concrete, flagged item for the next iteration, not something to paper over by adding more predictors to this pass.

In [ ]:
from libpysal.weights import Queen
from esda.moran import Moran

mu_full = np.exp(X.values @ nb_full.params.drop("alpha").values + offset.values)
alpha_hat = nb_full.params["alpha"]
var_full = mu_full + alpha_hat * mu_full ** 2
analysis_df["pearson_resid"] = (y.values - mu_full) / np.sqrt(var_full)

resid_map_df = constituencies[["WKR_NR", "geometry"]].rename(columns={"WKR_NR": "constituency_code"}).merge(
    analysis_df[["constituency_code", "pearson_resid"]], on="constituency_code", how="left"
).dropna(subset=["pearson_resid"]).reset_index(drop=True)

w = Queen.from_dataframe(resid_map_df, use_index=False)
w.transform = "r"
moran_resid = Moran(resid_map_df["pearson_resid"].values, w, permutations=999)

print(f"Moran's I on NB2 Pearson residuals = {moran_resid.I:.4f}")
print(f"p-value (999 permutations) = {moran_resid.p_sim:.4f}")
print(f"z-score = {moran_resid.z_sim:.2f}")
print(f"\nFor comparison, Moran's I on the raw racialised_rate (\u00a72.6) = 0.28, p = 0.001")

In [ ]:
# Task 1 settled on keeping pool_count as the primary specification (§3.4b), so the
# check above (on nb_full's residuals) remains the primary result. For completeness,
# we also check whether the density-only (log) variant's residuals show materially
# different spatial clustering -- i.e. whether the choice of density covariate matters
# for the unexplained spatial structure, not just for in-sample/CV fit.
mu_density = np.exp(X_v_density.values @ nb_v_density.params.drop("alpha").values + offset.values)
alpha_density = nb_v_density.params["alpha"]
var_density = mu_density + alpha_density * mu_density ** 2
resid_density = (y.values - mu_density) / np.sqrt(var_density)

resid_map_density = constituencies[["WKR_NR", "geometry"]].rename(columns={"WKR_NR": "constituency_code"}).merge(
    analysis_df[["constituency_code"]].assign(pearson_resid=resid_density), on="constituency_code", how="left"
).dropna(subset=["pearson_resid"]).reset_index(drop=True)

moran_density = Moran(resid_map_density["pearson_resid"].values, w, permutations=999)
print(f"Moran's I, pool_count-based model residuals (primary, above)  = {moran_resid.I:.4f} "
      f"(p={moran_resid.p_sim:.3f})")
print(f"Moran's I, density-only (log) model residuals                 = {moran_density.I:.4f} "
      f"(p={moran_density.p_sim:.3f})")

**Takeaway (density-covariate comparison).** The density-only variant's residual Moran's I (~0.11) is essentially the same as the pool_count-based model's (0.118) -- the two differ by less than either model's own permutation noise would allow us to distinguish. Swapping which density covariate goes into the model does not meaningfully change the amount of residual spatial clustering left over; the unexplained spatial structure is a property of the outcome and the predictor set as a whole (most likely something like a state-level effect or genuine spatial autocorrelation in unmeasured factors), not an artifact of choosing `pool_count` over `population_density_per_km2`. This reinforces §3.6's original conclusion that a spatial-lag/error term or state random effect is the right next step, independent of which density measure the model eventually settles on.

**Takeaway.** Moran's I on the model's Pearson residuals is 0.118 (p=0.004, z=3.50) — still significant, but less than half of the raw `racialised_rate`'s I=0.28. So this model explains part of the spatial clustering §2.6 found (mainly through `foreign_population_pct`, which is itself spatially structured — the Ruhr and Berlin hotspots are both higher-foreign-population-share areas), but a meaningful amount of spatial structure remains unexplained. This is the clearest concrete item for the next iteration: a spatial-lag/spatial-error NB2, or a state-level random effect (cheaper to implement), rather than adding more socioeconomic covariates to this specification.

## 3.7 Part 3 synthesis: what this preliminary model shows

**What this preliminary model supports:**
- A genuine, now triply-confirmed effect of `foreign_population_pct` on the racialised-review rate, net of venue density and five other predictors (IRR=1.034, p<0.0001 in the primary `pool_count`-based spec) -- the single most robust quantitative finding across all three notebooks. §3.4b confirms this holds up (IRR 1.026-1.034, p<0.05 in every case) whether the density covariate is `pool_count`, `population_density_per_km2` (log), or both together -- it is not an artifact of which density measure is chosen.
- `pool_count` remains the recommended density covariate, but only marginally. §3.4b tested `population_density_per_km2` as an alternative, motivated by `02_socioeconomic_political_relations.ipynb`'s finding that density is the more fundamental driver in raw correlation terms (density vs `racialised_rate` ρ=0.597 vs `pool_count`'s ρ=-0.556, with the two strongly collinear at ρ=-0.794). Once folded into this notebook's full multivariate NB2 alongside `foreign_population_pct` and the other five predictors, that raw-correlation advantage does not carry over: `pool_count` still edges out density-only and both-together on AIC (2200.6 vs 2202.8 vs 2201.6), BIC, and 5-fold CV MAE (0.610pp vs 0.613pp vs 0.615pp), and LR tests confirm adding the other density measure to either single-covariate model does not earn its extra parameter (p=0.32 and p=0.074 respectively). The margins are small, so this is a marginal, revisable call, not a decisive rejection of density.
- The 6-predictor set as a whole adds real explanatory power over `pool_count` alone (LR=77.2, p=1.3e-14) and improves out-of-sample prediction (5-fold CV MAE 0.61pp vs. 0.69pp) -- a modest but genuine predictive gain.
- Coefficients are stable under the `reliable_n` robustness check (§3.5) -- now correctly **247/299** constituencies (corrected from a stale 288/299 in this section, which predated `descriptive_analysis.ipynb`'s tightening of the reliability bar from `racialised_reviews>=5` to `>=10`). All 7 coefficients keep their sign on the stricter 247-constituency subset; the largest shifts (`unemployment_total` -0.039, `sgb_ii_recipients_pct` +0.026) are larger than the old check reported, consistent with a bigger, more different set of excluded constituencies (52 now vs. 11 before), but still well within the range expected from sampling noise given the coefficients' standard errors in §3.3.

**What this preliminary model does not (yet) support, and why:**
- Individual effects of `sgb_ii_recipients_pct` and `unemployment_total` -- both dip below significance once modelled jointly (VIF 26.8/25.8, §3.2), because they're too collinear with each other to separate. §2.4b's partial correlations should be read as evidence of a shared "deprivation" signal here, not as two independent effects. (Notebook 02 also flags that 20 constituencies -- Berlin 12, Hamburg 6, Bremen 2 -- share state-wide-constant values on several of the ingredient columns behind these two predictors and others; this is a data-granularity caveat rather than a bug, and it applies to this model too. It did not warrant re-running the full robustness suite excluding those 20 here, since notebook 02 found no sign flips and only modest (up to 0.09) shifts when doing so at the correlation level.)
- `turnout_pct`, `second_vote_spd_pct`, `second_vote_bsw_pct` -- all non-significant once placed alongside the stronger predictors; their §2.4b partial correlations were real but too small to survive joint estimation at n=299.
- A model that fully explains the spatial clustering §2.6 found: residual Moran's I=0.12 (p=0.004) is roughly half the raw rate's I=0.28, meaning genuine spatial structure remains after controlling for these six predictors and venue density. §3.6's extended check confirms this is not sensitive to the `pool_count`-vs-density choice either (density-only variant's residual I~0.11 is statistically indistinguishable from the primary model's).
- A model using all 18 §2.4 predictors together, tried and explicitly rejected in §3.2 for this pass (VIF up to 79.7, MLE fails to converge) -- a full accounting of every predictor isn't viable at n=299 without either dimensionality reduction (e.g. a deprivation composite index) or a different estimation approach (e.g. regularisation).
- A joint `pool_count` + `population_density_per_km2` specification (§3.4b): technically estimable (VIF 7.24, under the conventional danger threshold) but not supported by the LR tests, so not adopted. If `population_density_per_km2` is used in a future iteration, note that **the log-transform is a permanent requirement, not just a diagnostic check for this round** -- the raw scale is skewed enough (skew~3.3 vs. `pool_count`'s ~0.9) that the untransformed NB2 fails to converge outright.

**Concrete next-iteration items, in priority order:** (1) a spatial-lag/error NB2 or state-level random effect to address the residual clustering in §3.6, which §3.4b confirms is present regardless of the density covariate used; (2) resolve the `sgb_ii_recipients_pct`/`unemployment_total` collinearity, either by combining them into one deprivation index or picking one on theoretical grounds; (3) revisit whether `population_density_per_km2` should replace `pool_count` once a further iteration either resolves the density/pool_count near-tie more decisively or brings in additional predictors that might interact differently with a true density measure than with a venue count; (4) revisit once `theme`/`approach` IAA is finalised, since the outcome label itself (`racialised`) is what all of this is built on. This is a first, deliberately simple predictive pass, consistent with the plan agreed before starting -- the model, predictor set, and diagnostics above are all expected to be revised as that work continues.